# EV Tetraspanin Assortment Analysis — Rewrite

## Overview
This notebook models the assortment of three tetraspanins (CD9, CD81, CD63)
on the surface of extracellular vesicles (EVs) across **6 scenarios**:

| Model | Population structure | Assortment |
|-------|---------------------|------------|
| 1A | Single Pop A | Independent |
| 1B | Single Pop A | Linked (phi) |
| 2A | Pop B (CD63-high) + Pop A | Independent |
| 2B | Pop B (CD63-high) + Pop A | Linked |
| 3A | Pop B (CD63-high) + Pop C (CD9-high) + Pop A | Independent |
| 3B | Pop B (CD63-high) + Pop C (CD9-high) + Pop A | Linked |

**Run cells in order: 1.00 → 1.01 → 1.02 → 1.03 → 1.04 → 1.05 → 1.06 → 1.07**

All outputs (Excel + debug text) written to `OUTPUT_DIR` (configured in Cell 1.01).


In [1]:
# Cell 1.00
"""
================================================================================
CELL 1.00: ALL IMPORTS
================================================================================
Run this cell FIRST. No other cell should contain import statements.
================================================================================
"""

# ── Core ──────────────────────────────────────────────────────────────────────
import os, sys, time, warnings, io, logging, itertools
from datetime import datetime
from pathlib import Path
from copy import deepcopy

# ── Numerical / Statistical ───────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.optimize import minimize, differential_evolution
from scipy.stats import chi2 as chi2_dist, chi2_contingency
from scipy.special import gammaln

# ── Bootstrap / Progress ─────────────────────────────────────────────────────
from tqdm import tqdm

# ── Excel output ──────────────────────────────────────────────────────────────
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import (
    PatternFill, Font, Alignment, Border, Side, numbers
)
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

warnings.filterwarnings("ignore")

print("=" * 70)
print("CELL 1.00: ALL IMPORTS LOADED")
print("=" * 70)
print(f"  numpy      {np.__version__}")
print(f"  pandas     {pd.__version__}")
print(f"  openpyxl   {openpyxl.__version__}")
print(f"  scipy      available")
print(f"  tqdm       available")
print("✓ All imports successful")


CELL 1.00: ALL IMPORTS LOADED
  numpy      2.4.3
  pandas     3.0.1
  openpyxl   3.1.5
  scipy      available
  tqdm       available
✓ All imports successful


In [2]:
# Cell 1.01
"""
================================================================================
CELL 1.01: ALL VARIABLE DEFINITIONS
================================================================================
All constants and configurable parameters. Edit values HERE only.
================================================================================
"""

# ── File paths  ───────────────────────────────────────────────────────────────
# Update these to point to your data directory
#DATA_DIR   = r"."          # Folder containing Phenotype.csv and CNV CSVs
DATA_DIR = "/mnt/c/Users/pdeho/OneDrive - University of California, San Diego Health/Projects/Exosomes/CopyPatterns_DiFI"
#OUTPUT_DIR = r"./outputs"  # Where Excel and debug files are written
OUTPUT_DIR = "/mnt/c/Users/pdeho/OneDrive - University of California, San Diego Health/Projects/Exosomes/CopyPatterns_DiFI/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

#DATA_FILE = os.path.join(DATA_DIR, "Phenotype.csv")
DATA_FILE = "/mnt/c/Users/pdeho/OneDrive - University of California, San Diego Health/Projects/Exosomes/CopyPatterns_DiFI/Phenotype.csv"

CNV_FILES = {
    "CD9":  os.path.join(DATA_DIR, "CD9_CNV.csv"),
    "CD63": os.path.join(DATA_DIR, "CD63_CNV.csv"),
    "CD81": os.path.join(DATA_DIR, "CD81_CNV.csv"),
}

# ── Data structure ────────────────────────────────────────────────────────────
N_PHENOTYPES = 8   # 2^3 combinations
N_MARKERS    = 3   # CD9, CD81, CD63
MARKER_NAMES = ["CD9", "CD81", "CD63"]

# Phenotype labels (order matches Phenotype.csv rows)
# Pattern: CD9 / CD81 / CD63
PHENOTYPE_LABELS = ["+/+/+", "+/+/-", "+/-/+", "+/-/-",
                    "-/+/+", "-/+/-", "-/-/+", "-/-/-"]

# Marker positive indices (which rows are + for each marker)
CD9_POS  = [0, 1, 2, 3]   # CD9+  phenotypes (first position = +)
CD81_POS = [0, 1, 4, 5]   # CD81+ phenotypes (second position = +)
CD63_POS = [0, 2, 4, 6]   # CD63+ phenotypes (third position = +)

MARKER_POS_IDX = {
    "CD9":  CD9_POS,
    "CD81": CD81_POS,
    "CD63": CD63_POS,
}

# ── CNV thresholds (bimodal CD63 classification) ──────────────────────────────
CD63_LOW_MAX  = 2   # copy# 1–2 → Pop A (low-copy)
CD63_HIGH_MIN = 9   # copy# 9+  → Pop B (high-copy)

# ── Sub-population parameters (from CNV replicate analysis) ──────────────────
# These are the CNV-derived priors used to constrain the optimisation.
#
#   POP_B_FRAC_OF_CD63   – fraction of CD63+ EVs that belong to Pop B
#                          (CD63-high, ≥9 copies)
#   POP_B_FRAC_SD        – standard deviation from 5 replicates;
#                          mean ± 1 SD defines the optimizer bounds for pop size
#   POP_B_FRAC_CV        – coefficient of variation (SD/mean), retained for reference
#   POP_C_FRAC_OF_CD9    – fraction of CD9+ EVs that belong to Pop C
#                          (CD9-high, ≥9 copies, Scenario 3 only)
#   POP_C_FRAC_CV        – CV for Pop C fraction

# These are set to None here and calculated from CNV data in Cell 1.02.
# If Cell 1.02 has not been run, Cell 1.05 will raise a clear error.
POP_B_FRAC_OF_CD63  = None   # fraction of CD63+ EVs in Pop B  → calculated by Cell 1.02
POP_B_FRAC_SD       = None   # SD across replicates            → calculated by Cell 1.02
POP_B_FRAC_CV       = None   # CV (SD/mean)                    → calculated by Cell 1.02
POP_C_FRAC_OF_CD9   = None   # fraction of CD9+ EVs in Pop C  → calculated by Cell 1.02
POP_C_FRAC_SD       = None   # SD across replicates            → calculated by Cell 1.02
POP_C_FRAC_CV       = None   # CV (SD/mean)                    → calculated by Cell 1.02

# ── Phi (linkage disequilibrium) bounds ───────────────────────────────────────
# phi = P(CD9+,CD81+) - P(CD9+)*P(CD81+); bounded on the probability scale.
# Theoretical max is min(p9*(1-p81), p81*(1-p9)); ±0.25/+0.50 is a safe range
# for typical tetraspanin marginals (~0.3–0.5). Adjust if marginals differ greatly.
# Phi bounds are computed dynamically from the observed marginals in Cell 1.04
# (see PHI_BOUNDS dict populated there). These constants are retained as
# fallback defaults only, used if Cell 1.04 has not been run.
PHI_BOUND_LO_FALLBACK = -0.25
PHI_BOUND_HI_FALLBACK =  0.25

# ── Statistical parameters ────────────────────────────────────────────────────
ALPHA          = 0.05   # significance level
CONFIDENCE     = 0.95   # CI level
CI_LO, CI_HI  = 2.5, 97.5

# ── Optimisation ──────────────────────────────────────────────────────────────
# ── Regularization weights (N-scaled, applied per parameter class) ────────────
# Three separate weights are used because different parameter classes have
# different priors and different biological justification for constraint:
#
# REG_FRAC: Applied to fB and fC (population size fractions).
#   These have CNV-derived priors (mean ± SD from replicate measurements).
#   Pulling them toward their CNV values is biologically justified.
#   Recommended: 0.01–0.1 (weak; the CNV bounds already constrain the range).
#
# REG_MARG: Applied to p9, p81, p63 (marginal marker probabilities).
#   No biological prior exists for these — they are free parameters.
#   With N=18,007 data points and 3 parameters, they are well determined.
#   Recommended: 0.0 (no penalty; regularization cannot improve well-determined params).
#
# REG_PHI: Applied to phi (LD coefficients).
#   No prior exists. Pulling phi toward 0 (independence) or any other value
#   introduces bias without reducing variance.
#   Recommended: 0.0 (no penalty; phi must be free to find its true value).
#
REG_FRAC = 0.01   # penalty on fB, fC toward CNV-derived means
REG_MARG = 0.0    # no penalty on marginal probabilities
REG_PHI  = 0.0    # no penalty on phi (LD coefficients)
REG_SUBPOP_DIVERGE = 0.05  # penalty pulling sub-pop marginals toward Pop A values
                            # 0.0 = off, 0.05 = gentle, 0.5 = strong
                            # Applied to: p9_B vs p9_A, p81_B vs p81_A (Pop B)
                            #             p81_C vs p81_A, p63_C vs p63_A (Pop C)
REGULARIZATION_WEIGHT = REG_FRAC  # alias kept for module template and legacy print statements

# ── Phi barrier function weight ────────────────────────────────────────────────
# PHI_BARRIER_ALPHA controls the strength of the log barrier applied to phi
# values as they approach their parametric theoretical maximum.
# f(phi) = PHI_BARRIER_ALPHA × [-ln(1 - phi/phi_ceil) - ln(1 + phi/phi_floor)]
# where phi_ceil = min(p9*(1-p81), p81*(1-p9))  [computed from current marginals]
# and   phi_floor = min(p9*p81, (1-p9)*(1-p81)) [theoretical negative minimum]
#
# The barrier contributes ~0.35 NLL units at 50% of the boundary,
# ~1.15 units at 90%, and diverges at the boundary itself.
# This is comparable to the NLL contribution of ~5 misclassified EVs.
# Setting to 0.0 disables the barrier entirely (reverts to hard bounds only).
PHI_BARRIER_ALPHA = 0.5
PENALTY_WEIGHT        = 1.0     # Constraint-violation penalty
MAX_ITER              = 5000    # L-BFGS-B iterations (per-run)
DE_MAXITER            = 1600     # Differential-evolution outer iterations
DE_POPSIZE            = 50      # Population multiplier
BOUND_EPS             = 0.001   # Probability lower bound
BOUND_MAX             = 0.999   # Probability upper bound
RANDOM_SEED           = 42

# ── Cell 1.08: Robustness analysis configuration ──────────────────────────────
# Which model options and assortment modes to test in the robustness cell.
ROB_OPTIONS = [1, 2, 3]       # model options to evaluate
ROB_LINKED  = [False, True]   # False = independent, True = linked

# Sub-fit DE settings (used by LOO, per-rep, and reg-path fits in Cell 1.08).
# These fits operate on subsampled data and do not need the full gold-standard
# budget. Main fits in Cells 1.06/1.07 continue to use DE_MAXITER/DE_POPSIZE.
DE_MAXITER_SUB = 800   # half of DE_MAXITER; sufficient for consistency sub-fits
DE_POPSIZE_SUB = 25    # half of DE_POPSIZE

# Regularization path (Cell 1.08 Section 3).
# Part A: wide empirical checkpoints that prove reg=0 is optimal at large values.
# Part B: targeted fine-grid confirming the near-zero region for publication.
#REG_PATH_EMPIRICAL = [0, 10, 50, 100, 500, 1000, 2000]   # Part A checkpoints
REG_PATH_EMPIRICAL = [0, 10, 50, 100, 1000]   # Part A checkpoints
#REG_PATH_FINE      = [0, 0.001, 0.01, 0.1, 0.5, 1.0]     # Part B fine grid
REG_PATH_FINE      = [0, 0.01, 0.1, 1.0]     # Part B fine grid

# EWC (Evidence-Weighted Composite) scoring weights — Cell 1.08 Section 4.
# TECHNICAL REPLICATE configuration (active):
#   BIC+BF raised to 55% because LOO across technical replicates measures
#   measurement precision, not biological generalizability.
# BIOLOGICAL REPLICATE alternative (uncomment to switch):
#   EWC_W_ACC=0.20, EWC_W_R2=0.15, EWC_W_STAB=0.10, EWC_W_BIC=0.25,
#   EWC_W_BF=0.15,  EWC_W_BOOT=0.10, EWC_W_K=0.05
EWC_W_ACC   = 0.20   # Consistency-RMSE accuracy (lower = better)
EWC_W_R2    = 0.13   # Consistency-R² (higher = better)
EWC_W_STAB  = 0.12   # Consistency fold stability — SD of RMSE (lower = better)
EWC_W_BIC   = 0.35   # ΔBIC within linked group; 0 for independent models
EWC_W_BF    = 0.00   # Bayes factor exp(−0.5·ΔBIC); near-zero when ΔBIC > 10
EWC_W_BOOT  = 0.10   # Bootstrap convergence rate (fraction of fits converged)
EWC_W_K     = 0.10   # Parameter count k (fewer = better; direct parsimony)
#"The BF component was removed because BF = exp(−0.5·ΔBIC) is a direct transform of the
#ΔBIC component already in the EWC. Retaining both counted the same parsimony signal twice, artificially 
#penalizing models with small but positive ΔBIC. The 20% weight was redistributed to fit quality metrics."
assert abs(EWC_W_ACC + EWC_W_R2 + EWC_W_STAB + EWC_W_BIC +
           EWC_W_BF + EWC_W_BOOT + EWC_W_K - 1.0) < 1e-9, \
    f"EWC weights must sum to 1.0 — got {EWC_W_ACC+EWC_W_R2+EWC_W_STAB+EWC_W_BIC+EWC_W_BF+EWC_W_BOOT+EWC_W_K:.6f}"

# EWC fallback scores for missing data (model not run or BIC unavailable).
# 0.0 = penalize fully (a missing/failed model should not receive neutral credit).
EWC_FALLBACK_BF   = 0.0   # n_bf when BF is nan: model has no BIC → worst score
EWC_FALLBACK_K    = 0.0   # n_k  when k is nan:  unknown complexity → worst score

# ── Bootstrap ─────────────────────────────────────────────────────────────────
N_BOOTSTRAP = 1000

# ── Parallelism ───────────────────────────────────────────────────────────────
# DE_WORKERS: cores used by differential_evolution population evaluation
#   -1 = all logical cores, 1 = single-threaded (safe fallback)
#   Set to -1 for speed, 1 if running on a shared machine or WSL with limited RAM.
# BOOTSTRAP_JOBS: cores for parallel bootstrap (via joblib)
#   -1 = all cores, 1 = single-threaded
# DE_WORKERS must be 1 when running in Jupyter on WSL/Windows.
# workers=-1 uses multiprocessing.ProcessPoolExecutor which deadlocks
# in Jupyter because forked child processes inherit the kernel's ZMQ sockets.
# The bootstrap (joblib prefer="threads") is safe and DOES run in parallel.
DE_WORKERS      = 1    # DO NOT change to -1 in Jupyter on WSL — causes silent hang
BOOTSTRAP_JOBS  = -1   # thread-based parallelism: safe in all environments

# ── Helper: TeeOutput (write to console AND file simultaneously) ──────────────
class TeeOutput:
    """Duplicate stdout/stderr to a log file."""
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj); f.flush()
    def flush(self):
        for f in self.files: f.flush()

def start_logging(cell_name):
    """Open a timestamped debug log and tee stdout/stderr into it."""
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = os.path.join(OUTPUT_DIR, f"{cell_name}_Debug_{ts}.txt")
    try:
        lf = open(log_path, "w", encoding="utf-8")
        sys.stdout = TeeOutput(sys.__stdout__, lf)
        sys.stderr = TeeOutput(sys.__stderr__, lf)
        print("=" * 70)
        print(f"{cell_name.upper()} - DEBUG LOG")
        print("=" * 70)
        print(f"Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Log file  : {log_path}\n")
        return lf, ts
    except Exception as e:
        print(f"⚠ Could not create log file: {e}")
        return None, datetime.now().strftime("%Y%m%d_%H%M%S")

def stop_logging(lf):
    """Restore stdout/stderr and close the log file."""
    sys.stdout = sys.__stdout__
    sys.stderr = sys.__stderr__
    if lf:
        lf.close()

def style_header_row(ws, row_num, fill_hex="2F5597", font_hex="FFFFFF"):
    """Apply bold white-on-blue formatting to a header row."""
    fill = PatternFill("solid", fgColor=fill_hex)
    font = Font(bold=True, color=font_hex)
    for cell in ws[row_num]:
        cell.fill = fill
        cell.font = font
        cell.alignment = Alignment(horizontal="center", wrap_text=True)

def auto_width(ws, min_w=10, max_w=40):
    """Auto-size column widths."""
    for col in ws.columns:
        length = max((len(str(c.value or "")) for c in col), default=0)
        ws.column_dimensions[col[0].column_letter].width = max(min_w, min(max_w, length + 2))

def save_wb(wb, cell_name, ts):
    """Save workbook to outputs with timestamped name."""
    fname = f"{cell_name}_{ts}.xlsx"
    fpath = os.path.join(OUTPUT_DIR, fname)
    wb.save(fpath)
    print(f"\n✓ Saved: {fname}")
    return fpath

def require_cnv_data():
    """Raise a clear error if Cell 1.02 has not been run."""
    missing = [v for v in ['POP_B_FRAC_OF_CD63','POP_B_FRAC_SD','POP_C_FRAC_OF_CD9','POP_C_FRAC_SD']
               if globals().get(v) is None]
    if missing:
        raise RuntimeError(
            f"⚠ CNV variables not yet computed: {missing}\n"
            f"  Run Cell 1.02 before Cell 1.05/1.06/1.07."
        )

def require_independence_analysis():
    """Raise a clear error if Cell 1.04 has not been run (needed for phi bounds)."""
    if 'PHI_BOUNDS' not in globals():
        raise RuntimeError(
            "⚠ PHI_BOUNDS not computed.\n"
            "  Run Cell 1.04 before Cell 1.05/1.06/1.07."
        )

print("=" * 70)
print("CELL 1.01: VARIABLE DEFINITIONS COMPLETE")
print("=" * 70)
print(f"  Output dir : {os.path.abspath(OUTPUT_DIR)}")
print(f"  Data file  : {DATA_FILE}")
print(f"  Bootstrap  : {N_BOOTSTRAP:,} samples")
print(f"  Regularize : REG_FRAC={REG_FRAC}  REG_MARG={REG_MARG}  REG_PHI={REG_PHI}")
print(f"  Phi barrier: PHI_BARRIER_ALPHA={PHI_BARRIER_ALPHA}")
print(f"  Seed       : {RANDOM_SEED}")


CELL 1.01: VARIABLE DEFINITIONS COMPLETE
  Output dir : /mnt/c/Users/pdeho/OneDrive - University of California, San Diego Health/Projects/Exosomes/CopyPatterns_DiFI/outputs
  Data file  : /mnt/c/Users/pdeho/OneDrive - University of California, San Diego Health/Projects/Exosomes/CopyPatterns_DiFI/Phenotype.csv
  Bootstrap  : 1,000 samples
  Regularize : REG_FRAC=0.01  REG_MARG=0.0  REG_PHI=0.0
  Phi barrier: PHI_BARRIER_ALPHA=0.5
  Seed       : 42


In [3]:
# Cell 1.02
"""
================================================================================
CELL 1.02: CNV DATA LOADING & SUB-POPULATION PARAMETER ESTIMATION
================================================================================
PURPOSE:
  Load per-replicate copy-number variation (CNV) data for each tetraspanin.
  Identify the bimodal CD63 distribution to estimate Pop B size.
  Identify high-copy CD9 EVs to estimate Pop C size (Scenario 3).
  Updates POP_B_FRAC_OF_CD63, POP_B_FRAC_CV, POP_C_FRAC_OF_CD9, POP_C_FRAC_CV.

OUTPUT:
  Excel: Cell_1.02_CNV_Data_<ts>.xlsx
    Tabs: CD9, CD63, CD81, PopB_Replicates, PopC_Replicates, Summary
  Debug: Cell_1.02_Debug_<ts>.txt
================================================================================
"""

lf, ts = start_logging("Cell_1.02")

print("=" * 70)
print("CELL 1.02: CNV DATA LOADING & SUB-POPULATION ESTIMATION")
print("=" * 70)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print("STEP 1: Locating CNV files")
print("-" * 60)
for marker, fpath in CNV_FILES.items():
    exists = os.path.exists(fpath)
    status = "✓ Found" if exists else "✗ NOT FOUND"
    print(f"  {status}: {marker} → {fpath}")

print("\nSTEP 2: Loading CNV data")
print("-" * 60)
# ── Load each CNV file ────────────────────────────────────────────────────────
cnv_data = {}
for marker, fpath in CNV_FILES.items():
    if not os.path.exists(fpath):
        print(f"  ⚠ {marker}: file not found at {fpath}  (will be skipped)")
        continue
    df = pd.read_csv(fpath, header=None)
    # Expected format: col0=copy#, col1..5=R1..R5 percentages
    df.columns = ["Copy_Number", "R1", "R2", "R3", "R4", "R5"]
    df = df[pd.to_numeric(df["Copy_Number"], errors="coerce").notna()].copy()
    df["Copy_Number"] = df["Copy_Number"].astype(int)
    for r in ["R1","R2","R3","R4","R5"]:
        df[r] = pd.to_numeric(df[r], errors="coerce").fillna(0.0)
    df = df.sort_values("Copy_Number").reset_index(drop=True)
    cnv_data[marker] = df
    print(f"  ✓ {marker}: {len(df)} copy-number rows, copies {df.Copy_Number.min()}–{df.Copy_Number.max()}")

# ── Estimate Pop B (CD63-high) per replicate ──────────────────────────────────
pop_b_per_rep = {}
pop_c_per_rep = {}

if "CD63" in cnv_data:
    df63 = cnv_data["CD63"]
    for r in ["R1","R2","R3","R4","R5"]:
        high = df63.loc[df63["Copy_Number"] >= CD63_HIGH_MIN, r].sum() / 100.0
        pop_b_per_rep[r] = high
    vals_b = np.array(list(pop_b_per_rep.values()))
    POP_B_FRAC_OF_CD63 = float(vals_b.mean())
    POP_B_FRAC_SD      = float(vals_b.std(ddof=1))
    POP_B_FRAC_CV      = float(POP_B_FRAC_SD / POP_B_FRAC_OF_CD63) if POP_B_FRAC_OF_CD63 > 0 else 0.08
    print(f"\n  CD63 Pop B (≥{CD63_HIGH_MIN} copies):")
    for r, v in pop_b_per_rep.items():
        print(f"    {r}: {v*100:.2f}%")
    print(f"\n  Pop B per-replicate summary:")
    print(f"  {'Replicate':<12} {'High-Copy %':>12} {'Low-Copy %':>12}")
    print(f"  {'-'*38}")
    for r, v in pop_b_per_rep.items():
        print(f"  {r:<12} {v*100:>11.2f}% {(1-v)*100:>11.2f}%")
    b_vals = np.array(list(pop_b_per_rep.values()))
    print(f"  {'Mean':<12} {POP_B_FRAC_OF_CD63*100:>11.2f}%")
    print(f"  {'SD':<12} {b_vals.std(ddof=1)*100:>11.2f}%")
    print(f"  {'CV':<12} {POP_B_FRAC_CV:>11.4f}")
    print(f"  → Pop B (CD63-high ≥{CD63_HIGH_MIN} copies): {POP_B_FRAC_OF_CD63*100:.2f}% of CD63+ EVs")

if "CD9" in cnv_data:
    df9 = cnv_data["CD9"]
    for r in ["R1","R2","R3","R4","R5"]:
        high = df9.loc[df9["Copy_Number"] >= CD63_HIGH_MIN, r].sum() / 100.0
        pop_c_per_rep[r] = high
    vals_c = np.array(list(pop_c_per_rep.values()))
    POP_C_FRAC_OF_CD9 = float(vals_c.mean())
    POP_C_FRAC_SD     = float(vals_c.std(ddof=1))
    POP_C_FRAC_CV     = float(POP_C_FRAC_SD / POP_C_FRAC_OF_CD9) if POP_C_FRAC_OF_CD9 > 0 else 0.30
    print(f"\n  CD9 Pop C (≥{CD63_HIGH_MIN} copies):")
    for r, v in pop_c_per_rep.items():
        print(f"    {r}: {v*100:.2f}%")
    print(f"\n  Pop C per-replicate summary:")
    print(f"  {'Replicate':<12} {'High-Copy %':>12} {'Low-Copy %':>12}")
    print(f"  {'-'*38}")
    for r, v in pop_c_per_rep.items():
        print(f"  {r:<12} {v*100:>11.2f}% {(1-v)*100:>11.2f}%")
    c_vals = np.array(list(pop_c_per_rep.values()))
    print(f"  {'Mean':<12} {POP_C_FRAC_OF_CD9*100:>11.2f}%")
    print(f"  {'SD':<12} {c_vals.std(ddof=1)*100:>11.2f}%")
    print(f"  {'CV':<12} {POP_C_FRAC_CV:>11.4f}")
    print(f"  → Pop C (CD9-high ≥{CD63_HIGH_MIN} copies): {POP_C_FRAC_OF_CD9*100:.2f}% of CD9+ EVs")

print("\nSTEP 3: Building Excel output")
print("-" * 60)
# ── Build Excel workbook ──────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)  # remove default sheet

for marker, df in cnv_data.items():
    ws = wb.create_sheet(marker)
    ws.append(["Copy_Number", "R1 (%)", "R2 (%)", "R3 (%)", "R4 (%)", "R5 (%)", "Mean (%)", "SD (%)"])
    style_header_row(ws, 1)
    for _, row in df.iterrows():
        reps = [row[f"R{i}"] for i in range(1,6)]
        ws.append([int(row["Copy_Number"])] + reps +
                  [round(np.mean(reps), 4), round(np.std(reps, ddof=1), 4)])
    auto_width(ws)

# ── Derive exact EV counts from the phenotype CSV for accurate bound reporting ─
# We read the phenotype file here so bound EV counts are exact, not approximate.
# This does NOT affect Cell 1.03 — it is a read-only peek used only for Excel output.
print("STEP 3a: Reading phenotype file for exact EV counts (bound reporting only)")
print("-" * 60)
_ph_df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
_rep_cols = [c for c in _ph_df.columns if "replicate" in c.lower()]
_first_rep_vals = [pd.to_numeric(_ph_df[c].iloc[0], errors='coerce') for c in _rep_cols]
if all(pd.isna(v) for v in _first_rep_vals):
    _ph_df = _ph_df.iloc[1:].reset_index(drop=True)
for c in _rep_cols:
    _ph_df[c] = pd.to_numeric(_ph_df[c], errors='coerce').fillna(0).astype(int)
_ph_data  = _ph_df[_rep_cols].values[:8].astype(int)
_ph_counts = _ph_data.sum(axis=1)
_EXACT_TOTAL    = int(_ph_counts.sum())
_EXACT_CD63_POS = int(_ph_counts[CD63_POS].sum())
_EXACT_CD9_POS  = int(_ph_counts[CD9_POS].sum())
_N_REPS_CNV     = len(pop_b_per_rep)   # replicates found in CNV data
print(f"  TOTAL_EVS (from phenotype CSV) = {_EXACT_TOTAL:,}")
print(f"  CD63_POSITIVE_EVS              = {_EXACT_CD63_POS:,}  ({_EXACT_CD63_POS/_EXACT_TOTAL*100:.2f}%)")
print(f"  CD9_POSITIVE_EVS               = {_EXACT_CD9_POS:,}  ({_EXACT_CD9_POS/_EXACT_TOTAL*100:.2f}%)")
print(f"  N replicates in CNV data       = {_N_REPS_CNV}")

# Pop B replicate sheet
ws_b = wb.create_sheet("PopB_Replicates")
ws_b.append(["Replicate", "CD63-High Fraction (of CD63+)", "CD63-High %"])
style_header_row(ws_b, 1)
for r, v in pop_b_per_rep.items():
    ws_b.append([r, round(v, 6), round(v * 100, 3)])
ws_b.append(["Mean", round(POP_B_FRAC_OF_CD63, 6), round(POP_B_FRAC_OF_CD63 * 100, 3)])
ws_b.append(["SD",   round(POP_B_FRAC_SD, 6),       round(POP_B_FRAC_SD * 100, 3)])
ws_b.append(["CV",   round(POP_B_FRAC_CV, 6),        ""])
ws_b.append(["Lower bound (Mean − 1SD)", round((POP_B_FRAC_OF_CD63 - POP_B_FRAC_SD), 6),
             round((POP_B_FRAC_OF_CD63 - POP_B_FRAC_SD) * 100, 3)])
ws_b.append(["Upper bound (Mean + 1SD)", round((POP_B_FRAC_OF_CD63 + POP_B_FRAC_SD), 6),
             round((POP_B_FRAC_OF_CD63 + POP_B_FRAC_SD) * 100, 3)])
auto_width(ws_b)

# Pop C replicate sheet
ws_c = wb.create_sheet("PopC_Replicates")
ws_c.append(["Replicate", "CD9-High Fraction (of CD9+)", "CD9-High %"])
style_header_row(ws_c, 1)
for r, v in pop_c_per_rep.items():
    ws_c.append([r, round(v, 6), round(v * 100, 3)])
ws_c.append(["Mean", round(POP_C_FRAC_OF_CD9, 6), round(POP_C_FRAC_OF_CD9 * 100, 3)])
ws_c.append(["SD",   round(POP_C_FRAC_SD, 6),      round(POP_C_FRAC_SD * 100, 3)])
ws_c.append(["CV",   round(POP_C_FRAC_CV, 6),       ""])
ws_c.append(["Lower bound (Mean − 1SD)", round((POP_C_FRAC_OF_CD9 - POP_C_FRAC_SD), 6),
             round((POP_C_FRAC_OF_CD9 - POP_C_FRAC_SD) * 100, 3)])
ws_c.append(["Upper bound (Mean + 1SD)", round((POP_C_FRAC_OF_CD9 + POP_C_FRAC_SD), 6),
             round((POP_C_FRAC_OF_CD9 + POP_C_FRAC_SD) * 100, 3)])
auto_width(ws_c)

# Summary sheet — bounds as % of parent marker AND as % of total EVs and EV counts
# EV counts here are approximate; Cell 1.03 will have exact values once phenotype data loads.
# We estimate total EVs from the CNV-derived fractions using a placeholder of 18007
# (the known dataset size — update this if your dataset changes).
_b_mean_tot = POP_B_FRAC_OF_CD63 * _EXACT_CD63_POS / _EXACT_TOTAL
_b_sd_tot   = POP_B_FRAC_SD      * _EXACT_CD63_POS / _EXACT_TOTAL
_b_lo_tot   = max(0, _b_mean_tot - _b_sd_tot)
_b_hi_tot   = _b_mean_tot + _b_sd_tot

_c_mean_tot = POP_C_FRAC_OF_CD9 * _EXACT_CD9_POS / _EXACT_TOTAL
_c_sd_tot   = POP_C_FRAC_SD     * _EXACT_CD9_POS / _EXACT_TOTAL
_c_lo_tot   = max(0, _c_mean_tot - _c_sd_tot)
_c_hi_tot   = _c_mean_tot + _c_sd_tot

ws_s = wb.create_sheet("Summary")
ws_s.append(["Parameter", "Value (%)", "Value (fraction)", "Description"])
style_header_row(ws_s, 1)

summary_rows = [
    # ── Pop B ──────────────────────────────────────────────────────────────────
    ["--- Pop B (CD63-high, ≥9 copies) ---", "", "", ""],
    ["Pop B mean (% of CD63+)",
     round(POP_B_FRAC_OF_CD63 * 100, 4), round(POP_B_FRAC_OF_CD63, 6),
     "Mean fraction of CD63+ EVs in Pop B across 5 replicates"],
    ["Pop B SD (% of CD63+)",
     round(POP_B_FRAC_SD * 100, 4), round(POP_B_FRAC_SD, 6),
     "Standard deviation across 5 replicates"],
    ["Pop B lower bound (Mean−SD, % of CD63+)",
     round((POP_B_FRAC_OF_CD63 - POP_B_FRAC_SD) * 100, 4),
     round(POP_B_FRAC_OF_CD63 - POP_B_FRAC_SD, 6), "Optimizer lower bound for Pop B fraction of CD63+"],
    ["Pop B upper bound (Mean+SD, % of CD63+)",
     round((POP_B_FRAC_OF_CD63 + POP_B_FRAC_SD) * 100, 4),
     round(POP_B_FRAC_OF_CD63 + POP_B_FRAC_SD, 6), "Optimizer upper bound for Pop B fraction of CD63+"],
    ["Pop B mean (% of total EVs)",
     round(_b_mean_tot * 100, 4), round(_b_mean_tot, 6),
     "Mean Pop B as fraction of all EVs"],
    ["Pop B SD (% of total EVs)",
     round(_b_sd_tot * 100, 4), round(_b_sd_tot, 6),
     "SD of Pop B fraction of all EVs"],
    ["Pop B lower bound (# EVs)",
     round(_b_lo_tot * _EXACT_TOTAL, 1), "",
     f"Lower bound EV count (exact, based on {_EXACT_TOTAL:,} total EVs)"],
    ["Pop B upper bound (# EVs)",
     round(_b_hi_tot * _EXACT_TOTAL, 1), "",
     f"Upper bound EV count (exact, based on {_EXACT_TOTAL:,} total EVs)"],
    # ── Pop C ──────────────────────────────────────────────────────────────────
    ["--- Pop C (CD9-high, ≥9 copies) ---", "", "", ""],
    ["Pop C mean (% of CD9+)",
     round(POP_C_FRAC_OF_CD9 * 100, 4), round(POP_C_FRAC_OF_CD9, 6),
     "Mean fraction of CD9+ EVs in Pop C across 5 replicates"],
    ["Pop C SD (% of CD9+)",
     round(POP_C_FRAC_SD * 100, 4), round(POP_C_FRAC_SD, 6),
     "Standard deviation across 5 replicates"],
    ["Pop C lower bound (Mean−SD, % of CD9+)",
     round((POP_C_FRAC_OF_CD9 - POP_C_FRAC_SD) * 100, 4),
     round(max(0, POP_C_FRAC_OF_CD9 - POP_C_FRAC_SD), 6),
     "Optimizer lower bound for Pop C fraction of CD9+"],
    ["Pop C upper bound (Mean+SD, % of CD9+)",
     round((POP_C_FRAC_OF_CD9 + POP_C_FRAC_SD) * 100, 4),
     round(POP_C_FRAC_OF_CD9 + POP_C_FRAC_SD, 6),
     "Optimizer upper bound for Pop C fraction of CD9+"],
    ["Pop C mean (% of total EVs)",
     round(_c_mean_tot * 100, 4), round(_c_mean_tot, 6),
     "Mean Pop C as fraction of all EVs"],
    ["Pop C SD (% of total EVs)",
     round(_c_sd_tot * 100, 4), round(_c_sd_tot, 6),
     "SD of Pop C fraction of all EVs"],
    ["Pop C lower bound (# EVs)",
     round(_c_lo_tot * _EXACT_TOTAL, 1), "",
     f"Lower bound EV count (exact, based on {_EXACT_TOTAL:,} total EVs)"],
    ["Pop C upper bound (# EVs)",
     round(_c_hi_tot * _EXACT_TOTAL, 1), "",
     f"Upper bound EV count (exact, based on {_EXACT_TOTAL:,} total EVs)"],
    # ── Other parameters ──────────────────────────────────────────────────────
    ["--- Other parameters ---", "", "", ""],
    ["CD63_HIGH_MIN",  CD63_HIGH_MIN, "",
     "Minimum copy number defining high-copy (Pop B/C)"],
    ["N_Replicates", _N_REPS_CNV, "", "Number of biological replicates detected in CNV data"],
    ["POP_B_FRAC_CV",  round(POP_B_FRAC_CV, 6), "", "CV (SD/Mean) for Pop B — used in legacy bounds"],
    ["POP_C_FRAC_CV",  round(POP_C_FRAC_CV, 6), "", "CV (SD/Mean) for Pop C — used in legacy bounds"],
]
for r in summary_rows:
    ws_s.append(r)
auto_width(ws_s)

fpath_cnv = save_wb(wb, "Cell_1.02_CNV_Data", ts)

print(f"\n✓ Updated globals:")
print(f"  POP_B_FRAC_OF_CD63 = {POP_B_FRAC_OF_CD63:.4f}  ({POP_B_FRAC_OF_CD63*100:.2f}%)")
print(f"  POP_B_FRAC_CV      = {POP_B_FRAC_CV:.4f}")
print(f"  POP_C_FRAC_OF_CD9  = {POP_C_FRAC_OF_CD9:.4f}  ({POP_C_FRAC_OF_CD9*100:.2f}%)")
print(f"  POP_C_FRAC_CV      = {POP_C_FRAC_CV:.4f}")

stop_logging(lf)


CELL_1.02 - DEBUG LOG
Timestamp : 2026-04-13 15:40:34
Log file  : /mnt/c/Users/pdeho/OneDrive - University of California, San Diego Health/Projects/Exosomes/CopyPatterns_DiFI/outputs/Cell_1.02_Debug_20260413_154034.txt

CELL 1.02: CNV DATA LOADING & SUB-POPULATION ESTIMATION
Timestamp: 2026-04-13 15:40:34

STEP 1: Locating CNV files
------------------------------------------------------------
  ✓ Found: CD9 → /mnt/c/Users/pdeho/OneDrive - University of California, San Diego Health/Projects/Exosomes/CopyPatterns_DiFI/CD9_CNV.csv
  ✓ Found: CD63 → /mnt/c/Users/pdeho/OneDrive - University of California, San Diego Health/Projects/Exosomes/CopyPatterns_DiFI/CD63_CNV.csv
  ✓ Found: CD81 → /mnt/c/Users/pdeho/OneDrive - University of California, San Diego Health/Projects/Exosomes/CopyPatterns_DiFI/CD81_CNV.csv

STEP 2: Loading CNV data
------------------------------------------------------------
  ✓ CD9: 20 copy-number rows, copies 1–20
  ✓ CD63: 20 copy-number rows, copies 1–20
  ✓ CD81: 20 c

  Option 1A bootstrap: 100%|##########| 1000/1000 [00:03<00:00, 268.60it/s]


  Bootstrap wall time: 3.9s

  Bootstrap results:
    Successful fits : 1000/1000  (100.0%)
    ✓ Acceptable success rate

────────────────────────────────────────────────────────────
  OPTION 2A — Independent assortment
────────────────────────────────────────────────────────────
  Parameters : 7
  Bounds range summary: lo=[0.0010..0.1060]  hi=[0.1234..0.9990]

  STEP 1 — Global search (DE, 7 params, workers=1)...
  Optimization done in 3.0s  fun=33617.556078

  STEP 2 — Fitted parameters:
  --------------------------------------------------
  Population A:
    P(CD9+) = 0.5084  (50.84%)
    P(CD81+) = 0.3323  (33.23%)
    P(CD63+) = 0.2069  (20.69%)
  Population B:
    P(CD9+) = 0.4410  (44.10%)
    P(CD81+) = 0.2926  (29.26%)
    P(CD63+) = 1.0000  (100.00%)
    Size (frac of total) = 0.0283  (2.83%)

  STEP 3 — Observed vs. predicted phenotype counts:
  Phenotype    Observed  Predicted   Residual   Std.Res
  ----------------------------------------------------
  +/+/+           1,1

  Option 2A bootstrap: 100%|##########| 1000/1000 [00:23<00:00, 42.04it/s]


  Bootstrap wall time: 24.7s

  Bootstrap results:
    Successful fits : 1000/1000  (100.0%)
    ✓ Acceptable success rate

────────────────────────────────────────────────────────────
  OPTION 3A — Independent assortment
────────────────────────────────────────────────────────────
  Parameters : 11
  Bounds range summary: lo=[0.0010..0.1060]  hi=[0.0358..0.9990]

  STEP 1 — Global search (DE, 11 params, workers=1)...
  Optimization done in 11.9s  fun=33594.563781

  STEP 2 — Fitted parameters:
  --------------------------------------------------
  Population A:
    P(CD9+) = 0.4982  (49.82%)
    P(CD81+) = 0.3288  (32.88%)
    P(CD63+) = 0.2070  (20.70%)
  Population B:
    P(CD9+) = 0.4291  (42.91%)
    P(CD81+) = 0.2840  (28.40%)
    P(CD63+) = 1.0000  (100.00%)
    Size (frac of total) = 0.0283  (2.83%)
  Population C:
    P(CD9+) = 1.0000  (100.00%)
    P(CD81+) = 0.4834  (48.34%)
    P(CD63+) = 0.2024  (20.24%)
    Size (frac of total) = 0.0181  (1.81%)

  STEP 3 — Observed vs. p

...


  Bootstrap results:
    Successful fits : 1000/1000  (100.0%)
    ✓ Acceptable success rate

────────────────────────────────────────────────────────────
  OPTION 2B — Linked assortment (phi)
────────────────────────────────────────────────────────────

  STEP 1 — Global search (DE, 10 params, workers=1)...
  Optimization done in 11.7s  fun=31932.089525

  STEP 2 — Fitted parameters:
  --------------------------------------------------
  Population A:
    P(CD9+) = 0.5066  (50.66%)
    P(CD81+) = 0.3302  (33.02%)
    P(CD63+) = 0.2063  (20.63%)
    phi(CD9–CD81) = +0.095454  (co-occurrence excess)
    phi(CD9–CD63) = -0.010493  (mutual exclusion)
  Population B:
    P(CD9+) = 0.4991  (49.91%)
    P(CD81+) = 0.3401  (34.01%)
    P(CD63+) = 1.0000  (100.00%)
    phi(CD9–CD81) = +0.163468  (co-occurrence excess)
    phi(CD9–CD63) = +0.000000  (~independent)
    Size (frac of total) = 0.0283  (2.83%)

  STEP 3 — Observed vs. predicted phenotype counts:
  Phenotype    Observed  Predicted 

  Option 2B bootstrap: 100%|##########| 1000/1000 [00:40<00:00, 24.66it/s]



  Bootstrap results:
    Successful fits : 1000/1000  (100.0%)
    ✓ Acceptable success rate

────────────────────────────────────────────────────────────
  OPTION 3B — Linked assortment (phi)
────────────────────────────────────────────────────────────

  STEP 1 — Global search (DE, 15 params, workers=1)...
  Optimization done in 54.7s  fun=31917.895661

  STEP 2 — Fitted parameters:
  --------------------------------------------------
  Population A:
    P(CD9+) = 0.4972  (49.72%)
    P(CD81+) = 0.3306  (33.06%)
    P(CD63+) = 0.2061  (20.61%)
    phi(CD9–CD81) = +0.097470  (co-occurrence excess)
    phi(CD9–CD63) = -0.010653  (mutual exclusion)
  Population B:
    P(CD9+) = 0.4897  (48.97%)
    P(CD81+) = 0.3313  (33.13%)
    P(CD63+) = 1.0000  (100.00%)
    phi(CD9–CD81) = +0.163468  (co-occurrence excess)
    phi(CD9–CD63) = +0.000000  (~independent)
    Size (frac of total) = 0.0283  (2.83%)
  Population C:
    P(CD9+) = 1.0000  (100.00%)
    P(CD81+) = 0.3265  (32.65%)
    P(CD

  Option 3B bootstrap: 100%|##########| 1000/1000 [02:32<00:00,  6.54it/s]



  Bootstrap results:
    Successful fits : 958/1000  (95.8%)
    ✓ Acceptable success rate

Building Excel output...

✓ Saved: Cell_1.07_Linked_Models_20260413_154214.xlsx

✓ Linked models complete.
  Tabs: Option1B/2B/3B_Params, Option1B/2B/3B_Phenotypes,
        Model_Comparison_All6, Population_Data, Model, Option_Desc
CELL_1.08 - DEBUG LOG
Timestamp : 2026-04-13 15:46:45
Log file  : /mnt/c/Users/pdeho/OneDrive - University of California, San Diego Health/Projects/Exosomes/CopyPatterns_DiFI/outputs/Cell_1.08_Debug_20260413_154645.txt

CELL 1.08: REPLICATE ROBUSTNESS ANALYSIS

────────────────────────────────────────────────────────────
SECTION 1: Replicate Consistency Check
  (Technical replicates: measures internal consistency, NOT generalizability)
────────────────────────────────────────────────────────────

  Option 1A...
    Rep 1 (held out, consistency check): RMSE=159.2  R²=0.7859
    Rep 2 (held out, consistency check): RMSE=289.2  R²=0.6682
    Rep 3 (held out, consistency

In [4]:
# Cell 1.03
"""
================================================================================
CELL 1.03: LOAD & VALIDATE PHENOTYPE DATA
================================================================================
PURPOSE:
  Load phenotype count matrix (8 phenotypes × 5 replicates) from CSV.
  Calculate TOTAL_EVS, CD63_POSITIVE_EVS, observed_counts, replicate_data.

OUTPUT:
  Excel: Cell_1.03_Phenotype_Data_<ts>.xlsx
    Tabs: Raw_Data, Observed_Counts, Replicate_Marginals
  Debug: Cell_1.03_Debug_<ts>.txt
================================================================================
"""

lf, ts = start_logging("Cell_1.03")

print("=" * 70)
print("CELL 1.03: LOAD & VALIDATE PHENOTYPE DATA")
print("=" * 70)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print("STEP 1: Loading CSV file")
print("-" * 60)
# ── Load CSV ──────────────────────────────────────────────────────────────────
phenotype_df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
print(f"  ✓ Loaded : {os.path.basename(DATA_FILE)}")
print(f"  Shape    : {phenotype_df.shape[0]} rows × {phenotype_df.shape[1]} columns")
print(f"  Columns  : {list(phenotype_df.columns)}")

print("\nSTEP 2: Identifying replicate columns")
print("-" * 60)
# Find replicate columns (they are already in the column headers)
replicate_cols = [c for c in phenotype_df.columns if "replicate" in c.lower()]
N_REPLICATES   = len(replicate_cols)
print(f"  Replicate columns found in headers: {replicate_cols}")

# Drop the sub-header row (row 0: contains marker names CD9-PE, CD81-PE/Cy7, etc.)
# This row has NaN in all replicate columns — detect and remove it
first_row_rep_vals = [pd.to_numeric(phenotype_df[c].iloc[0], errors='coerce')
                      for c in replicate_cols]
if all(pd.isna(v) for v in first_row_rep_vals):
    print(f"  ✓ Sub-header row detected at index 0 (all replicate values are NaN) — dropping it")
    phenotype_df = phenotype_df.iloc[1:].reset_index(drop=True)
    print(f"  Shape after drop: {phenotype_df.shape}")
else:
    print(f"  No sub-header row detected — using all rows")

for c in replicate_cols:
    phenotype_df[c] = pd.to_numeric(phenotype_df[c], errors="coerce").fillna(0).astype(int)

print(f"  ✓ Found {N_REPLICATES} replicate columns:")
for rc in replicate_cols:
    print(f"    • {rc}")

print("\nSTEP 3: Extracting counts and totals")
print("-" * 60)
# ── Extract arrays ────────────────────────────────────────────────────────────
replicate_data  = phenotype_df[replicate_cols].values[:8].astype(int)   # (8, 5)
observed_counts = replicate_data.sum(axis=1).astype(int)                 # (8,)
TOTAL_EVS       = int(observed_counts.sum())
CD63_POSITIVE_EVS = int(observed_counts[CD63_POS].sum())
CD9_POSITIVE_EVS  = int(observed_counts[CD9_POS].sum())
CD81_POSITIVE_EVS = int(observed_counts[CD81_POS].sum())

print(f"\n  TOTAL_EVS         = {TOTAL_EVS:,}")
print(f"  CD63_POSITIVE_EVS = {CD63_POSITIVE_EVS:,}  ({CD63_POSITIVE_EVS/TOTAL_EVS*100:.2f}%)")
print(f"  CD9_POSITIVE_EVS  = {CD9_POSITIVE_EVS:,}  ({CD9_POSITIVE_EVS/TOTAL_EVS*100:.2f}%)")
print(f"  CD81_POSITIVE_EVS = {CD81_POSITIVE_EVS:,}  ({CD81_POSITIVE_EVS/TOTAL_EVS*100:.2f}%)")

print(f"\n  TOTAL_EVS         = {TOTAL_EVS:,}")
print(f"  CD9_POSITIVE_EVS  = {CD9_POSITIVE_EVS:,}  ({CD9_POSITIVE_EVS/TOTAL_EVS*100:.2f}%)")
print(f"  CD81_POSITIVE_EVS = {CD81_POSITIVE_EVS:,}  ({CD81_POSITIVE_EVS/TOTAL_EVS*100:.2f}%)")
print(f"  CD63_POSITIVE_EVS = {CD63_POSITIVE_EVS:,}  ({CD63_POSITIVE_EVS/TOTAL_EVS*100:.2f}%)")

print(f"\nSTEP 4: Replicate breakdown")
print("-" * 60)
print(f"  {'Replicate':<14} {'Count':>8}  {'% of Total':>10}")
print(f"  {'-'*36}")
for j, rc in enumerate(replicate_cols):
    col_sum = replicate_data[:, j].sum()
    print(f"  {rc:<14} {col_sum:>8,}  ({col_sum/TOTAL_EVS*100:>8.1f}%)")
print(f"  {'TOTAL':<14} {TOTAL_EVS:>8,}  ({100.0:>8.1f}%)")

print(f"\nSTEP 5: Phenotype distribution (pooled)")
print("-" * 60)
print(f"  {'Phenotype':<10} {'Count':>8}  {'Percentage':>10}")
print(f"  {'-'*32}")
for i, lbl in enumerate(PHENOTYPE_LABELS):
    pct = observed_counts[i]/TOTAL_EVS*100
    print(f"  {lbl:<10} {observed_counts[i]:>8,}  ({pct:>8.2f}%)")
print(f"  {'-'*32}")
print(f"  {'TOTAL':<10} {TOTAL_EVS:>8,}  ({100.0:>8.2f}%)")
print(f"\n  Most common : {PHENOTYPE_LABELS[np.argmax(observed_counts)]} ({observed_counts.max():,} EVs, {observed_counts.max()/TOTAL_EVS*100:.1f}%)")
print(f"  Least common: {PHENOTYPE_LABELS[np.argmin(observed_counts)]} ({observed_counts.min():,} EVs, {observed_counts.min()/TOTAL_EVS*100:.1f}%)")

# ── Replicate marginals ───────────────────────────────────────────────────────
rep_marginals = []
for j, rc in enumerate(replicate_cols):
    col = replicate_data[:, j]
    tot = col.sum()
    rep_marginals.append({
        "Replicate": rc,
        "Total":     int(tot),
        "p_CD9":     round(col[CD9_POS].sum()  / tot, 6),
        "p_CD81":    round(col[CD81_POS].sum() / tot, 6),
        "p_CD63":    round(col[CD63_POS].sum() / tot, 6),
    })
df_rep_marg = pd.DataFrame(rep_marginals)
print(f"\nReplicate marginals:")
print(df_rep_marg.to_string(index=False))

# ── Excel ─────────────────────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

# Raw data tab
ws_raw = wb.create_sheet("Raw_Data")
ws_raw.append(["Phenotype"] + replicate_cols)
style_header_row(ws_raw, 1)
for i, lbl in enumerate(PHENOTYPE_LABELS):
    ws_raw.append([lbl] + list(replicate_data[i]))
auto_width(ws_raw)

# Observed counts tab
ws_obs = wb.create_sheet("Observed_Counts")
ws_obs.append(["Phenotype", "Count", "Frequency"])
style_header_row(ws_obs, 1)
for i, lbl in enumerate(PHENOTYPE_LABELS):
    ws_obs.append([lbl, int(observed_counts[i]), round(observed_counts[i]/TOTAL_EVS, 6)])
ws_obs.append(["TOTAL", TOTAL_EVS, 1.0])
auto_width(ws_obs)

# Marginals tab
ws_marg = wb.create_sheet("Replicate_Marginals")
for r in dataframe_to_rows(df_rep_marg, index=False, header=True):
    ws_marg.append(r)
style_header_row(ws_marg, 1)
auto_width(ws_marg)

save_wb(wb, "Cell_1.03_Phenotype_Data", ts)
stop_logging(lf)


In [5]:
# Cell 1.04
"""
================================================================================
CELL 1.04: INDEPENDENCE ANALYSIS — CHI-SQUARE & CRAMÉR'S V
================================================================================
PURPOSE:
  Test whether each pair of tetraspanins (and all three together) assort
  independently in the pooled data (Option 1 / Population 1 only).

  For each pair:
    1. Build 2×2 contingency table from observed_counts
    2. Chi-square test of independence (scipy.stats.chi2_contingency)
    3. Cramér's V = sqrt(χ²/N) for 2×2 tables
    4. Odds ratio

  Three-way:
    - Cochran–Mantel–Haenszel approach via log-linear model goodness-of-fit
    - Mutual information / Three-way interaction term

OUTPUT:
  Excel: Cell_1.04_Independence_Analysis_<ts>.xlsx
    Tabs: Pairwise_Tests, ThreeWay_Test, Summary
  Debug: Cell_1.04_Debug_<ts>.txt
================================================================================
"""

lf, ts = start_logging("Cell_1.04")

print("=" * 70)
print("CELL 1.04: INDEPENDENCE ANALYSIS — CHI-SQUARE & CRAMÉR'S V")
print("=" * 70)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
print(f"PURPOSE: Test whether tetraspanins assort independently")
print(f"  H₀ (null hypothesis): Markers sort randomly (independent)")
print(f"  Significance threshold: α = {ALPHA}")
print(f"  Total EVs: {TOTAL_EVS:,}\n")

print("STEP 1: Marginal frequencies")
print("-" * 60)

obs = observed_counts.astype(float)
N   = TOTAL_EVS

# Helper: build 2×2 table for a pair of markers
def contingency_2x2(obs, idx_A_pos, idx_B_pos):
    """
    Returns 2×2 table:
      [[A+B+, A+B-],
       [A-B+, A-B-]]
    """
    AB_pp = obs[[i for i in idx_A_pos if i in idx_B_pos]].sum()
    AB_pn = obs[[i for i in idx_A_pos if i not in idx_B_pos]].sum()
    AB_np = obs[[i for i in idx_B_pos if i not in idx_A_pos]].sum()
    AB_nn = obs[[i for i in range(8) if i not in idx_A_pos and i not in idx_B_pos]].sum()
    return np.array([[AB_pp, AB_pn], [AB_np, AB_nn]])

PAIRS = [
    ("CD9",  "CD81", CD9_POS,  CD81_POS),
    ("CD9",  "CD63", CD9_POS,  CD63_POS),
    ("CD81", "CD63", CD81_POS, CD63_POS),
]

p_cd9_marg  = obs[CD9_POS].sum()  / N
p_cd81_marg = obs[CD81_POS].sum() / N
p_cd63_marg = obs[CD63_POS].sum() / N
print(f"  P(CD9+)  = {p_cd9_marg:.4f}  ({p_cd9_marg*100:.2f}%)")
print(f"  P(CD81+) = {p_cd81_marg:.4f}  ({p_cd81_marg*100:.2f}%)")
print(f"  P(CD63+) = {p_cd63_marg:.4f}  ({p_cd63_marg*100:.2f}%)")

print("\nSTEP 2: Pairwise 2×2 chi-square tests")
print("-" * 60)
print(f"  FORMULA: χ² = Σ(observed − expected)² / expected")
print(f"  Cramér's V = √(χ²/N)  for 2×2 tables\n")

pair_results = []
print("\nPairwise independence tests:")
print("-" * 70)

for mA, mB, idxA, idxB in PAIRS:
    ct = contingency_2x2(obs, idxA, idxB)
    chi2, p, dof, expected = chi2_contingency(ct, correction=False)
    cramers_v = np.sqrt(chi2 / N)         # For 2×2: V = sqrt(chi2/N)
    # Odds ratio
    with np.errstate(divide="ignore", invalid="ignore"):
        or_val = (ct[0,0]*ct[1,1]) / (ct[0,1]*ct[1,0]) if ct[0,1]*ct[1,0] > 0 else np.inf
    significance = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    result = {
        "Pair":          f"{mA} vs {mB}",
        "Marker_A":      mA,
        "Marker_B":      mB,
        "N++":           int(ct[0,0]),
        "N+-":           int(ct[0,1]),
        "N-+":           int(ct[1,0]),
        "N--":           int(ct[1,1]),
        "Chi2":          round(chi2, 4),
        "df":            int(dof),
        "p_value":       round(p, 8),
        "Significance":  significance,
        "Cramers_V":     round(cramers_v, 4),
        "Odds_Ratio":    round(or_val, 4),
        "Independent":   "Yes" if p >= ALPHA else "No",
    }
    pair_results.append(result)
    print(f"\n  {mA} vs {mB}:")
    print(f"    Contingency table (observed):")
    print(f"      {'':>10}  {mB}+    {mB}-")
    print(f"      {mA}+    {int(ct[0,0]):>6,}  {int(ct[0,1]):>6,}  → {int(ct[0,:].sum()):>6,}")
    print(f"      {mA}-    {int(ct[1,0]):>6,}  {int(ct[1,1]):>6,}  → {int(ct[1,:].sum()):>6,}")
    print(f"      Total  {int(ct[:,0].sum()):>6,}  {int(ct[:,1].sum()):>6,}  → {int(ct.sum()):>6,}")
    print(f"    χ²={chi2:.4f}  df={dof}  p={p:.2e}  {significance}")
    print(f"    Cramér's V = {cramers_v:.4f}   Odds Ratio = {or_val:.4f}")
    print(f"    → {'✓ INDEPENDENT (p ≥ α)' if p >= ALPHA else '✗ NOT independent — linkage detected (p < α)'}")

df_pairs = pd.DataFrame(pair_results)

# ── Three-way test ────────────────────────────────────────────────────────────
print("\n\nSTEP 3: Three-way independence test (CD9 × CD81 × CD63)")
print("-" * 70)
print("  METHOD: Pearson χ² goodness-of-fit vs. expected under full independence")
print(f"  FORMULA: E_i = N × P(CD9)^a × P(CD81)^b × P(CD63)^c\n")
print("  Observed vs. Expected under independence:")
print(f"  {'Phenotype':<10} {'Observed':>10} {'Expected':>10} {'Residual':>10} {'Std.Res':>9}")
print(f"  {'-'*52}")

# Expected under full independence
p_cd9  = obs[CD9_POS].sum()  / N
p_cd81 = obs[CD81_POS].sum() / N
p_cd63 = obs[CD63_POS].sum() / N

patterns = [(1,1,1),(1,1,0),(1,0,1),(1,0,0),(0,1,1),(0,1,0),(0,0,1),(0,0,0)]
expected_indep = np.array([
    N * (p_cd9 if a else 1-p_cd9) * (p_cd81 if b else 1-p_cd81) * (p_cd63 if c else 1-p_cd63)
    for a, b, c in patterns
])
chi2_3way = float(np.sum((obs - expected_indep)**2 / np.maximum(expected_indep, 1e-10)))
df_3way   = N_PHENOTYPES - 1 - N_MARKERS   # = 4
p_3way    = float(1 - chi2_dist.cdf(chi2_3way, df_3way))
sig_3way  = "***" if p_3way < 0.001 else ("**" if p_3way < 0.01 else ("*" if p_3way < 0.05 else "ns"))

# Cramér's V (8-category version)
cramers_v_3way = np.sqrt(chi2_3way / (N * (N_PHENOTYPES - 1)))

# Mutual information (bits)
mi_3way = 0.0
for i in range(8):
    if obs[i] > 0 and expected_indep[i] > 0:
        mi_3way += (obs[i]/N) * np.log2(obs[i] / expected_indep[i])

for i, lbl in enumerate(PHENOTYPE_LABELS):
    res_i   = obs[i] - expected_indep[i]
    std_r_i = res_i / max(np.sqrt(expected_indep[i]), 1)
    print(f"  {lbl:<10} {int(obs[i]):>10,} {expected_indep[i]:>10.1f} {res_i:>+10.1f} {std_r_i:>+9.3f}")
print(f"  {'-'*52}")

print(f"\n  Marginal frequencies used:")
print(f"    P(CD9+) ={p_cd9:>8.4f}  P(CD81+) ={p_cd81:>8.4f}  P(CD63+) ={p_cd63:>8.4f}")
print(f"\n  Chi-square (df={df_3way}): {chi2_3way:.4f}")
print(f"  p-value             : {p_3way:.2e}  {sig_3way}")
print(f"  Cramér's V          : {cramers_v_3way:.4f}")
print(f"  Mutual Information  : {mi_3way:.4f} bits")
print(f"  → {'✓ INDEPENDENT (p ≥ α)' if p_3way >= ALPHA else '✗ NOT independent — three-way linkage detected (p < α)'}")

print(f"\nSTEP 4: Summary")
print("-" * 60)
print(f"  {'Pair/Test':<30} {'χ²':>8}  {'p':>10}  {'Sig':>4}  {'V':>7}  {'Result':>14}")
print(f"  {'-'*72}")
for row in pair_results:
    print(f"  {row['Pair']:<30} {row['Chi2']:>8.3f}  {row['p_value']:>10.2e}  {row['Significance']:>4}  {row['Cramers_V']:>7.4f}  {'Independent' if row['Independent']=='Yes' else 'Linked':>14}")
print(f"  {'CD9×CD81×CD63 (3-way)':<30} {chi2_3way:>8.3f}  {p_3way:>10.2e}  {sig_3way:>4}  {cramers_v_3way:>7.4f}  {'Independent' if p_3way >= ALPHA else 'Linked':>14}")

three_way_result = {
    "Test":            "CD9 × CD81 × CD63 (3-way)",
    "Chi2":            round(chi2_3way, 4),
    "df":              df_3way,
    "p_value":         round(p_3way, 8),
    "Significance":    sig_3way,
    "Cramers_V":       round(cramers_v_3way, 4),
    "Mutual_Info_bits": round(mi_3way, 4),
    "Independent":     "Yes" if p_3way >= ALPHA else "No",
    "p_CD9":           round(p_cd9, 6),
    "p_CD81":          round(p_cd81, 6),
    "p_CD63":          round(p_cd63, 6),
}

# ── Excel output ──────────────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

# Pairwise sheet
ws_pw = wb.create_sheet("Pairwise_Tests")
cols_pw = list(df_pairs.columns)
ws_pw.append(cols_pw)
style_header_row(ws_pw, 1)
for _, row in df_pairs.iterrows():
    ws_pw.append([row[c] for c in cols_pw])
auto_width(ws_pw)

# Three-way sheet
ws_3w = wb.create_sheet("ThreeWay_Test")
ws_3w.append(["Parameter", "Value"])
style_header_row(ws_3w, 1)
for k, v in three_way_result.items():
    ws_3w.append([k, v])
auto_width(ws_3w)

# Observed vs Expected sheet
ws_oe = wb.create_sheet("Observed_vs_Expected")
ws_oe.append(["Phenotype", "Observed", "Expected_Indep", "Obs_Freq", "Exp_Freq", "Residual", "Std_Residual"])
style_header_row(ws_oe, 1)
for i, lbl in enumerate(PHENOTYPE_LABELS):
    res = obs[i] - expected_indep[i]
    std_res = res / np.sqrt(max(expected_indep[i], 1))
    ws_oe.append([lbl, int(obs[i]), round(expected_indep[i],2),
                  round(obs[i]/N,6), round(expected_indep[i]/N,6),
                  round(res,2), round(std_res,4)])
auto_width(ws_oe)

# Summary
ws_sum = wb.create_sheet("Summary")
ws_sum.append(["Test", "Chi2", "df", "p_value", "Significance", "Cramers_V", "Independent"])
style_header_row(ws_sum, 1)
for row in pair_results:
    ws_sum.append([row["Pair"], row["Chi2"], row["df"], row["p_value"],
                   row["Significance"], row["Cramers_V"], row["Independent"]])
ws_sum.append(["CD9×CD81×CD63 (3-way)", three_way_result["Chi2"], three_way_result["df"],
               three_way_result["p_value"], three_way_result["Significance"],
               three_way_result["Cramers_V"], three_way_result["Independent"]])
auto_width(ws_sum)

save_wb(wb, "Cell_1.04_Independence_Analysis", ts)

# ── Compute phi bounds from observed marginals ────────────────────────────────
# Theoretical bounds: phi_max(X,Y) = min(pX*(1-pY), pY*(1-pX))
#                     phi_min(X,Y) = max(-pX*pY,    -(1-pX)*(1-pY))
# These are the tightest possible bounds given the observed marginals.
# They replace fixed constants in make_bounds(), making the optimizer
# self-adapting to different experiments with different marginal frequencies.
def _phi_bounds(pA, pB):
    return max(-pA*pB, -(1-pA)*(1-pB)), min(pA*(1-pB), pB*(1-pA))

PHI_BOUNDS = {
    ("CD9",  "CD81"): _phi_bounds(p_cd9_marg,  p_cd81_marg),
    ("CD9",  "CD63"): _phi_bounds(p_cd9_marg,  p_cd63_marg),
    ("CD81", "CD63"): _phi_bounds(p_cd81_marg, p_cd63_marg),
}

# Also store the observed phi values from pooled data as starting-point references
PHI_OBS = {
    ("CD9",  "CD81"): obs[CD9_POS].sum()/N * obs[CD81_POS].sum()/N,   # placeholder
    ("CD9",  "CD63"): (obs[0]+obs[1])/N - p_cd9_marg*p_cd81_marg,     # will compute below
}
# Recalculate correctly
_p_cd9_cd81_pp = (obs[0]+obs[1]) / N   # P(CD9+, CD81+)
_p_cd9_cd63_pp = (obs[0]+obs[2]) / N   # P(CD9+, CD63+)
_p_cd81_cd63_pp= (obs[0]+obs[4]) / N   # P(CD81+,CD63+)
PHI_OBS = {
    ("CD9",  "CD81"): round(_p_cd9_cd81_pp - p_cd9_marg*p_cd81_marg, 6),
    ("CD9",  "CD63"): round(_p_cd9_cd63_pp - p_cd9_marg*p_cd63_marg, 6),
    ("CD81", "CD63"): round(_p_cd81_cd63_pp- p_cd81_marg*p_cd63_marg, 6),
}

print(f"\nPhi bounds computed from observed marginals:")
for pair, (lo, hi) in PHI_BOUNDS.items():
    obs_phi = PHI_OBS[pair]
    print(f"  phi{pair}: [{lo:+.6f}, {hi:+.6f}]  observed = {obs_phi:+.6f}")
print(f"\n✓ PHI_BOUNDS and PHI_OBS stored for use in Cell 1.05 make_bounds()")

stop_logging(lf)


In [6]:
# Cell 1.05
"""
================================================================================
CELL 1.05: MODEL ENGINE — DIRECT PROBABILITY MODEL FUNCTIONS
================================================================================
PURPOSE:
  Define all model functions shared by Cells 1.06 and 1.07.

  SCENARIO TAXONOMY (6 models)
  ┌─────────────┬──────────────────────────────────────────────────────────┐
  │  Scenario   │ Description                                             │
  ├─────────────┼──────────────────────────────────────────────────────────┤
  │  Option 1A  │ 0 sub-pops, single Pop A (independent assortment)       │
  │  Option 1B  │ 0 sub-pops, single Pop A (linked assortment)            │
  │  Option 2A  │ 1 sub-pop  (Pop B = CD63-high), independent             │
  │  Option 2B  │ 1 sub-pop  (Pop B = CD63-high), linked                  │
  │  Option 3A  │ 2 sub-pops (Pop B CD63-high + Pop C CD9-high), indep.   │
  │  Option 3B  │ 2 sub-pops (Pop B CD63-high + Pop C CD9-high), linked   │
  └─────────────┴──────────────────────────────────────────────────────────┘

  INDEPENDENT models: each marker's probability in each population is an
    independent free parameter; predicted P(phenotype) = product of marginals.

  LINKED models: add a linkage-disequilibrium (phi) parameter for each
    population.  phi is defined as:
      phi = P(CD9+,CD81+) − P(CD9+)·P(CD81+)
    and adjusts the 8-cell probability table via the standard LD correction.
    CD63 is treated as independent of the CD9–CD81 linkage within each pop
    (the CNV data supports this).

OUTPUTS:
  None (only defines functions; no file output)
================================================================================
"""

# Verify prerequisite cells have been run
require_cnv_data()
require_independence_analysis()

# phenotype patterns for all 8 combinations
# Order: [0]+/+/+  [1]+/+/-  [2]+/-/+  [3]+/-/-
#        [4]-/+/+  [5]-/+/-  [6]-/-/+  [7]-/-/-
_PATTERNS = np.array([
    [1,1,1],[1,1,0],[1,0,1],[1,0,0],
    [0,1,1],[0,1,0],[0,0,1],[0,0,0]
], dtype=float)  # columns: CD9, CD81, CD63

def probs_independent(p9, p81, p63):
    """
    Returns 8-vector of phenotype probabilities under full independence.
    p9, p81, p63: marginal P(marker+) for CD9, CD81, CD63.
    """
    probs = np.empty(8)
    for i, (a, b, c) in enumerate(_PATTERNS):
        probs[i] = ((p9 if a else 1-p9) *
                    (p81 if b else 1-p81) *
                    (p63 if c else 1-p63))
    return probs
    
def probs_popc(p81, p63, phi_81_63=0.0):
    """
    8-vector for Pop C (p9=1.0 fixed): all EVs are CD9+.
    phi_81_63 is the CD81–CD63 LD coefficient within Pop C:
      P(CD81+,CD63+) = p81*p63 + phi_81_63
    CD9- phenotypes (indices 4-7) are all zero by definition.
    """
    p81p63   = np.clip(p81*p63          + phi_81_63, 0, 1)
    p81np63  = np.clip(p81*(1-p63)      - phi_81_63, 0, 1)
    np81p63  = np.clip((1-p81)*p63      - phi_81_63, 0, 1)
    np81np63 = np.clip((1-p81)*(1-p63)  + phi_81_63, 0, 1)
    tot = p81p63 + p81np63 + np81p63 + np81np63
    if tot > 0:
        p81p63/=tot; p81np63/=tot; np81p63/=tot; np81np63/=tot
    return np.array([
        p81p63,    # +/+/+  (CD9+, CD81+, CD63+)
        p81np63,   # +/+/-  (CD9+, CD81+, CD63-)
        np81p63,   # +/-/+  (CD9+, CD81-, CD63+)
        np81np63,  # +/-/-  (CD9+, CD81-, CD63-)
        0.0, 0.0, 0.0, 0.0   # -/+/+, -/+/-, -/-/+, -/-/- all zero
    ])
    
def probs_linked(p9, p81, p63, phi_9_81, phi_9_63=0.0):
    """
    8-vector of phenotype probabilities with two LD (phi) parameters.
    ...
    Returns None if phi_9_81 exceeds the theoretical maximum given p9 and p81,
    rather than silently clipping. This surfaces the constraint violation to
    make_objective, which returns 1e12 and steers the optimizer away.
    """
    # ── Step 1: CD9–CD81 joint ────────────────────────────────────────────────
    # Check parametric feasibility BEFORE clipping.
    # phi_9_81 must satisfy: phi_9_81 <= min(p9*(1-p81), p81*(1-p9))
    #                        phi_9_81 >= max(-p9*p81, -(1-p9)*(1-p81))
    phi_ceil =  min(p9*(1-p81), p81*(1-p9))
    phi_floor = max(-p9*p81,   -(1-p9)*(1-p81))
    if phi_9_81 > phi_ceil * (1 + 1e-6) or phi_9_81 < phi_floor * (1 + 1e-6):
        return None   # parametric constraint violated — signal to objective function
    p9p81   = p9*p81        + phi_9_81
    p9np81  = p9*(1-p81)    - phi_9_81
    np9p81  = (1-p9)*p81    - phi_9_81
    np9np81 = (1-p9)*(1-p81)+ phi_9_81
    # Small numerical tolerances may produce tiny negatives; clip only those
    p9p81   = max(p9p81,   0.0)
    p9np81  = max(p9np81,  0.0)
    np9p81  = max(np9p81,  0.0)
    np9np81 = max(np9np81, 0.0)
    tot = p9p81 + p9np81 + np9p81 + np9np81
    if tot > 0:
        p9p81 /= tot; p9np81 /= tot; np9p81 /= tot; np9np81 /= tot

    # ── Step 2: CD63 conditional on CD9 status ────────────────────────────────
    p63_pos = np.clip(p63 + phi_9_63 / p9,       0, 1) if p9 > 0 else p63
    p63_neg = np.clip(p63 - phi_9_63 / (1-p9),   0, 1) if p9 < 1 else p63

    # ── Step 3: Build 8-cell table ────────────────────────────────────────────
    return np.array([
        p9p81   * p63_pos,      # +/+/+
        p9p81   * (1-p63_pos),  # +/+/-
        p9np81  * p63_pos,      # +/-/+
        p9np81  * (1-p63_pos),  # +/-/-
        np9p81  * p63_neg,      # -/+/+
        np9p81  * (1-p63_neg),  # -/+/-
        np9np81 * p63_neg,      # -/-/+
        np9np81 * (1-p63_neg),  # -/-/-
    ])

def mix_pops(*args):
    """
    Mix multiple population probability arrays.
    Args alternating: (probs_pop1, frac1, probs_pop2, frac2, ..., probs_popA)
    The final population's fraction = 1 - sum(other fracs).
    Usage:
      Option 2: mix_pops(probs_b, f_b, probs_a)
      Option 3: mix_pops(probs_b, f_b, probs_c, f_c, probs_a)
    """
    # Separate into (probs, frac) pairs + remainder
    pops = []
    i = 0
    while i < len(args) - 1:
        pops.append((args[i], args[i+1]))
        i += 2
    remainder_probs = args[-1]
    frac_remainder  = 1.0 - sum(f for _, f in pops)
    mixed = frac_remainder * remainder_probs
    for probs, frac in pops:
        mixed = mixed + frac * probs
    return mixed

# ── Objective function factory ────────────────────────────────────────────────
def _neg_log_lik(predicted_probs, obs_counts, total):
    """Negative multinomial log-likelihood (up to constant)."""
    pp = np.maximum(predicted_probs, 1e-15)
    pp = pp / pp.sum()
    return -float(np.dot(obs_counts, np.log(pp)))

def _regularization(params, ref_params, weights, n_obs=1):
    """
    Weighted L2 penalty: each parameter gets its own weight.
    penalty = n_obs * sum_i( weights[i] * (params[i] - ref[i])^2 )

    weights: array of per-parameter regularization strengths.
             0.0 for a parameter means no penalty applied.
             Use REG_FRAC for fB/fC, REG_MARG for marginals, REG_PHI for phi.
    """
    w = np.asarray(weights, dtype=float)
    diff = params - ref_params
    return float(n_obs * np.dot(w, diff ** 2))

def _chi2_gof(obs, expected, total):
    """Pearson chi-square goodness-of-fit."""
    exp = np.maximum(expected * total, 1e-10)
    return float(np.sum((obs - exp)**2 / exp))

def make_objective(option, linked, obs_counts, total, cd63_pos_total, cd9_pos_total,
                   pop_b_frac_of_cd63, pop_c_frac_of_cd9):
    """
    Return an objective function for scipy.optimize.

    Parameter layout by option / linkage:
      Option 1, indep:  p9_A, p81_A, p63_A                         → 3 params
      Option 1, linked: p9_A, p81_A, p63_A, phi_9_81_A, phi_9_63_A → 5 params
      Option 2, indep:  fB, p9_B, p81_B, p63_B, p9_A, p81_A, p63_A → 7 params
      Option 2, linked: fB, p9_B, p81_B, p63_B, phi_9_81_B, p9_A, p81_A, p63_A, phi_9_81_A, phi_9_63_A → 10 params
      Option 3, indep:  fB, p9_B, p81_B, p63_B,
                        fC, p9_C, p81_C, p63_C,
                        p9_A, p81_A, p63_A               → 11 params
      Option 3, linked: same + phi_B, phi_C, phi_A       → 14 params
    """
    def obj(params):
        try:
            pp, ref_p, weights = _decode_params(params, option, linked,
                                                obs_counts, total,
                                                cd63_pos_total, cd9_pos_total,
                                                pop_b_frac_of_cd63, pop_c_frac_of_cd9)
            if pp is None:
                return 1e12
            nll  = _neg_log_lik(pp, obs_counts, total)
            reg  = _regularization(params, ref_p, weights, n_obs=total)
            # ── Log barrier for phi parameters ────────────────────────────────
            # Applies an increasing penalty as phi approaches its theoretical
            # maximum (or minimum) given the CURRENT marginal probabilities.
            # Uses a symmetric two-sided log barrier:
            #   f(phi) = alpha * [-ln(1 - phi/phi_ceil) - ln(1 + phi/phi_floor)]
            # This gives gentle gradient pressure far from the boundary and
            # rapidly increasing cost near it, preventing:
            #   (a) phi hitting the ceiling (causing p9_B=p81_B degeneracy)
            #   (b) phi hitting the floor (equally degenerate, mutual exclusion extreme)
            # The parametric ceiling/floor updates at every function evaluation,
            # so the constraint adapts as the marginals change during optimisation.
            phi_barrier = 0.0
            if PHI_BARRIER_ALPHA > 0 and linked:
                def _log_barrier_phi(phi_val, p_a, p_b):
                    """Two-sided log barrier for a single phi parameter."""
                    phi_val = float(phi_val)
                    p_a = float(p_a); p_b = float(p_b)
                    phi_ceil  =  min(p_a * (1 - p_b), p_b * (1 - p_a))
                    phi_floor = -min(p_a * p_b, (1 - p_a) * (1 - p_b))
                    # Guard against numerical edge cases
                    if phi_ceil <= 0 or phi_floor >= 0:
                        return 0.0
                    margin_hi = phi_ceil  - phi_val
                    margin_lo = phi_val   - phi_floor
                    if margin_hi <= 0 or margin_lo <= 0:
                        return 1e6  # already outside feasible region
                    return PHI_BARRIER_ALPHA * (
                        -np.log(margin_hi / phi_ceil) -
                        np.log(margin_lo / abs(phi_floor))
                    )

                if option == 1:
                    # phi_9_81_A at position 3, phi_9_63_A at position 4
                    phi_barrier += _log_barrier_phi(params[3], params[0], params[1])
                    phi_barrier += _log_barrier_phi(params[4], params[0], params[2])

                elif option == 2:
                    # phi_9_81_B at position 4 (after fB, p9B, p81B, p63B)
                    phi_barrier += _log_barrier_phi(params[4], params[1], params[2])
                    # phi_9_81_A at position 8, phi_9_63_A at position 9
                    phi_barrier += _log_barrier_phi(params[8], params[5], params[6])
                    phi_barrier += _log_barrier_phi(params[9], params[5], params[7])

                elif option == 3:
                    # phi_9_81_B at position 5 (after fB, fC, p9B, p81B, p63B)
                    phi_barrier += _log_barrier_phi(params[5], params[2], params[3])
                    # phi_81_63_C at position 9 (after fC, p9C, p81C, p63C)
                    phi_barrier += _log_barrier_phi(params[9], params[6], params[7])
                    # phi_9_81_A at position 13, phi_9_63_A at position 14
                    phi_barrier += _log_barrier_phi(params[13], params[10], params[11])
                    phi_barrier += _log_barrier_phi(params[14], params[10], params[12])

            # ── Sub-population divergence penalty ─────────────────────────────
            # Penalizes Pop B/C marginals that diverge far from Pop A values.
            # Uses the CURRENT Pop A values (from params) as the reference,
            # so this is a relative shrinkage — it relaxes as Pop A moves.
            # Formula: REG_SUBPOP_DIVERGE * N * sum((p_subpop - p_A)^2)
            diverge_penalty = 0.0
            if REG_SUBPOP_DIVERGE > 0:
                if option == 2:
                    # params layout: fB(0), p9B(1), p81B(2), p63B(3), [phi_B(4)],
                    #                p9A(4or5), p81A(5or6), p63A(6or7)
                    _off = 5 if linked else 4   # offset to Pop A params
                    diverge_penalty += REG_SUBPOP_DIVERGE * total * (
                        (params[1] - params[_off])**2 +      # p9_B vs p9_A
                        (params[2] - params[_off + 1])**2    # p81_B vs p81_A
                    )
                elif option == 3:
                    # params layout (linked):
                    #   fB(0), fC(1), p9B(2), p81B(3), p63B(4), phi_B(5),
                    #   p9C(6), p81C(7), p63C(8), phi_C(9),
                    #   p9A(10), p81A(11), p63A(12), phi9_81A(13), phi9_63A(14)
                    # params layout (independent):
                    #   fB(0), fC(1), p9B(2), p81B(3), p63B(4),
                    #   p9C(5), p81C(6), p63C(7),
                    #   p9A(8), p81A(9), p63A(10)
                    if linked:
                        _p9A, _p81A, _p63A = params[10], params[11], params[12]
                        _p9B, _p81B        = params[2],  params[3]
                        _p81C, _p63C       = params[7],  params[8]
                    else:
                        _p9A, _p81A, _p63A = params[8],  params[9],  params[10]
                        _p9B, _p81B        = params[2],  params[3]
                        _p81C, _p63C       = params[6],  params[7]
                    diverge_penalty += REG_SUBPOP_DIVERGE * total * (
                        (_p9B  - _p9A )**2 +   # p9_B  vs p9_A
                        (_p81B - _p81A)**2 +   # p81_B vs p81_A
                        (_p81C - _p81A)**2 +   # p81_C vs p81_A
                        (_p63C - _p63A)**2      # p63_C vs p63_A
                    )
            return nll + reg + phi_barrier + diverge_penalty
        except Exception as _obj_exc:
            import traceback as _tb
            print(f"\n⚠ OBJECTIVE FUNCTION ERROR (option={option}, linked={linked}):")
            print(f"  {type(_obj_exc).__name__}: {_obj_exc}")
            _tb.print_exc()
            return 1e12
    return obj

def _decode_params(params, option, linked,
                   obs_counts, total, cd63_pos_total, cd9_pos_total,
                   pop_b_frac_of_cd63, pop_c_frac_of_cd9):
    """Decode flat param vector → predicted 8-probabilities."""
    obs_freq = obs_counts / total
    idx = 0

    def _get(n):
        nonlocal idx
        v = params[idx:idx+n]
        idx += n
        return v

    if option == 1:
        p9_A, p81_A, p63_A = _get(3)
        phi_9_81_A, phi_9_63_A = (_get(1)[0], _get(1)[0]) if linked else (0.0, 0.0)
        if not linked:
            pp = probs_independent(p9_A, p81_A, p63_A)
        else:
            pp = probs_linked(p9_A, p81_A, p63_A, phi_9_81_A, phi_9_63_A)
        # Option 1 has no fB/fC — no CNV prior — so all weights are 0
        if linked:
            ref_p   = np.array([0.0, 0.0, 0.0, 0.0, 0.0])
            weights = np.array([REG_MARG, REG_MARG, REG_MARG, REG_PHI, REG_PHI])
        else:
            ref_p   = np.array([0.0, 0.0, 0.0])
            weights = np.array([REG_MARG, REG_MARG, REG_MARG])

    elif option == 2:
        fB_raw, = _get(1)
        fB_of_cd63 = float(np.clip(fB_raw, 0.001, 0.999))
        fB = fB_of_cd63 * cd63_pos_total / total
        p9_B, p81_B, p63_B = _get(3)
        # phi_9_63_B = 0 always: p63_B=1.0, so P(CD9+,CD63+)-P(CD9+)*1.0 = 0
        phi_9_81_B = _get(1)[0] if linked else 0.0
        phi_9_63_B = 0.0
        p9_A, p81_A, p63_A = _get(3)
        phi_9_81_A, phi_9_63_A = (_get(1)[0], _get(1)[0]) if linked else (0.0, 0.0)
        if not linked:
            prob_B = probs_independent(p9_B, p81_B, 1.0)
            prob_A = probs_independent(p9_A, p81_A, p63_A)
        else:
            prob_B = probs_linked(p9_B, p81_B, 1.0,   phi_9_81_B, phi_9_63_B)
            prob_A = probs_linked(p9_A, p81_A, p63_A, phi_9_81_A, phi_9_63_A)
            if prob_B is None or prob_A is None:
                return None, None, None
        pp = mix_pops(prob_B, fB, prob_A)
        # fB_of_cd63 pulled toward CNV prior; everything else unpenalized
        if linked:
            ref_p   = np.array([pop_b_frac_of_cd63,
                                 0., 0., 0., 0.,    # p9_B p81_B p63_B phi_9_81_B
                                 0., 0., 0., 0., 0.]) # p9_A p81_A p63_A phi_9_81_A phi_9_63_A
            weights = np.array([REG_FRAC,
                                 REG_MARG, REG_MARG, REG_MARG, REG_PHI,
                                 REG_MARG, REG_MARG, REG_MARG, REG_PHI, REG_PHI])
        else:
            ref_p   = np.array([pop_b_frac_of_cd63,
                                 0., 0., 0.,    # p9_B p81_B p63_B
                                 0., 0., 0.])   # p9_A p81_A p63_A
            weights = np.array([REG_FRAC,
                                 REG_MARG, REG_MARG, REG_MARG,
                                 REG_MARG, REG_MARG, REG_MARG])

    elif option == 3:
        fB_raw, = _get(1)
        fC_raw, = _get(1)
        fB_of_cd63 = float(np.clip(fB_raw, 0.001, 0.999))
        fC_of_cd9  = float(np.clip(fC_raw, 0.001, 0.999))
        fB = fB_of_cd63 * cd63_pos_total / total
        fC = fC_of_cd9  * cd9_pos_total  / total
        p9_B, p81_B, p63_B = _get(3)
        # phi_9_63_B = 0: p63_B=1.0
        phi_9_81_B = _get(1)[0] if linked else 0.0
        phi_9_63_B = 0.0
        p9_C, p81_C, p63_C = _get(3)
        # phi_9_81_C = phi_9_63_C = 0: p9_C=1.0 makes both identically zero
        phi_81_63_C = _get(1)[0] if linked else 0.0
        # phi_9_81_C and phi_9_63_C remain identically 0 (p9_C=1.0)
        p9_A, p81_A, p63_A = _get(3)
        phi_9_81_A, phi_9_63_A = (_get(1)[0], _get(1)[0]) if linked else (0.0, 0.0)
        if not linked:
            prob_B = probs_independent(p9_B, p81_B, 1.0)
            prob_C = probs_independent(1.0,  p81_C, p63_C)
            prob_A = probs_independent(p9_A, p81_A, p63_A)
        else:
            prob_B = probs_linked(p9_B, p81_B, 1.0,   phi_9_81_B, phi_9_63_B)
            prob_C = probs_popc(p81_C, p63_C, phi_81_63_C)
            prob_A = probs_linked(p9_A, p81_A, p63_A, phi_9_81_A, phi_9_63_A)
            if prob_B is None or prob_A is None:
                return None, None, None
        pp = mix_pops(prob_B, fB, prob_C, fC, prob_A)
        # fB and fC pulled toward CNV priors; everything else unpenalized
        if linked:
            ref_p   = np.array([pop_b_frac_of_cd63, pop_c_frac_of_cd9,
                                 0., 0., 0., 0.,   # p9_B p81_B p63_B phi_9_81_B
                                 0., 0., 0.,        # p9_C p81_C p63_C
                                 0.,                # phi_81_63_C
                                 0., 0., 0., 0., 0.]) # p9_A p81_A p63_A phi_9_81_A phi_9_63_A
            weights = np.array([REG_FRAC, REG_FRAC,
                                 REG_MARG, REG_MARG, REG_MARG, REG_PHI,
                                 REG_MARG, REG_MARG, REG_MARG,
                                 REG_PHI,
                                 REG_MARG, REG_MARG, REG_MARG, REG_PHI, REG_PHI])
        else:
            ref_p   = np.array([pop_b_frac_of_cd63, pop_c_frac_of_cd9,
                                 0., 0., 0.,    # p9_B p81_B p63_B
                                 0., 0., 0.,    # p9_C p81_C p63_C
                                 0., 0., 0.])   # p9_A p81_A p63_A
            weights = np.array([REG_FRAC, REG_FRAC,
                                 REG_MARG, REG_MARG, REG_MARG,
                                 REG_MARG, REG_MARG, REG_MARG,
                                 REG_MARG, REG_MARG, REG_MARG])
    else:
        raise ValueError(f"Unknown option: {option}")

    if pp is None or np.any(np.isnan(pp)) or np.any(pp < 0):
        return None, None, None
    # Soft constraint: penalize phi values that exceed the theoretical maximum
    # given the CURRENT marginals. This prevents the degenerate symmetric solution
    # where phi is at the ceiling and p9_B = p81_B becomes arbitrary.
    # Only applied when option >= 2 (Pop B exists) and linked.
    if linked and option >= 2 and 'p9_B' in dir():
        # _get has already been called; retrieve from local scope
        pass  # penalty is applied in make_objective via the violation term below
    return pp, ref_p, weights

def make_bounds(option, linked, pop_b_frac_of_cd63, pop_b_sd, pop_c_frac_of_cd9, pop_c_sd):
    """Return (lower, upper) bound arrays for the parameter vector.

    Bounds for population size fractions use mean ± 1 SD from the CNV
    replicate data, which directly represents the biological measurement
    uncertainty. This is tighter and more principled than the previous ±2×CV.
    """
    eps, hi = BOUND_EPS, BOUND_MAX
    # Use data-driven bounds from Cell 1.04 if available; fall back to config constants
    if 'PHI_BOUNDS' in globals():
        phi_9_81_bound  = PHI_BOUNDS.get(("CD9","CD81"),  (PHI_BOUND_LO_FALLBACK, PHI_BOUND_HI_FALLBACK))
        phi_9_63_bound  = PHI_BOUNDS.get(("CD9","CD63"),  (PHI_BOUND_LO_FALLBACK, PHI_BOUND_HI_FALLBACK))
        phi_81_63_bound = PHI_BOUNDS.get(("CD81","CD63"), (PHI_BOUND_LO_FALLBACK, PHI_BOUND_HI_FALLBACK))
    else:
        phi_9_81_bound = phi_9_63_bound = phi_81_63_bound = (PHI_BOUND_LO_FALLBACK, PHI_BOUND_HI_FALLBACK)
    # Add a minimum positivity constraint for Pop B marginals to prevent degenerate solutions.
    # Pop B is the CD63-high population — biologically it is expected to have
    # substantial CD9 and CD81 expression. A floor of 5% prevents the optimizer
    # from collapsing p9_B and p81_B to near-zero, which creates phi bound violations
    # and biologically implausible solutions.
    POP_B_MARG_MIN = 0.05   # minimum 5% positivity for free marginals in Pop B

    two_phi   = [phi_9_81_bound, phi_9_63_bound]  # Pop A: both phi free
    one_phi_B = [phi_9_81_bound]                  # Pop B: only phi_9_81 free
    # Pop C: no phi parameters (p9_C=1.0 makes both phi identically zero)

    if option == 1:
        base = [(eps, hi)] * 3
        phi  = two_phi if linked else []
        return list(zip(*(base + phi)))

    b_lo = max(eps, pop_b_frac_of_cd63 - pop_b_sd)
    b_hi = min(hi,  pop_b_frac_of_cd63 + pop_b_sd)
    c_lo = max(eps, pop_c_frac_of_cd9  - pop_c_sd)
    c_hi = min(hi,  pop_c_frac_of_cd9  + pop_c_sd)

    if option == 2:
        base = ([(b_lo, b_hi)] +                         # fB_of_cd63
                [(POP_B_MARG_MIN, hi), (POP_B_MARG_MIN, hi), (eps, hi)] +  # p9_B, p81_B, p63_B
                (one_phi_B if linked else []) +           # phi_9_81_B only
                [(eps, hi)] * 3 +                        # p9_A, p81_A, p63_A
                (two_phi   if linked else []))            # phi_9_81_A, phi_9_63_A
        return list(zip(*base))

    if option == 3:
        # phi_81_63_bound already computed above from PHI_BOUNDS or fallback
        one_phi_C = [phi_81_63_bound]
        base = ([(b_lo, b_hi)] +                                          # fB_of_cd63
                [(c_lo, c_hi)] +                                          # fC_of_cd9
                [(POP_B_MARG_MIN, hi), (POP_B_MARG_MIN, hi), (eps, hi)] +# p9_B, p81_B, p63_B
                (one_phi_B if linked else []) +                           # phi_9_81_B only
                [(eps, hi)] * 3 +                                         # p9_C, p81_C, p63_C
                (one_phi_C if linked else []) +                           # phi_81_63_C only
                [(eps, hi)] * 3 +                                         # p9_A, p81_A, p63_A
                (two_phi   if linked else []))                            # phi_9_81_A, phi_9_63_A
        return list(zip(*base))

def param_count(option, linked):
    base = {1:3, 2:7, 3:11}[option]
    # Pop A: 2 free phi (phi_9_81, phi_9_63)
    # Pop B: 1 free phi (phi_9_81 only; phi_9_63=0 because p63_B=1.0)
    # Pop C: 1 free phi (phi_81_63 only; phi_9_81=phi_9_63=0 because p9_C=1.0)
    phi_count = {1:2, 2:3, 3:4}[option]
    return base + (phi_count if linked else 0)

def decode_result(params, option, linked, obs_counts, total,
                  cd63_pos_total, cd9_pos_total,
                  pop_b_frac_of_cd63, pop_c_frac_of_cd9):
    """Return a dict with all interpreted parameters from a result vector."""
    obs_freq = obs_counts / total
    idx = 0
    def _g(n):
        nonlocal idx; v=params[idx:idx+n]; idx+=n; return v

    out = {}
    if option == 1:
        p9A, p81A, p63A = _g(3)
        phi_9_81_A, phi_9_63_A = (_g(1)[0], _g(1)[0]) if linked else (0.0, 0.0)
        out.update({"p9_A":p9A,"p81_A":p81A,"p63_A":p63A,
                    "phi_9_81_A":phi_9_81_A,"phi_9_63_A":phi_9_63_A})
        pp = probs_linked(p9A,p81A,p63A,phi_9_81_A,phi_9_63_A) if linked else probs_independent(p9A,p81A,p63A)
        out["pop_fracs"] = {"A":1.0}

    elif option == 2:
        fB_raw = _g(1)[0]
        fB_of_cd63 = float(np.clip(fB_raw, 0.001, 0.999))
        fB = fB_of_cd63 * cd63_pos_total / total
        p9B, p81B, p63B = _g(3)
        phi_9_81_B = _g(1)[0] if linked else 0.0
        phi_9_63_B = 0.0   # fixed: p63_B=1.0 forces phi_9_63_B=0
        p9A, p81A, p63A = _g(3)
        phi_9_81_A, phi_9_63_A = (_g(1)[0], _g(1)[0]) if linked else (0.0, 0.0)
        out.update({"fB_of_cd63":fB_of_cd63,"fB":fB,
                    "p9_B":p9B,"p81_B":p81B,"p63_B":1.0,
                    "phi_9_81_B":phi_9_81_B,"phi_9_63_B":phi_9_63_B,
                    "p9_A":p9A,"p81_A":p81A,"p63_A":p63A,
                    "phi_9_81_A":phi_9_81_A,"phi_9_63_A":phi_9_63_A})
        pB = probs_linked(p9B,p81B,1.0,  phi_9_81_B,phi_9_63_B) if linked else probs_independent(p9B,p81B,1.0)
        pA = probs_linked(p9A,p81A,p63A, phi_9_81_A,phi_9_63_A) if linked else probs_independent(p9A,p81A,p63A)
        pp = mix_pops(pB, fB, pA)
        out["pop_fracs"] = {"B":fB, "A":1-fB}

    elif option == 3:
        fB_raw = _g(1)[0]
        fC_raw = _g(1)[0]
        fB_of_cd63 = float(np.clip(fB_raw, 0.001, 0.999))
        fC_of_cd9  = float(np.clip(fC_raw, 0.001, 0.999))
        fB = fB_of_cd63 * cd63_pos_total / total
        fC = fC_of_cd9  * cd9_pos_total  / total
        p9B, p81B, p63B = _g(3)
        phi_9_81_B = _g(1)[0] if linked else 0.0
        phi_9_63_B = 0.0   # fixed: p63_B=1.0 forces phi_9_63_B=0
        p9C, p81C, p63C = _g(3)
        phi_9_81_C = 0.0   # fixed: p9_C=1.0 forces phi_9_81_C=0
        phi_9_63_C = 0.0   # fixed: p9_C=1.0 forces phi_9_63_C=0
        phi_81_63_C = _g(1)[0] if linked else 0.0   # free: both CD81 and CD63 vary in Pop C
        p9A, p81A, p63A = _g(3)
        phi_9_81_A, phi_9_63_A = (_g(1)[0], _g(1)[0]) if linked else (0.0, 0.0)
        out.update({"fB_of_cd63":fB_of_cd63,"fB":fB,
                    "fC_of_cd9":fC_of_cd9,"fC":fC,
                    "p9_B":p9B,"p81_B":p81B,"p63_B":1.0,
                    "phi_9_81_B":phi_9_81_B,"phi_9_63_B":phi_9_63_B,
                    "p9_C":1.0,"p81_C":p81C,"p63_C":p63C,
                    "phi_9_81_C":phi_9_81_C,"phi_9_63_C":phi_9_63_C,
                    "phi_81_63_C":phi_81_63_C,
                    "p9_A":p9A,"p81_A":p81A,"p63_A":p63A,
                    "phi_9_81_A":phi_9_81_A,"phi_9_63_A":phi_9_63_A})
        pB = probs_linked(p9B,p81B,1.0,  phi_9_81_B,phi_9_63_B) if linked else probs_independent(p9B,p81B,1.0)
        pC = probs_popc(p81C, p63C, phi_81_63_C) if linked else probs_independent(1.0,p81C,p63C)
        pA = probs_linked(p9A,p81A,p63A, phi_9_81_A,phi_9_63_A) if linked else probs_independent(p9A,p81A,p63A)
        pp = mix_pops(pB, fB, pC, fC, pA)
        out["pop_fracs"] = {"B":fB,"C":fC,"A":max(0,1-fB-fC)}

    pp = np.maximum(pp, 0); pp /= pp.sum()
    out["predicted_probs"] = pp
    # Store per-population probability arrays for Population_Data tab
    # These are the unmixed phenotype distributions for each sub-population
    if option == 1:
        out["pop_probs"] = {"A": pp}  # single pop = same as mixed
    elif option == 2:
        out["pop_probs"] = {"B": np.maximum(pB,0)/max(np.maximum(pB,0).sum(),1e-15),
                            "A": np.maximum(pA,0)/max(np.maximum(pA,0).sum(),1e-15)}
    elif option == 3:
        out["pop_probs"] = {"B": np.maximum(pB,0)/max(np.maximum(pB,0).sum(),1e-15),
                            "C": np.maximum(pC,0)/max(np.maximum(pC,0).sum(),1e-15),
                            "A": np.maximum(pA,0)/max(np.maximum(pA,0).sum(),1e-15)}
    out["chi2_gof"] = _chi2_gof(obs_counts, pp, total)
    out["nll"]      = _neg_log_lik(pp, obs_counts, total)
    # AIC: 2k - 2*log-likelihood
    k = param_count(option, linked)
    ll = -out["nll"]
    out["AIC"]      = 2*k - 2*ll
    out["BIC"]      = k*np.log(total) - 2*ll
    out["n_params"] = k
    return out

# ── Write importable module for parallel DE workers ───────────────────────────
# differential_evolution(workers=-1) uses multiprocessing, which requires the
# objective function to be picklable. Functions defined in a Jupyter kernel are
# NOT picklable. Solution: write the model engine to a .py file and import it.
import multiprocessing as _mp
# NOTE: Do NOT call set_start_method('fork') in Jupyter on WSL/Windows.
# Fork duplicates ZMQ sockets and deadlocks the kernel.
# DE_WORKERS=1 in Cell 1.01 avoids multiprocessing entirely.
print(f"  Multiprocessing start method: {_mp.get_start_method()} (not changed — DE_WORKERS=1)")

_ev_obj_path = os.path.join(DATA_DIR, 'ev_objective_rw.py')

_MODULE_SOURCE = '''
import numpy as np
REGULARIZATION_WEIGHT = {reg_weight}
REG_FRAC = {reg_frac}
REG_MARG = {reg_marg}
REG_PHI  = {reg_phi}
BOUND_EPS = {bound_eps}

_PATTERNS = np.array([[1,1,1],[1,1,0],[1,0,1],[1,0,0],
                       [0,1,1],[0,1,0],[0,0,1],[0,0,0]], dtype=float)

def probs_independent(p9, p81, p63):
    probs = np.empty(8)
    for i, (a,b,c) in enumerate(_PATTERNS):
        probs[i] = (p9 if a else 1-p9)*(p81 if b else 1-p81)*(p63 if c else 1-p63)
    return probs

def probs_linked(p9, p81, p63, phi_9_81, phi_9_63=0.0):
    p9p81=np.clip(p9*p81+phi_9_81,0,1); p9np81=np.clip(p9*(1-p81)-phi_9_81,0,1)
    np9p81=np.clip((1-p9)*p81-phi_9_81,0,1); np9np81=np.clip((1-p9)*(1-p81)+phi_9_81,0,1)
    tot=p9p81+p9np81+np9p81+np9np81
    if tot>0: p9p81/=tot; p9np81/=tot; np9p81/=tot; np9np81/=tot
    p63p=np.clip(p63+phi_9_63/p9,0,1) if p9>0 else p63
    p63n=np.clip(p63-phi_9_63/(1-p9),0,1) if p9<1 else p63
    return np.array([p9p81*p63p,p9p81*(1-p63p),p9np81*p63p,p9np81*(1-p63p),
                     np9p81*p63n,np9p81*(1-p63n),np9np81*p63n,np9np81*(1-p63n)])

def mix_pops(*args):
    pops=[]
    i=0
    while i<len(args)-1: pops.append((args[i],args[i+1])); i+=2
    mixed=(1-sum(f for _,f in pops))*args[-1]
    for probs,frac in pops: mixed=mixed+frac*probs
    return mixed

def _neg_log_lik(pp, obs, total):
    pp=np.maximum(pp,1e-15); pp/=pp.sum()
    return -float(np.dot(obs,np.log(pp)))

def _reg(params, ref, weights, n_obs=1):
    w = np.asarray(weights, dtype=float)
    return float(n_obs * np.dot(w, (params-ref)**2))

def _decode_and_predict(params, option, linked, obs, total, cd63_pos, cd9_pos, fB_cd63, fC_cd9):
    idx=0
    def g(n):
        nonlocal idx; v=params[idx:idx+n]; idx+=n; return v
    if option==1:
        p9A,p81A,p63A=g(3)
        phi81A,phi63A=(g(1)[0],g(1)[0]) if linked else (0.0,0.0)
        pp=probs_linked(p9A,p81A,p63A,phi81A,phi63A) if linked else probs_independent(p9A,p81A,p63A)
        if linked:
            ref=np.array([0.,0.,0.,0.,0.]); w=np.array([REG_MARG,REG_MARG,REG_MARG,REG_PHI,REG_PHI])
        else:
            ref=np.array([0.,0.,0.]); w=np.array([REG_MARG,REG_MARG,REG_MARG])
    elif option==2:
        fB_raw=g(1)[0]; fB=np.clip(fB_raw,{bound_eps},{bound_max})*cd63_pos/total
        p9B,p81B,p63B=g(3)
        phi81B=g(1)[0] if linked else 0.0; phi63B=0.0
        p9A,p81A,p63A=g(3)
        phi81A,phi63A=(g(1)[0],g(1)[0]) if linked else (0.0,0.0)
        pB=probs_linked(p9B,p81B,1.0,phi81B,phi63B) if linked else probs_independent(p9B,p81B,1.0)
        pA=probs_linked(p9A,p81A,p63A,phi81A,phi63A) if linked else probs_independent(p9A,p81A,p63A)
        pp=mix_pops(pB,fB,pA)
        if linked:
            ref=np.array([fB_cd63,0.,0.,0.,0.,0.,0.,0.,0.,0.])
            w=np.array([REG_FRAC,REG_MARG,REG_MARG,REG_MARG,REG_PHI,REG_MARG,REG_MARG,REG_MARG,REG_PHI,REG_PHI])
        else:
            ref=np.array([fB_cd63,0.,0.,0.,0.,0.,0.])
            w=np.array([REG_FRAC,REG_MARG,REG_MARG,REG_MARG,REG_MARG,REG_MARG,REG_MARG])
    else:
        fB_raw=g(1)[0]; fC_raw=g(1)[0]
        fB=np.clip(fB_raw,{bound_eps},{bound_max})*cd63_pos/total
        fC=np.clip(fC_raw,{bound_eps},{bound_max})*cd9_pos/total
        p9B,p81B,p63B=g(3)
        phi81B=g(1)[0] if linked else 0.0; phi63B=0.0
        p9C,p81C,p63C=g(3)
        phi81C=0.0; phi63C=0.0
        p9A,p81A,p63A=g(3)
        phi81A,phi63A=(g(1)[0],g(1)[0]) if linked else (0.0,0.0)
        pB=probs_linked(p9B,p81B,1.0,phi81B,phi63B) if linked else probs_independent(p9B,p81B,1.0)
        pC=probs_linked(1.0,p81C,p63C,phi81C,phi63C) if linked else probs_independent(1.0,p81C,p63C)
        pA=probs_linked(p9A,p81A,p63A,phi81A,phi63A) if linked else probs_independent(p9A,p81A,p63A)
        pp=mix_pops(pB,fB,pC,fC,pA)
        if linked:
            ref=np.array([fB_cd63,fC_cd9,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.])
            w=np.array([REG_FRAC,REG_FRAC,REG_MARG,REG_MARG,REG_MARG,REG_PHI,REG_MARG,REG_MARG,REG_MARG,REG_PHI,REG_MARG,REG_MARG,REG_MARG,REG_PHI,REG_PHI])
        else:
            ref=np.array([fB_cd63,fC_cd9,0.,0.,0.,0.,0.,0.,0.,0.,0.])
            w=np.array([REG_FRAC,REG_FRAC,REG_MARG,REG_MARG,REG_MARG,REG_MARG,REG_MARG,REG_MARG,REG_MARG,REG_MARG,REG_MARG])
    if np.any(np.isnan(pp)) or np.any(pp<0): return None,None,None
    return pp,ref,w

def make_obj_for_worker(option, linked, obs, total, cd63_pos, cd9_pos, fB_cd63, fC_cd9):
    def obj(params):
        try:
            pp,ref,w=_decode_and_predict(params,option,linked,obs,total,cd63_pos,cd9_pos,fB_cd63,fC_cd9)
            if pp is None: return 1e12
            return _neg_log_lik(pp,obs,total)+_reg(params,ref,w,total)
        except Exception as e:
            import sys as _sys
            print(f"\n⚠ WORKER OBJECTIVE ERROR (option={{option}}, linked={{linked}}): {{type(e).__name__}}: {{e}}", file=_sys.stderr)
            return 1e12
    return obj
'''.format(reg_weight=REGULARIZATION_WEIGHT, reg_frac=REG_FRAC, reg_marg=REG_MARG,
           reg_phi=REG_PHI, bound_eps=BOUND_EPS, bound_max=BOUND_MAX)
try:
    with open(_ev_obj_path, 'w') as _f:
        _f.write(_MODULE_SOURCE)
    if DATA_DIR not in sys.path:
        sys.path.insert(0, DATA_DIR)
    import importlib.util as _ilu
    _spec = _ilu.spec_from_file_location('ev_objective_rw', _ev_obj_path)
    _ev_mod = _ilu.module_from_spec(_spec)
    _spec.loader.exec_module(_ev_mod)
    sys.modules['ev_objective_rw'] = _ev_mod
    print(f"  ✓ Importable objective written: {_ev_obj_path}")
    _PARALLEL_DE_AVAILABLE = True
except Exception as _e:
    print(f"  ⚠ Could not write ev_objective_rw.py: {_e}")
    print(f"    DE will run with workers=1 (slower but safe)")
    _PARALLEL_DE_AVAILABLE = False

print("=" * 70)
print("CELL 1.05: MODEL ENGINE DEFINED")
print("=" * 70)
print("  Functions: probs_independent, probs_linked, mix_pops,")
print("             make_objective, make_bounds, decode_result")
print("  6 model scenarios: Options 1/2/3 × Independent/Linked")
print(f"  Parallel DE available: {_PARALLEL_DE_AVAILABLE}")


In [7]:
# Cell 1.06
"""
================================================================================
CELL 1.06: INDEPENDENT ASSORTMENT MODELS — OPTIONS 1, 2, 3
================================================================================
PURPOSE:
  Fit direct-probability models for each of the three population options
  under the assumption of INDEPENDENT assortment.
  Perform bootstrapping (N_BOOTSTRAP samples) to estimate 95% CIs.
  Report chi-square GOF, AIC, BIC as robustness metrics.

  Option 1A: Single Pop A, markers independent
  Option 2A: Pop B (CD63-high) + Pop A, markers independent in each pop
  Option 3A: Pop B (CD63-high) + Pop C (CD9-high) + Pop A, independent

OUTPUT:
  Excel: Cell_1.06_Independent_Models_<ts>.xlsx
    Tabs: Option1A_Params, Option2A_Params, Option3A_Params,
          Option1A_Phenotypes, Option2A_Phenotypes, Option3A_Phenotypes,
          Model_Comparison
  Debug: Cell_1.06_Debug_<ts>.txt
================================================================================
"""

lf, ts = start_logging("Cell_1.06")

print("=" * 70)
print("CELL 1.06: INDEPENDENT ASSORTMENT MODELS (OPTIONS 1A, 2A, 3A)")
print("=" * 70)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
print("CONFIGURATION")
print("-" * 60)
print(f"  Total EVs         : {TOTAL_EVS:,}")
print(f"  CD63+ EVs         : {CD63_POSITIVE_EVS:,}  ({CD63_POSITIVE_EVS/TOTAL_EVS*100:.2f}%)")
print(f"  CD9+  EVs         : {CD9_POSITIVE_EVS:,}  ({CD9_POSITIVE_EVS/TOTAL_EVS*100:.2f}%)")
print(f"  Pop B (of CD63+)  : {POP_B_FRAC_OF_CD63*100:.2f}% ± CV={POP_B_FRAC_CV:.4f}")
print(f"  Pop C (of CD9+)   : {POP_C_FRAC_OF_CD9*100:.2f}% ± CV={POP_C_FRAC_CV:.4f}")
print(f"  Regularization    : REG_FRAC={REG_FRAC}  REG_MARG={REG_MARG}  REG_PHI={REG_PHI}")
print(f"  DE maxiter        : {DE_MAXITER}  popsize={DE_POPSIZE}")
print(f"  Bootstrap samples : {N_BOOTSTRAP:,}")
print(f"  Random seed       : {RANDOM_SEED}")
print(f"  Assortment type   : INDEPENDENT (no phi)\n")

np.random.seed(RANDOM_SEED)
_cell_start = time.time()

OPTIONS = [1, 2, 3]
LINKED  = False

indep_results = {}   # keyed by option number

for opt in OPTIONS:
    print(f"\n{'─'*60}")
    print(f"  OPTION {opt}A — Independent assortment")
    print(f"{'─'*60}")

    n_params = param_count(opt, LINKED)
    bounds_lo, bounds_hi = make_bounds(opt, LINKED,
                                        POP_B_FRAC_OF_CD63, POP_B_FRAC_SD,
                                        POP_C_FRAC_OF_CD9,  POP_C_FRAC_SD)
    bounds_scipy = list(zip(bounds_lo, bounds_hi))
    obj = make_objective(opt, LINKED, observed_counts, TOTAL_EVS,
                         CD63_POSITIVE_EVS, CD9_POSITIVE_EVS,
                         POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)

    b_lo, b_hi = bounds_lo, bounds_hi
    print(f"  Parameters : {n_params}")
    print(f"  Bounds range summary: lo=[{np.array(b_lo).min():.4f}..{np.array(b_lo).max():.4f}]  "
          f"hi=[{np.array(b_hi).min():.4f}..{np.array(b_hi).max():.4f}]")

    # ── Global search (differential evolution) ────────────────────────────────
    _de_workers = DE_WORKERS if _PARALLEL_DE_AVAILABLE else 1
    _de_updating = 'deferred' if _de_workers != 1 else 'immediate'
    print(f"\n  STEP 1 — Global search (DE, {n_params} params, workers={_de_workers})...")
    t0 = time.time()
    # Use importable objective for parallel workers; fall back to local obj for workers=1
    _de_obj = _ev_mod.make_obj_for_worker(opt, LINKED, observed_counts, TOTAL_EVS,
                  CD63_POSITIVE_EVS, CD9_POSITIVE_EVS,
                  POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9) if _PARALLEL_DE_AVAILABLE else obj
    de_result = differential_evolution(
        _de_obj, bounds_scipy,
        seed=RANDOM_SEED, maxiter=DE_MAXITER, popsize=DE_POPSIZE,
        tol=1e-10, mutation=(0.5, 1.5), recombination=0.7,
        polish=True, workers=_de_workers, updating=_de_updating
    )
    # ── Local refinement ──────────────────────────────────────────────────────
    lbfgs = minimize(obj, de_result.x, method="L-BFGS-B",
                     bounds=bounds_scipy,
                     options={"maxiter": MAX_ITER, "ftol": 1e-14})
    best_x = lbfgs.x if lbfgs.fun < de_result.fun else de_result.x
    t_opt  = time.time() - t0
    print(f"  Optimization done in {t_opt:.1f}s  fun={min(lbfgs.fun,de_result.fun):.6f}")

    main_res = decode_result(best_x, opt, LINKED, observed_counts, TOTAL_EVS,
                              CD63_POSITIVE_EVS, CD9_POSITIVE_EVS,
                              POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)
    chi2_val = main_res["chi2_gof"]
    df_gof   = N_PHENOTYPES - 1 - n_params
    # df ≤ 0 means over-parameterized: GOF test is invalid (more params than data cells)
    if df_gof > 0:
        p_gof = 1 - chi2_dist.cdf(chi2_val, df_gof)
        gof_note = f"p={p_gof:.4f}{'  ✓ good fit' if p_gof>=0.05 else '  ✗ poor fit'}"
    else:
        p_gof = float("nan")
        gof_note = "⚠ df≤0 — over-parameterized, GOF test invalid"

    print(f"\n  STEP 2 — Fitted parameters:")
    print(f"  {'-'*50}")
    for pop_lbl in ["A", "B", "C"]:
        keys = [f"p9_{pop_lbl}", f"p81_{pop_lbl}", f"p63_{pop_lbl}"]
        vals = {k: main_res.get(k) for k in keys if main_res.get(k) is not None}
        if vals:
            print(f"  Population {pop_lbl}:")
            for k, v in vals.items():
                marker = k.split("_")[0].replace("p9","CD9").replace("p81","CD81").replace("p63","CD63")
                print(f"    P({marker}+) = {v:.4f}  ({v*100:.2f}%)")
        if f"fB" in main_res and pop_lbl == "B":
            print(f"    Size (frac of total) = {main_res['fB']:.4f}  ({main_res['fB']*100:.2f}%)")
        if f"fC" in main_res and pop_lbl == "C":
            print(f"    Size (frac of total) = {main_res['fC']:.4f}  ({main_res['fC']*100:.2f}%)")

    print(f"\n  STEP 3 — Observed vs. predicted phenotype counts:")
    print(f"  {'Phenotype':<10} {'Observed':>10} {'Predicted':>10} {'Residual':>10} {'Std.Res':>9}")
    print(f"  {'-'*52}")
    pp = main_res["predicted_probs"]
    rmse_val = np.sqrt(np.mean((observed_counts - pp*TOTAL_EVS)**2))
    ss_res = np.sum((observed_counts - pp*TOTAL_EVS)**2)
    ss_tot = np.sum((observed_counts - observed_counts.mean())**2)
    r2_val = 1 - ss_res/ss_tot
    for i, lbl in enumerate(PHENOTYPE_LABELS):
        pred_c = pp[i]*TOTAL_EVS
        res_v  = observed_counts[i] - pred_c
        std_r  = res_v / max(np.sqrt(pred_c), 1)
        print(f"  {lbl:<10} {observed_counts[i]:>10,} {pred_c:>10.1f} {res_v:>+10.1f} {std_r:>+9.3f}")
    print(f"\n  GOF χ²  = {chi2_val:.4f}  (df={df_gof}  {gof_note})")
    print(f"  RMSE    = {rmse_val:.4f}")
    print(f"  R²      = {r2_val:.4f}")
    print(f"  AIC     = {main_res['AIC']:.2f}   BIC = {main_res['BIC']:.2f}")

    # Store extra metrics
    main_res["rmse"] = rmse_val
    main_res["r2"]   = r2_val

    # ── Bootstrap ─────────────────────────────────────────────────────────────
    print(f"\n  STEP 4 — Bootstrap confidence intervals ({N_BOOTSTRAP:,} samples)...")
    boot_params = []
    rng_data    = replicate_data  # (8, 5)
    n_rep       = rng_data.shape[1]

    from joblib import Parallel, delayed

    def _one_boot(seed_offset):
        rng = np.random.default_rng(RANDOM_SEED + seed_offset)
        bi_idx = rng.choice(n_rep, n_rep, replace=True)
        b_obs  = rng_data[:, bi_idx].sum(axis=1).astype(float)
        b_total = int(b_obs.sum())
        b_cd63  = int(b_obs[CD63_POS].sum())
        b_cd9   = int(b_obs[CD9_POS].sum())
        b_obj   = make_objective(opt, LINKED, b_obs, b_total,
                                  b_cd63, b_cd9,
                                  POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)
        try:
            br = minimize(b_obj, best_x, method="L-BFGS-B",
                          bounds=bounds_scipy,
                          options={"maxiter": 2000, "ftol": 1e-10})
            if br.success:
                return br.x.copy()
            # Optimizer converged to 1e12 sentinel — objective function failed
            if br.fun >= 1e11:
                print(f"\n⚠ Bootstrap fit {seed_offset}: objective returned sentinel "
                      f"({br.fun:.3g}) — check for errors in make_objective")
            return None
        except Exception as _boot_exc:
            print(f"\n⚠ Bootstrap fit {seed_offset} raised {type(_boot_exc).__name__}: {_boot_exc}")
            return None

    print(f"  Running {N_BOOTSTRAP} bootstrap samples (BOOTSTRAP_JOBS={BOOTSTRAP_JOBS})...")
    t_boot = time.time()
    boot_raw = Parallel(n_jobs=BOOTSTRAP_JOBS, prefer="threads")(
        delayed(_one_boot)(i) for i in tqdm(range(N_BOOTSTRAP),
                                             desc=f"  Option {opt}A bootstrap", leave=True)
    )
    boot_params = [x for x in boot_raw if x is not None]
    print(f"  Bootstrap wall time: {time.time()-t_boot:.1f}s")

    boot_arr = np.array(boot_params)   # (n_success, n_params)
    n_success = len(boot_params)
    print(f"\n  Bootstrap results:")
    print(f"    Successful fits : {n_success}/{N_BOOTSTRAP}  ({100*n_success/N_BOOTSTRAP:.1f}%)")
    if n_success < N_BOOTSTRAP * 0.8:
        print(f"    ⚠ Low success rate — consider reducing regularization or increasing MAX_ITER")
    else:
        print(f"    ✓ Acceptable success rate")

    # Decode all bootstrap results
    boot_decoded = [decode_result(bx, opt, LINKED, observed_counts, TOTAL_EVS,
                                   CD63_POSITIVE_EVS, CD9_POSITIVE_EVS,
                                   POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)
                    for bx in boot_arr]

    # CI for predicted phenotype probabilities
    if boot_decoded:
        boot_probs = np.array([d["predicted_probs"] for d in boot_decoded])
        ci_lo_probs = np.percentile(boot_probs, CI_LO, axis=0)
        ci_hi_probs = np.percentile(boot_probs, CI_HI, axis=0)
    else:
        ci_lo_probs = main_res["predicted_probs"]
        ci_hi_probs = main_res["predicted_probs"]

    indep_results[opt] = {
        "option":      opt,
        "linked":      False,
        "main_res":    main_res,
        "best_x":      best_x,
        "boot_params": boot_arr,
        "boot_decoded":boot_decoded,
        "ci_lo_probs": ci_lo_probs,
        "ci_hi_probs": ci_hi_probs,
        "chi2_gof":    chi2_val,
        "df_gof":      df_gof,
        "p_gof":       p_gof,
    }

# ── Build Excel workbook ──────────────────────────────────────────────────────
print("\nBuilding Excel output...")

wb = Workbook()
wb.remove(wb.active)

def add_params_sheet(wb, opt, res, suffix="A"):
    ws = wb.create_sheet(f"Option{opt}{suffix}_Params")
    ws.append(["Parameter", "Value", "CI_Low_95", "CI_High_95", "Description"])
    style_header_row(ws, 1)
    mr = res["main_res"]
    bd = res["boot_decoded"]

    def ci_for(key):
        if not bd:
            return mr.get(key, np.nan), mr.get(key, np.nan)
        vals = [d.get(key, np.nan) for d in bd]
        vals = [v for v in vals if not np.isnan(v)]
        if not vals:
            return np.nan, np.nan
        return np.percentile(vals, CI_LO), np.percentile(vals, CI_HI)

    param_defs = {
        1: [("p9_A","P(CD9+) in Pop A"),("p81_A","P(CD81+) in Pop A"),
            ("p63_A","P(CD63+) in Pop A")],
        2: [("fB_of_cd63","Pop B fraction of CD63+ EVs"),
            ("fB","Pop B fraction of total EVs"),
            ("p9_B","P(CD9+) in Pop B"),("p81_B","P(CD81+) in Pop B"),("p63_B","P(CD63+) in Pop B [=1.0]"),
            ("p9_A","P(CD9+) in Pop A"),("p81_A","P(CD81+) in Pop A"),("p63_A","P(CD63+) in Pop A")],
        3: [("fB_of_cd63","Pop B fraction of CD63+ EVs"),("fB","Pop B fraction of total EVs"),
            ("fC_of_cd9","Pop C fraction of CD9+ EVs"),("fC","Pop C fraction of total EVs"),
            ("p9_B","P(CD9+) in Pop B"),("p81_B","P(CD81+) in Pop B"),("p63_B","P(CD63+) in Pop B [=1.0]"),
            ("p9_C","P(CD9+) in Pop C [=1.0]"),("p81_C","P(CD81+) in Pop C"),("p63_C","P(CD63+) in Pop C"),
            ("p9_A","P(CD9+) in Pop A"),("p81_A","P(CD81+) in Pop A"),("p63_A","P(CD63+) in Pop A")],
    }

    for key, desc in param_defs[opt]:
        val = mr.get(key, np.nan)
        lo, hi = ci_for(key)
        ws.append([key, round(float(val),6) if not np.isnan(val) else "N/A",
                   round(float(lo),6) if not np.isnan(lo) else "N/A",
                   round(float(hi),6) if not np.isnan(hi) else "N/A", desc])

    ws.append([])
    ws.append(["Robustness Metrics"])
    ws.append(["Chi2_GOF", round(res["chi2_gof"],4), "", "", "Pearson chi-sq goodness-of-fit"])
    ws.append(["df_GOF",   res["df_gof"], "", "", "Degrees of freedom"])
    ws.append(["p_GOF",    round(res["p_gof"],6), "", "", "p-value (>0.05 = good fit)"])
    ws.append(["AIC",      round(mr["AIC"],2)])
    ws.append(["BIC",      round(mr["BIC"],2)])
    ws.append(["Bootstrap_N_success", len(bd)])
    ws.append(["RMSE",  round(mr.get("rmse", float("nan")),4), "", "", "Root mean squared error (counts)"])
    ws.append(["R2",    round(mr.get("r2",   float("nan")),4), "", "", "R-squared (variance explained)"])
    auto_width(ws)

def add_phenotype_sheet(wb, opt, res, suffix="A"):
    ws = wb.create_sheet(f"Option{opt}{suffix}_Phenotypes")
    ws.append(["Phenotype","Observed","Obs_Freq","Pred_Freq","Pred_Count",
               "CI_Low","CI_High","Residual","Std_Residual"])
    style_header_row(ws, 1)
    pp = res["main_res"]["predicted_probs"]
    ci_lo = res["ci_lo_probs"]
    ci_hi = res["ci_hi_probs"]
    for i, lbl in enumerate(PHENOTYPE_LABELS):
        obs = observed_counts[i]
        pred_c = pp[i] * TOTAL_EVS
        res_v  = obs - pred_c
        std_r  = res_v / max(np.sqrt(pred_c), 1)
        ws.append([lbl, int(obs), round(obs/TOTAL_EVS,6),
                   round(pp[i],6), int(round(pred_c)),
                   round(ci_lo[i],6), round(ci_hi[i],6),
                   int(round(res_v)), round(std_r,4)])
    auto_width(ws)

for opt in OPTIONS:
    add_params_sheet(wb, opt, indep_results[opt], "A")
    add_phenotype_sheet(wb, opt, indep_results[opt], "A")

# ── Model comparison sheet ────────────────────────────────────────────────────
ws_cmp = wb.create_sheet("Model_Comparison")
ws_cmp.append(["Model","Option","Linkage","Chi2_GOF","df","p_GOF","AIC","BIC","N_params","RMSE","R2"])
style_header_row(ws_cmp, 1)
for opt in OPTIONS:
    r = indep_results[opt]
    ws_cmp.append([f"Option {opt}A", opt, "Independent",
                   round(r["chi2_gof"],4), r["df_gof"], round(r["p_gof"],6),
                   round(r["main_res"]["AIC"],2), round(r["main_res"]["BIC"],2),
                   param_count(opt, False),
                   round(r["main_res"].get("rmse", float("nan")),4),
                   round(r["main_res"].get("r2",   float("nan")),4)])
auto_width(ws_cmp)

# ── Population_Data tab — TargetOutput layout ─────────────────────────────────
# Columns: Short_Rep | CD9 | CD81 | CD63 | Observed |
#   then for each option×population: Pred_Counts | CI_Width | CI_Upper | CI_Lower | Phi | Theta
ws_pop = wb.create_sheet("Population_Data")

# Build header rows (3 levels matching TargetOutput)
opt_labels    = ["Option 1A", "Option 2A", "Option 2A", "Option 3A", "Option 3A", "Option 3A"]
pop_labels    = ["Pop A",     "Pop A",     "Pop B",     "Pop A",     "Pop B",     "Pop C"]
n_opt_cols    = len(opt_labels)
base_cols     = 5   # Short_Rep, CD9, CD81, CD63, Observed
cols_per_pop  = 8   # Pred_Counts, CI_Width, CI_Upper, CI_Lower, Phi_9_81, Phi_9_63, Phi_81_63, Thetacols_per_pop  = 8   # Pred_Counts, CI_Width, CI_Upper, CI_Lower, Phi_9_81, Phi_9_63, Phi_81_63, Theta

# Row 1: Option spans
row1 = [None]*base_cols
for lbl in opt_labels:
    row1 += [lbl] + [None]*(cols_per_pop-1)
ws_pop.append(row1)

# Row 2: Population spans
row2 = [None]*base_cols
for lbl in pop_labels:
    row2 += [lbl] + [None]*(cols_per_pop-1)
ws_pop.append(row2)

# Row 3: Sub-column headers
row3 = [None, "Phenotyping", None, None, "Observed"] + \
       ["Predicted_Counts","CI_Width","CI_Upper","CI_Lower","Phi_9_81","Phi_9_63","Phi_81_63","Theta"] * n_opt_cols
ws_pop.append(row3)

# Row 4: Marker headers
row4 = ["Short_Rep","CD9-PE","CD81-PE/Cy7","CD63-AF647","Counts"] + \
       ["Pred_Counts","CI_Width","CI_Upper","CI_Lower","Phi_9_81","Phi_9_63","Phi_81_63","Theta"] * n_opt_cols
ws_pop.append(row4)
style_header_row(ws_pop, 4)

# Map each (opt, pop) to its per-population prob arrays (NOT the mixed distribution)
# Scale by the population's fraction of total EVs to get per-pop predicted counts
def _pop_counts_and_ci(res, opt, pop):
    """Return (pred_counts_array, ci_lo_array, ci_hi_array, phi_9_81, phi_9_63, phi_81_63) scaled to EV counts."""
    mr      = res["main_res"]
    bd      = res["boot_decoded"]
    frac    = mr["pop_fracs"].get(pop, 0.0)
    pp      = mr["pop_probs"].get(pop)
    phi_9_81  = mr.get(f"phi_9_81_{pop}")
    phi_9_63  = mr.get(f"phi_9_63_{pop}")
    phi_81_63 = mr.get(f"phi_81_63_{pop}")
    if pp is None or frac == 0:
        return None, None, None, phi_9_81, phi_9_63, phi_81_63
    n_pop = frac * TOTAL_EVS
    pred  = pp * n_pop
    # Bootstrap CI on per-Pop Counts
    if bd:
        boot_pp = np.array([d["pop_probs"].get(pop, np.zeros(8)) for d in bd
                            if "pop_probs" in d and pop in d["pop_probs"]])
        boot_frac = np.array([d["pop_fracs"].get(pop, 0.0) for d in bd
                               if "pop_fracs" in d])
        if len(boot_pp) > 10:
            min_len = min(len(boot_pp), len(boot_frac))
            boot_counts = boot_pp[:min_len] * (boot_frac[:min_len, None] * TOTAL_EVS)
            ci_lo = np.percentile(boot_counts, CI_LO, axis=0)
            ci_hi = np.percentile(boot_counts, CI_HI, axis=0)
        else:
            ci_lo = pred; ci_hi = pred
    else:
        ci_lo = pred; ci_hi = pred
    return pred, ci_lo, ci_hi, phi_9_81, phi_9_63, phi_81_63

pop_data_map = [
    (1, "A", *_pop_counts_and_ci(indep_results[1], 1, "A")),
    (2, "A", *_pop_counts_and_ci(indep_results[2], 2, "A")),
    (2, "B", *_pop_counts_and_ci(indep_results[2], 2, "B")),
    (3, "A", *_pop_counts_and_ci(indep_results[3], 3, "A")),
    (3, "B", *_pop_counts_and_ci(indep_results[3], 3, "B")),
    (3, "C", *_pop_counts_and_ci(indep_results[3], 3, "C")),
]

marker_signs = [("+","+","+"),("+","+","-"),("+","-","+"),("+","-","-"),
                ("-","+","+"),("-","+","-"),("-","-","+"),(  "-","-","-")]

for i, lbl in enumerate(PHENOTYPE_LABELS):
    cd9s, cd81s, cd63s = marker_signs[i]
    obs_c = int(observed_counts[i])
    data_row = [lbl, cd9s, cd81s, cd63s, obs_c]
    for opt, pop, pred, ci_lo, ci_hi, phi_9_81, phi_9_63, phi_81_63 in pop_data_map:
        if pred is not None:
            pred_c     = int(round(float(pred[i])))
            ci_lo_v    = int(round(float(ci_lo[i])))
            ci_hi_v    = int(round(float(ci_hi[i])))
            ci_w       = ci_hi_v - ci_lo_v
            phi_81_v   = round(float(phi_9_81),  6) if phi_9_81  is not None else "NA"
            phi_63_v   = round(float(phi_9_63),  6) if phi_9_63  is not None else "NA"
            phi_8163_v = round(float(phi_81_63), 6) if phi_81_63 is not None else "NA"
            _pop_pp    = indep_results[opt]["main_res"]["pop_probs"].get(pop)
            theta_v    = round(float(_pop_pp[i]), 6) if _pop_pp is not None else "NA"
        else:
            pred_c = ci_lo_v = ci_hi_v = ci_w = phi_81_v = phi_63_v = phi_8163_v = theta_v = "NA"
        data_row += [pred_c, ci_w, ci_hi_v, ci_lo_v, phi_81_v, phi_63_v, phi_8163_v, theta_v]
    ws_pop.append(data_row)

# ── Summary rows: total EVs per Pop And EV count with ≥1 tetraspanin ──────────
ws_pop.append([])  # blank separator

# Row: Total EVs in each population
total_row = ["Total EVs", "", "", "", TOTAL_EVS]
for opt, pop, pred, ci_lo, ci_hi, phi_9_81, phi_9_63, phi_81_63 in pop_data_map:
    mr   = indep_results[opt]["main_res"]
    frac = mr["pop_fracs"].get(pop)
    n_pop = int(round(frac * TOTAL_EVS)) if frac is not None else "NA"
    total_row += [n_pop, None, None, None, None, None, None, None]
ws_pop.append(total_row)

# Row: EVs with ≥1 tetraspanin (all phenotypes except -/-/-)
# Index 7 is -/-/- (no markers); all others have ≥1
pos_idx = [0,1,2,3,4,5,6]  # indices 0-6 all have at least one marker
any_marker_row = ["EVs with ≥1 Tetraspanin", "", "", "",
                  int(observed_counts[pos_idx].sum())]
for opt, pop, pred, ci_lo, ci_hi, phi_9_81, phi_9_63, phi_81_63 in pop_data_map:
    if pred is not None:
        n_pos     = int(round(float(pred[pos_idx].sum())))
        ci_lo_pos = int(round(float(ci_lo[pos_idx].sum()))) if ci_lo is not None else "NA"
        ci_hi_pos = int(round(float(ci_hi[pos_idx].sum()))) if ci_hi is not None else "NA"
        ci_w_pos  = (ci_hi_pos - ci_lo_pos) if (ci_lo_pos != "NA" and ci_hi_pos != "NA") else "NA"
        any_marker_row += [n_pos, ci_w_pos, ci_hi_pos, ci_lo_pos, "NA", "NA", "NA", "NA"]
    else:
        any_marker_row += ["NA", "NA", "NA", "NA", "NA", "NA", "NA", "NA"]
ws_pop.append(any_marker_row)

auto_width(ws_pop)

# ── Model tab — matching TargetOutput Model sheet ─────────────────────────────
ws_model = wb.create_sheet("Model")
ws_model.append([None, "Tetraspanin Fraction (fitted marginals)", None, None])
ws_model.append([None, "Option 1A","Option 2A",None,"Option 3A",None,None])
ws_model.append([None, "Population 1","Population 1","Population B",
                  "Population 1","Population B","Population C"])
style_header_row(ws_model, 3)

param_rows = [
    ("CD9",   "p9_A",  "p9_A",  "p9_B",  "p9_A",  "p9_B",  "p9_C"),
    ("CD63",  "p63_A", "p63_A", "p63_B", "p63_A", "p63_B", "p63_C"),
    ("CD81",  "p81_A", "p81_A", "p81_B", "p81_A", "p81_B", "p81_C"),
]
for row_lbl, *keys in param_rows:
    row_vals = [row_lbl]
    for k, opt in zip(keys, [1,2,2,3,3,3]):
        v = indep_results[opt]["main_res"].get(k, None)
        row_vals.append(round(v, 4) if v is not None else "N/A")
    ws_model.append(row_vals)

# Metrics rows — value once per option, blank for sub-populations
# Column layout: Opt1A-PopA | Opt2A-PopA | Opt2A-PopB | Opt3A-PopA | Opt3A-PopB | Opt3A-PopC
# opt_col_map: (opt, is_first_pop_of_this_opt)
opt_col_map = [(1,True),(2,True),(2,False),(3,True),(3,False),(3,False)]
for metric, key in [("Model RMSE","rmse"),("Model R²","r2"),("AIC","AIC"),("BIC","BIC")]:
    row_vals = [metric]
    for opt, is_first in opt_col_map:
        if is_first:
            v = indep_results[opt]["main_res"].get(key, float("nan"))
            row_vals.append(round(v,4) if not (isinstance(v,float) and np.isnan(v)) else "N/A")
        else:
            row_vals.append(None)  # blank for sub-Pop Columns — metric is per-model not per-pop
    ws_model.append(row_vals)

# EV count summary rows
ws_model.append([])
def _ev_count(opt, pop, results):
    mr   = results[opt]["main_res"]
    frac = mr["pop_fracs"].get(pop)
    return int(round(frac * TOTAL_EVS)) if frac is not None else "NA"

def _ev_pos_count(opt, pop, results):
    mr   = results[opt]["main_res"]
    frac = mr["pop_fracs"].get(pop)
    pp   = mr["pop_probs"].get(pop)
    if frac is None or pp is None:
        return "NA"
    return int(round(float(pp[:7].sum()) * frac * TOTAL_EVS))

ws_model.append(["Total EVs in Population"] +
    [_ev_count(opt, pop, indep_results)
     for opt, pop in [(1,"A"),(2,"A"),(2,"B"),(3,"A"),(3,"B"),(3,"C")]])
ws_model.append(["EVs with ≥1 Tetraspanin"] +
    [_ev_pos_count(opt, pop, indep_results)
     for opt, pop in [(1,"A"),(2,"A"),(2,"B"),(3,"A"),(3,"B"),(3,"C")]])
auto_width(ws_model)

# ── Option_Desc tab ───────────────────────────────────────────────────────────
ws_desc = wb.create_sheet("Option_Desc")
ws_desc.append([None, None, None, "Population", None, None])
ws_desc.append([None, None, "Assortment_Type", "A", "B", "C"])
style_header_row(ws_desc, 2)
desc_rows = [
    ("Option", 1, "Independent", "Remaining", "NA", "NA"),
    ("Option", 2, "Independent", "Remaining", f"~{POP_B_FRAC_OF_CD63*100:.2f}% of CD63+", "NA"),
    ("Option", 3, "Independent", "Remaining", f"~{POP_B_FRAC_OF_CD63*100:.2f}% of CD63+",
                                               f"~{POP_C_FRAC_OF_CD9*100:.2f}% of CD9+"),
]
for r in desc_rows:
    ws_desc.append(list(r))
auto_width(ws_desc)

fpath_indep = save_wb(wb, "Cell_1.06_Independent_Models", ts)
print(f"\n✓ Independent models complete.")
print(f"  Total cell wall time: {time.time()-_cell_start:.1f}s  ({(time.time()-_cell_start)/60:.1f} min)")
print(f"  Tabs created: Option1A_Params, Option2A_Params, Option3A_Params,")
print(f"                Option1A_Phenotypes, Option2A_Phenotypes, Option3A_Phenotypes,")
print(f"                Model_Comparison, Population_Data, Model, Option_Desc")
stop_logging(lf)


In [8]:
# Cell 1.07
"""
================================================================================
CELL 1.07: LINKED ASSORTMENT MODELS — OPTIONS 1, 2, 3 (WITH PHI)
================================================================================
PURPOSE:
  Fit direct-probability models for each of the three population options
  under the assumption of LINKED assortment (CD9–CD81 linkage disequilibrium).
  The phi (φ) parameter for each population captures the co-occurrence
  excess of CD9+ and CD81+ beyond what independent assortment predicts:
    phi = P(CD9+,CD81+) − P(CD9+)·P(CD81+)
  Phi is estimated separately for each population and can differ between them.
  CD63 is treated as independent of the CD9–CD81 linkage.

  Options: 1B, 2B, 3B (same population structure as 1A/2A/3A, plus phi).
  Bootstrap: N_BOOTSTRAP samples for 95% CIs.
  Robustness: chi-sq GOF, AIC, BIC.

OUTPUT:
  Excel: Cell_1.07_Linked_Models_<ts>.xlsx
    Tabs: Option1B_Params, Option2B_Params, Option3B_Params,
          Option1B_Phenotypes, Option2B_Phenotypes, Option3B_Phenotypes,
          Model_Comparison_All6
  Debug: Cell_1.07_Debug_<ts>.txt
================================================================================
"""

lf, ts = start_logging("Cell_1.07")

print("=" * 70)
print("CELL 1.07: LINKED ASSORTMENT MODELS WITH PHI (OPTIONS 1B, 2B, 3B)")
print("=" * 70)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
print("CONFIGURATION")
print("-" * 60)
print(f"  Total EVs         : {TOTAL_EVS:,}")
print(f"  CD63+ EVs         : {CD63_POSITIVE_EVS:,}  ({CD63_POSITIVE_EVS/TOTAL_EVS*100:.2f}%)")
print(f"  CD9+  EVs         : {CD9_POSITIVE_EVS:,}  ({CD9_POSITIVE_EVS/TOTAL_EVS*100:.2f}%)")
print(f"  Pop B (of CD63+)  : {POP_B_FRAC_OF_CD63*100:.2f}% ± CV={POP_B_FRAC_CV:.4f}")
print(f"  Pop C (of CD9+)   : {POP_C_FRAC_OF_CD9*100:.2f}% ± CV={POP_C_FRAC_CV:.4f}")
print(f"  Regularization    : REG_FRAC={REG_FRAC}  REG_MARG={REG_MARG}  REG_PHI={REG_PHI}")
print(f"  DE maxiter        : {DE_MAXITER}  popsize={DE_POPSIZE}")
print(f"  Bootstrap samples : {N_BOOTSTRAP:,}")
print(f"  Random seed       : {RANDOM_SEED + 1}")
print(f"  Assortment type   : LINKED (phi = CD9–CD81 LD coefficient per population)\n")
print(f"  phi definition    : phi = P(CD9+,CD81+) − P(CD9+)·P(CD81+)")
print(f"  phi = 0 → independent; phi > 0 → co-occurrence excess; phi < 0 → mutual exclusion\n")

np.random.seed(RANDOM_SEED + 1)

OPTIONS = [1, 2, 3]
LINKED  = True

linked_results = {}

for opt in OPTIONS:
    print(f"\n{'─'*60}")
    print(f"  OPTION {opt}B — Linked assortment (phi)")
    print(f"{'─'*60}")

    n_params = param_count(opt, LINKED)
    bounds_lo, bounds_hi = make_bounds(opt, LINKED,
                                        POP_B_FRAC_OF_CD63, POP_B_FRAC_SD,
                                        POP_C_FRAC_OF_CD9,  POP_C_FRAC_SD)
    bounds_scipy = list(zip(bounds_lo, bounds_hi))
    obj = make_objective(opt, LINKED, observed_counts, TOTAL_EVS,
                         CD63_POSITIVE_EVS, CD9_POSITIVE_EVS,
                         POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)

    _de_workers = DE_WORKERS if _PARALLEL_DE_AVAILABLE else 1
    _de_updating = 'deferred' if _de_workers != 1 else 'immediate'
    print(f"\n  STEP 1 — Global search (DE, {n_params} params, workers={_de_workers})...")
    t0 = time.time()
    _de_obj = _ev_mod.make_obj_for_worker(opt, LINKED, observed_counts, TOTAL_EVS,
                  CD63_POSITIVE_EVS, CD9_POSITIVE_EVS,
                  POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9) if _PARALLEL_DE_AVAILABLE else obj
    # Uniform DE settings across all options — sized for the most complex model
    # (Option 3B, 15 params) so all NLL values are optimized to the same standard.
    de_result = differential_evolution(
        _de_obj, bounds_scipy,
        seed=RANDOM_SEED+1, maxiter=DE_MAXITER, popsize=DE_POPSIZE,
        tol=1e-10, mutation=(0.5, 1.5), recombination=0.7,
        polish=True, workers=_de_workers, updating=_de_updating
    )
    lbfgs = minimize(obj, de_result.x, method="L-BFGS-B",
                     bounds=bounds_scipy,
                     options={"maxiter": MAX_ITER, "ftol": 1e-14})
    best_x = lbfgs.x if lbfgs.fun < de_result.fun else de_result.x
    t_opt  = time.time() - t0
    print(f"  Optimization done in {t_opt:.1f}s  fun={min(lbfgs.fun,de_result.fun):.6f}")

    main_res = decode_result(best_x, opt, LINKED, observed_counts, TOTAL_EVS,
                              CD63_POSITIVE_EVS, CD9_POSITIVE_EVS,
                              POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)
    chi2_val = main_res["chi2_gof"]
    df_gof   = N_PHENOTYPES - 1 - n_params
    # df ≤ 0 means over-parameterized: GOF test is invalid (more params than data cells)
    if df_gof > 0:
        p_gof = 1 - chi2_dist.cdf(chi2_val, df_gof)
        gof_note = f"p={p_gof:.4f}{'  ✓ good fit' if p_gof>=0.05 else '  ✗ poor fit'}"
    else:
        p_gof = float("nan")
        gof_note = "⚠ df≤0 — over-parameterized, GOF test invalid"
    print(f"\n  STEP 2 — Fitted parameters:")
    print(f"  {'-'*50}")
    for pop_lbl in ["A","B","C"]:
        keys = [f"p9_{pop_lbl}", f"p81_{pop_lbl}", f"p63_{pop_lbl}"]
        vals = {k: main_res.get(k) for k in keys if main_res.get(k) is not None}
        if vals:
            print(f"  Population {pop_lbl}:")
            for k, v in vals.items():
                marker = k.split("_")[0].replace("p9","CD9").replace("p81","CD81").replace("p63","CD63")
                print(f"    P({marker}+) = {v:.4f}  ({v*100:.2f}%)")
            for phi_pair, phi_label in [(f"phi_9_81_{pop_lbl}", "CD9–CD81"),
                                        (f"phi_9_63_{pop_lbl}", "CD9–CD63"),
                                        (f"phi_81_63_{pop_lbl}", "CD81–CD63")]:
                if phi_pair in main_res:
                    phi_v = main_res[phi_pair]
                    interp = "co-occurrence excess" if phi_v > 0.005 else ("mutual exclusion" if phi_v < -0.005 else "~independent")
                    print(f"    phi({phi_label}) = {phi_v:+.6f}  ({interp})")
        if "fB" in main_res and pop_lbl == "B":
            print(f"    Size (frac of total) = {main_res['fB']:.4f}  ({main_res['fB']*100:.2f}%)")
        if "fC" in main_res and pop_lbl == "C":
            print(f"    Size (frac of total) = {main_res['fC']:.4f}  ({main_res['fC']*100:.2f}%)")

    print(f"\n  STEP 3 — Observed vs. predicted phenotype counts:")
    print(f"  {'Phenotype':<10} {'Observed':>10} {'Predicted':>10} {'Residual':>10} {'Std.Res':>9}")
    print(f"  {'-'*52}")
    pp = main_res["predicted_probs"]
    rmse_val = np.sqrt(np.mean((observed_counts - pp*TOTAL_EVS)**2))
    ss_res = np.sum((observed_counts - pp*TOTAL_EVS)**2)
    ss_tot = np.sum((observed_counts - observed_counts.mean())**2)
    r2_val = 1 - ss_res/ss_tot
    for i, lbl in enumerate(PHENOTYPE_LABELS):
        pred_c = pp[i]*TOTAL_EVS
        res_v  = observed_counts[i] - pred_c
        std_r  = res_v / max(np.sqrt(pred_c), 1)
        print(f"  {lbl:<10} {observed_counts[i]:>10,} {pred_c:>10.1f} {res_v:>+10.1f} {std_r:>+9.3f}")
    print(f"\n  GOF χ²  = {chi2_val:.4f}  (df={df_gof}  {gof_note})")
    print(f"  RMSE    = {rmse_val:.4f}")
    print(f"  R²      = {r2_val:.4f}")
    print(f"  AIC     = {main_res['AIC']:.2f}   BIC = {main_res['BIC']:.2f}")
    main_res["rmse"] = rmse_val
    main_res["r2"]   = r2_val

    # ── Bootstrap ─────────────────────────────────────────────────────────────
    print(f"\n  STEP 4 — Bootstrap confidence intervals ({N_BOOTSTRAP:,} samples)...")
    boot_params = []
    n_rep = replicate_data.shape[1]

    for bi in tqdm(range(N_BOOTSTRAP), desc=f"  Option {opt}B bootstrap", leave=True):
        bi_idx  = np.random.choice(n_rep, n_rep, replace=True)
        b_obs   = replicate_data[:, bi_idx].sum(axis=1).astype(float)
        b_total = int(b_obs.sum())
        b_cd63  = int(b_obs[CD63_POS].sum())
        b_cd9   = int(b_obs[CD9_POS].sum())
        b_obj   = make_objective(opt, LINKED, b_obs, b_total,
                                  b_cd63, b_cd9,
                                  POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)
        try:
            br = minimize(b_obj, best_x, method="L-BFGS-B",
                          bounds=bounds_scipy,
                          options={"maxiter": 2000, "ftol": 1e-10})
            if br.success:
                boot_params.append(br.x.copy())
            elif br.fun >= 1e11:
                print(f"\n⚠ Bootstrap sample {bi}: objective returned sentinel "
                      f"({br.fun:.3g}) — check for errors in make_objective")
        except Exception as _boot_exc:
            print(f"\n⚠ Bootstrap sample {bi} raised {type(_boot_exc).__name__}: {_boot_exc}")

    boot_arr  = np.array(boot_params)
    n_success = len(boot_params)
    print(f"\n  Bootstrap results:")
    print(f"    Successful fits : {n_success}/{N_BOOTSTRAP}  ({100*n_success/N_BOOTSTRAP:.1f}%)")
    if n_success < N_BOOTSTRAP * 0.8:
        print(f"    ⚠ Low success rate — consider reducing regularization or increasing MAX_ITER")
    else:
        print(f"    ✓ Acceptable success rate")

    boot_decoded = [decode_result(bx, opt, LINKED, observed_counts, TOTAL_EVS,
                                   CD63_POSITIVE_EVS, CD9_POSITIVE_EVS,
                                   POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)
                    for bx in boot_arr]

    if boot_decoded:
        boot_probs = np.array([d["predicted_probs"] for d in boot_decoded])
        ci_lo_probs = np.percentile(boot_probs, CI_LO, axis=0)
        ci_hi_probs = np.percentile(boot_probs, CI_HI, axis=0)
    else:
        ci_lo_probs = main_res["predicted_probs"]
        ci_hi_probs = main_res["predicted_probs"]

    linked_results[opt] = {
        "option":      opt,
        "linked":      True,
        "main_res":    main_res,
        "best_x":      best_x,
        "boot_params": boot_arr,
        "boot_decoded":boot_decoded,
        "ci_lo_probs": ci_lo_probs,
        "ci_hi_probs": ci_hi_probs,
        "chi2_gof":    chi2_val,
        "df_gof":      df_gof,
        "p_gof":       p_gof,
    }

# ── Build Excel workbook ──────────────────────────────────────────────────────
print("\nBuilding Excel output...")

wb = Workbook()
wb.remove(wb.active)

def add_params_sheet_linked(wb, opt, res, suffix="B"):
    ws = wb.create_sheet(f"Option{opt}{suffix}_Params")
    ws.append(["Parameter", "Value", "CI_Low_95", "CI_High_95", "Description"])
    style_header_row(ws, 1)
    mr = res["main_res"]
    bd = res["boot_decoded"]

    def ci_for(key):
        if not bd:
            return np.nan, np.nan
        vals = [d.get(key, np.nan) for d in bd]
        vals = [v for v in vals if not np.isnan(v)]
        if not vals:
            return np.nan, np.nan
        return np.percentile(vals, CI_LO), np.percentile(vals, CI_HI)

    param_defs = {
        1: [("p9_A","P(CD9+) in Pop A"),("p81_A","P(CD81+) in Pop A"),
            ("p63_A","P(CD63+) in Pop A"),
            ("phi_9_81_A","Phi CD9-CD81 LD coeff in Pop A"),
            ("phi_9_63_A","Phi CD9-CD63 LD coeff in Pop A")],
        2: [("fB_of_cd63","Pop B fraction of CD63+ EVs"),
            ("fB","Pop B fraction of total EVs"),
            ("p9_B","P(CD9+) in Pop B"),("p81_B","P(CD81+) in Pop B"),("p63_B","P(CD63+) in Pop B [=1.0]"),
            ("phi_9_81_B","Phi CD9-CD81 LD coeff in Pop B"),
            ("phi_9_63_B","Phi CD9-CD63 LD coeff in Pop B"),
            ("p9_A","P(CD9+) in Pop A"),("p81_A","P(CD81+) in Pop A"),("p63_A","P(CD63+) in Pop A"),
            ("phi_9_81_A","Phi CD9-CD81 LD coeff in Pop A"),
            ("phi_9_63_A","Phi CD9-CD63 LD coeff in Pop A")],
        3: [("fB_of_cd63","Pop B fraction of CD63+ EVs"),("fB","Pop B fraction of total EVs"),
            ("fC_of_cd9","Pop C fraction of CD9+ EVs"),("fC","Pop C fraction of total EVs"),
            ("p9_B","P(CD9+) in Pop B"),("p81_B","P(CD81+) in Pop B"),("p63_B","P(CD63+) in Pop B [=1.0]"),
            ("phi_9_81_B","Phi CD9-CD81 LD coeff in Pop B"),
            ("phi_9_63_B","Phi CD9-CD63 LD coeff in Pop B"),
            ("p9_C","P(CD9+) in Pop C [=1.0]"),("p81_C","P(CD81+) in Pop C"),("p63_C","P(CD63+) in Pop C"),
            ("phi_9_81_C","Phi CD9-CD81 LD coeff in Pop C [=0, forced by p9_C=1.0]"),
            ("phi_9_63_C","Phi CD9-CD63 LD coeff in Pop C [=0, forced by p9_C=1.0]"),
            ("phi_81_63_C","Phi CD81-CD63 LD coeff in Pop C [free parameter]"),
            ("p9_A","P(CD9+) in Pop A"),("p81_A","P(CD81+) in Pop A"),("p63_A","P(CD63+) in Pop A"),
            ("phi_9_81_A","Phi CD9-CD81 LD coeff in Pop A"),
            ("phi_9_63_A","Phi CD9-CD63 LD coeff in Pop A")],
    }

    for key, desc in param_defs[opt]:
        val = mr.get(key, np.nan)
        lo, hi = ci_for(key)
        ws.append([key, round(float(val),6) if not np.isnan(val) else "N/A",
                   round(float(lo),6) if not np.isnan(lo) else "N/A",
                   round(float(hi),6) if not np.isnan(hi) else "N/A", desc])

    ws.append([])
    ws.append(["Robustness Metrics"])
    ws.append(["Chi2_GOF", round(res["chi2_gof"],4), "", "", "Pearson chi-sq goodness-of-fit"])
    ws.append(["df_GOF",   res["df_gof"]])
    ws.append(["p_GOF",    round(res["p_gof"],6)])
    ws.append(["AIC",      round(mr["AIC"],2)])
    ws.append(["BIC",      round(mr["BIC"],2)])
    ws.append(["Bootstrap_N_success", len(bd)])
    ws.append(["RMSE", round(mr.get("rmse", float("nan")),4), "", "", "Root mean squared error (counts)"])
    ws.append(["R2",   round(mr.get("r2",   float("nan")),4), "", "", "R-squared (variance explained)"])
    auto_width(ws)

def add_phenotype_sheet_linked(wb, opt, res, suffix="B"):
    ws = wb.create_sheet(f"Option{opt}{suffix}_Phenotypes")
    ws.append(["Phenotype","Observed","Obs_Freq","Pred_Freq","Pred_Count",
               "CI_Low","CI_High","Residual","Std_Residual"])
    style_header_row(ws, 1)
    pp = res["main_res"]["predicted_probs"]
    ci_lo = res["ci_lo_probs"]
    ci_hi = res["ci_hi_probs"]
    for i, lbl in enumerate(PHENOTYPE_LABELS):
        obs = observed_counts[i]
        pred_c = pp[i] * TOTAL_EVS
        res_v  = obs - pred_c
        std_r  = res_v / max(np.sqrt(pred_c), 1)
        ws.append([lbl, int(obs), round(obs/TOTAL_EVS,6),
                   round(pp[i],6), int(round(pred_c)),
                   round(ci_lo[i],6), round(ci_hi[i],6),
                   int(round(res_v)), round(std_r,4)])
    auto_width(ws)

for opt in OPTIONS:
    add_params_sheet_linked(wb, opt, linked_results[opt], "B")
    add_phenotype_sheet_linked(wb, opt, linked_results[opt], "B")

# All-6-model comparison (includes indep_results from Cell 1.06)
ws_cmp = wb.create_sheet("Model_Comparison_All6")
ws_cmp.append(["Model","Option","Linkage","Chi2_GOF","df","p_GOF","AIC","BIC","N_params","Best_Fit"])
style_header_row(ws_cmp, 1)

all_models = []
for opt in [1,2,3]:
    if opt in indep_results:
        r = indep_results[opt]
        all_models.append((f"Option {opt}A", opt, "Independent",
                           r["chi2_gof"], r["df_gof"], r["p_gof"],
                           r["main_res"]["AIC"], r["main_res"]["BIC"],
                           param_count(opt, False)))
    r = linked_results[opt]
    all_models.append((f"Option {opt}B", opt, "Linked",
                       r["chi2_gof"], r["df_gof"], r["p_gof"],
                       r["main_res"]["AIC"], r["main_res"]["BIC"],
                       param_count(opt, True)))

best_aic = min(m[6] for m in all_models)
best_bic = min(m[7] for m in all_models)
for row in all_models:
    aic_star = "★AIC" if abs(row[6] - best_aic) < 0.01 else ""
    bic_star = "★BIC" if abs(row[7] - best_bic) < 0.01 else ""
    flags = "  ".join(f for f in [aic_star, bic_star] if f)
    ws_cmp.append(list(row) + [flags if flags else ""])
auto_width(ws_cmp)

# ── Population_Data tab — all 6 models, matching TargetOutput layout ──────────
ws_pop = wb.create_sheet("Population_Data")
all_opt_labels = ["Option 1A","Option 2A","Option 2A","Option 3A","Option 3A","Option 3A",
                  "Option 1B","Option 2B","Option 2B","Option 3B","Option 3B","Option 3B"]
all_pop_labels = ["Pop A","Pop A","Pop B","Pop A","Pop B","Pop C",
                  "Pop A","Pop A","Pop B","Pop A","Pop B","Pop C"]
cols_per_pop   = 8   # Pred_Counts, CI_Width, CI_Upper, CI_Lower, Phi_9_81, Phi_9_63, Phi_81_63, Theta
base_cols      = 5

row1 = [None]*base_cols
for lbl in all_opt_labels:
    row1 += [lbl] + [None]*(cols_per_pop-1)
ws_pop.append(row1)

row2 = [None]*base_cols
for lbl in all_pop_labels:
    row2 += [lbl] + [None]*(cols_per_pop-1)
ws_pop.append(row2)

row3 = [None,"Phenotyping",None,None,"Observed"] + \
       ["Predicted_Counts","CI_Width","CI_Upper","CI_Lower","Phi_9_81","Phi_9_63","Phi_81_63","Theta"]*len(all_opt_labels)
ws_pop.append(row3)
row4 = ["Short_Rep","CD9-PE","CD81-PE/Cy7","CD63-AF647","Counts"] + \
       ["Pred_Counts","CI_Width","CI_Upper","CI_Lower","Phi_9_81","Phi_9_63","Phi_81_63","Theta"]*len(all_opt_labels)
ws_pop.append(row4)
style_header_row(ws_pop, 4)

# Use _pop_counts_and_ci (defined identically to Cell 1.06 version, but for both result sets)
def _pop_counts_and_ci_07(results_dict, opt, pop):
    if opt not in results_dict:
        return None, None, None, None, None, None
    res  = results_dict[opt]
    mr   = res["main_res"]
    bd   = res["boot_decoded"]
    frac    = mr["pop_fracs"].get(pop)
    pp      = mr["pop_probs"].get(pop)
    phi_9_81  = mr.get(f"phi_9_81_{pop}")
    phi_9_63  = mr.get(f"phi_9_63_{pop}")
    phi_81_63 = mr.get(f"phi_81_63_{pop}")
    if pp is None or frac is None or frac == 0:
        return None, None, None, phi_9_81, phi_9_63, phi_81_63
    n_pop = frac * TOTAL_EVS
    pred  = pp * n_pop
    if bd:
        boot_pp = np.array([d["pop_probs"].get(pop) for d in bd
                            if "pop_probs" in d and pop in d["pop_probs"]])
        boot_frac = np.array([d["pop_fracs"].get(pop) for d in bd
                               if "pop_fracs" in d and d["pop_fracs"].get(pop) is not None])
        if len(boot_pp) > 10:
            min_len = min(len(boot_pp), len(boot_frac))
            boot_counts = boot_pp[:min_len] * (boot_frac[:min_len, None] * TOTAL_EVS)
            ci_lo = np.percentile(boot_counts, CI_LO, axis=0)
            ci_hi = np.percentile(boot_counts, CI_HI, axis=0)
        else:
            ci_lo = pred; ci_hi = pred
    else:
        ci_lo = pred; ci_hi = pred
    return pred, ci_lo, ci_hi, phi_9_81, phi_9_63, phi_81_63

all_pop_data_map = [
    (1,"A",*_pop_counts_and_ci_07(indep_results,  1,"A")),
    (2,"A",*_pop_counts_and_ci_07(indep_results,  2,"A")),
    (2,"B",*_pop_counts_and_ci_07(indep_results,  2,"B")),
    (3,"A",*_pop_counts_and_ci_07(indep_results,  3,"A")),
    (3,"B",*_pop_counts_and_ci_07(indep_results,  3,"B")),
    (3,"C",*_pop_counts_and_ci_07(indep_results,  3,"C")),
    (1,"A",*_pop_counts_and_ci_07(linked_results, 1,"A")),
    (2,"A",*_pop_counts_and_ci_07(linked_results, 2,"A")),
    (2,"B",*_pop_counts_and_ci_07(linked_results, 2,"B")),
    (3,"A",*_pop_counts_and_ci_07(linked_results, 3,"A")),
    (3,"B",*_pop_counts_and_ci_07(linked_results, 3,"B")),
    (3,"C",*_pop_counts_and_ci_07(linked_results, 3,"C")),
]

marker_signs = [("+","+","+"),("+","+","-"),("+","-","+"),("+","-","-"),
                ("-","+","+"),("-","+","-"),("-","-","+"),(  "-","-","-")]
all_sources = [(indep_results,False),(indep_results,False),(indep_results,False),
               (indep_results,False),(indep_results,False),(indep_results,False),
               (linked_results,True),(linked_results,True),(linked_results,True),
               (linked_results,True),(linked_results,True),(linked_results,True)]

for i, lbl in enumerate(PHENOTYPE_LABELS):
    cd9s, cd81s, cd63s = marker_signs[i]
    data_row = [lbl, cd9s, cd81s, cd63s, int(observed_counts[i])]
    for (opt, pop, pred, ci_lo, ci_hi, phi_9_81, phi_9_63, phi_81_63), (src_dict, _) in zip(all_pop_data_map, all_sources):
        if pred is not None:
            pred_c      = int(round(float(pred[i])))
            ci_lo_v     = int(round(float(ci_lo[i])))
            ci_hi_v     = int(round(float(ci_hi[i])))
            ci_w        = ci_hi_v - ci_lo_v
            phi_81_out  = round(float(phi_9_81),  6) if phi_9_81  is not None else "NA"
            phi_63_out  = round(float(phi_9_63),  6) if phi_9_63  is not None else "NA"
            phi_8163_out= round(float(phi_81_63), 6) if phi_81_63 is not None else "NA"
            _pop_pp     = src_dict[opt]["main_res"]["pop_probs"].get(pop)
            theta_out   = round(float(_pop_pp[i]), 6) if _pop_pp is not None else "NA"
        else:
            pred_c = ci_lo_v = ci_hi_v = ci_w = phi_81_out = phi_63_out = phi_8163_out = theta_out = "NA"
        data_row += [pred_c, ci_w, ci_hi_v, ci_lo_v, phi_81_out, phi_63_out, phi_8163_out, theta_out]
    ws_pop.append(data_row)

# Summary rows
ws_pop.append([])
all_pops_order = [("A",1),("A",2),("B",2),("A",3),("B",3),("C",3),
                  ("A",1),("A",2),("B",2),("A",3),("B",3),("C",3)]
total_row = ["Total EVs", "", "", "", TOTAL_EVS]
any_marker_row = ["EVs with ≥1 Tetraspanin", "", "", "", int(observed_counts[:7].sum())]
for (src_dict, _), (pop, opt) in zip(all_sources, all_pops_order):
    if opt in src_dict:
        mr   = src_dict[opt]["main_res"]
        frac = mr["pop_fracs"].get(pop)
        pp   = mr["pop_probs"].get(pop)
        n_pop = int(round(frac * TOTAL_EVS)) if frac is not None else "NA"
        n_pos = int(round(float(pp[:7].sum()) * frac * TOTAL_EVS)) if (frac is not None and pp is not None) else "NA"
        total_row     += [n_pop, None, None, None, None, None, None, None]
        any_marker_row+= [n_pos, None, None, None, None, None, None, None]
    else:
        total_row     += ["NA"]*8
        any_marker_row+= ["NA"]*8
ws_pop.append(total_row)
ws_pop.append(any_marker_row)
auto_width(ws_pop)

# ── Model tab ─────────────────────────────────────────────────────────────────
ws_model = wb.create_sheet("Model")
ws_model.append([None,"Tetraspanin Fraction (fitted marginals)"])
ws_model.append([None,"Option 1A","Option 2A",None,"Option 3A",None,None,
                       "Option 1B","Option 2B",None,"Option 3B",None,None])
ws_model.append([None,"Pop A","Pop A","Pop B","Pop A","Pop B","Pop C",
                       "Pop A","Pop A","Pop B","Pop A","Pop B","Pop C"])
style_header_row(ws_model, 3)

for row_lbl, *keys in [
    ("CD9",  "p9_A","p9_A","p9_B","p9_A","p9_B","p9_C","p9_A","p9_A","p9_B","p9_A","p9_B","p9_C"),
    ("CD63", "p63_A","p63_A","p63_B","p63_A","p63_B","p63_C","p63_A","p63_A","p63_B","p63_A","p63_B","p63_C"),
    ("CD81", "p81_A","p81_A","p81_B","p81_A","p81_B","p81_C","p81_A","p81_A","p81_B","p81_A","p81_B","p81_C"),
]:
    row_vals = [row_lbl]
    for k, opt, linked_flag in zip(keys,
        [1,2,2,3,3,3,1,2,2,3,3,3],
        [False]*6 + [True]*6):
        src = indep_results if not linked_flag else linked_results
        v = src[opt]["main_res"].get(k, None) if opt in src else None
        row_vals.append(round(v,4) if v is not None else "N/A")
    ws_model.append(row_vals)

# Metrics: write once per model, blank for sub-population columns
# Col order: 1A-A | 2A-A | 2A-B | 3A-A | 3A-B | 3A-C | 1B-A | 2B-A | 2B-B | 3B-A | 3B-B | 3B-C
opt_col_map_07 = [(1,False,True),(2,False,True),(2,False,False),(3,False,True),(3,False,False),(3,False,False),
                  (1,True, True),(2,True, True),(2,True, False),(3,True, True),(3,True, False),(3,True, False)]
for metric, key in [("Model RMSE","rmse"),("Model R²","r2"),("AIC","AIC"),("BIC","BIC")]:
    row_vals = [metric]
    for opt, linked_flag, is_first in opt_col_map_07:
        src_d = linked_results if linked_flag else indep_results
        if is_first and opt in src_d:
            v = src_d[opt]["main_res"].get(key, float("nan"))
            row_vals.append(round(v,4) if not (isinstance(v,float) and np.isnan(v)) else "N/A")
        else:
            row_vals.append(None)
    ws_model.append(row_vals)

# EV count summary rows
ws_model.append([])
total_evs_row  = ["Total EVs in Population"]
any_marker_row = ["EVs with ≥1 Tetraspanin"]
for opt, linked_flag, pop in [(1,False,"A"),(2,False,"A"),(2,False,"B"),(3,False,"A"),(3,False,"B"),(3,False,"C"),
                               (1,True, "A"),(2,True, "A"),(2,True, "B"),(3,True, "A"),(3,True, "B"),(3,True, "C")]:
    src_d = linked_results if linked_flag else indep_results
    if opt in src_d:
        mr   = src_d[opt]["main_res"]
        frac = mr["pop_fracs"].get(pop)
        pp   = mr["pop_probs"].get(pop)
        total_evs_row .append(int(round(frac * TOTAL_EVS)) if frac is not None else "NA")
        any_marker_row.append(int(round(float(pp[:7].sum()) * frac * TOTAL_EVS))
                              if (frac is not None and pp is not None) else "NA")
    else:
        total_evs_row .append("NA")
        any_marker_row.append("NA")
ws_model.append(total_evs_row)
ws_model.append(any_marker_row)
auto_width(ws_model)

# ── Option_Desc tab ───────────────────────────────────────────────────────────
ws_desc = wb.create_sheet("Option_Desc")
ws_desc.append([None,None,None,"Population",None,None])
ws_desc.append([None,None,"Assortment_Type","A","B","C"])
style_header_row(ws_desc, 2)
for r in [
    ("Option",1,"Independent","Remaining","NA","NA"),
    ("Option",2,"Independent","Remaining",f"~{POP_B_FRAC_OF_CD63*100:.2f}% of CD63+","NA"),
    ("Option",3,"Independent","Remaining",f"~{POP_B_FRAC_OF_CD63*100:.2f}% of CD63+",
                                           f"~{POP_C_FRAC_OF_CD9*100:.2f}% of CD9+"),
    ("Option",4,"Linked","Remaining","NA","NA"),
    ("Option",5,"Linked","Remaining",f"~{POP_B_FRAC_OF_CD63*100:.2f}% of CD63+","NA"),
    ("Option",6,"Linked","Remaining",f"~{POP_B_FRAC_OF_CD63*100:.2f}% of CD63+",
                                      f"~{POP_C_FRAC_OF_CD9*100:.2f}% of CD9+"),
]:
    ws_desc.append(list(r))
auto_width(ws_desc)

fpath_linked = save_wb(wb, "Cell_1.07_Linked_Models", ts)
print(f"\n✓ Linked models complete.")
print(f"  Tabs: Option1B/2B/3B_Params, Option1B/2B/3B_Phenotypes,")
print(f"        Model_Comparison_All6, Population_Data, Model, Option_Desc")
stop_logging(lf)


In [9]:
# Cell 1.08
"""
================================================================================
CELL 1.08: REPLICATE ROBUSTNESS ANALYSIS
================================================================================
PURPOSE:
  Use the 5 biological replicates to assess model robustness via three methods:
  1. Leave-One-Out Cross-Validation (LOO-CV): fit on N-1 replicates, predict
     the held-out replicate. Reports LOO-RMSE and LOO-R² for each option.
  2. Per-replicate fitting: fit each replicate independently, report parameter
     spread (mean, SD, CV) to assess parameter stability.
  3. Regularization path: sweep REGULARIZATION_WEIGHT across a log-scale range,
     record LOO-CV RMSE at each value to identify the optimal regularization.

OUTPUT:
  Excel: Cell_1.08_Robustness_20260320_XXXXXX.xlsx
    Tabs: LOO_CV_Results, Per_Replicate_Fits, Reg_Path, Summary
  Debug: Cell_1.08_Debug_<ts>.txt
================================================================================
"""

lf, ts = start_logging("Cell_1.08")
_cell_start = time.time()

print("=" * 70)
print("CELL 1.08: REPLICATE ROBUSTNESS ANALYSIS")
print("=" * 70)

# ── Configuration (all values defined in Cell 1.01) ───────────────────────────
# ROB_OPTIONS, ROB_LINKED, DE_MAXITER_SUB, DE_POPSIZE_SUB,
# REG_PATH_EMPIRICAL, REG_PATH_FINE, EWC_W_*, EWC_FALLBACK_*
# are all set in Cell 1.01. Edit there, not here.
# REG_PATH_VALUES kept as alias for any legacy references:
REG_PATH_VALUES = np.logspace(-4, 0, 25)  # legacy alias; not used by Sections 3A/3B

# ── Helper: fit one dataset ────────────────────────────────────────────────────
def _fit_single(obs, total, cd63_pos, cd9_pos, option, linked,
                reg_weight_override=None, fast_mode=False):
    # Compute chi2, Cramers V, p-value, and observed/expected counts from predicted 8-prob vector.
    _de_maxiter = DE_MAXITER_SUB if fast_mode else DE_MAXITER
    _de_popsize = DE_POPSIZE_SUB if fast_mode else DE_POPSIZE
    import copy
    orig_reg = REGULARIZATION_WEIGHT
    # Temporarily override regularization if requested
    if reg_weight_override is not None:
        import ev_objective_rw as _evm
        # We use the local make_objective which reads REGULARIZATION_WEIGHT from globals
        # For simplicity, monkey-patch the global
        import builtins
        _saved = globals().get('REGULARIZATION_WEIGHT')

    obj = make_objective(option, linked, obs, total, cd63_pos, cd9_pos,
                         POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)
    bounds_lo, bounds_hi = make_bounds(option, linked,
                                       POP_B_FRAC_OF_CD63, POP_B_FRAC_SD,
                                       POP_C_FRAC_OF_CD9,  POP_C_FRAC_SD)
    bounds_scipy = list(zip(bounds_lo, bounds_hi))
    try:
        de = differential_evolution(obj, bounds_scipy, seed=RANDOM_SEED,
                                    maxiter=_de_maxiter, popsize=_de_popsize, tol=1e-9,
                                    polish=True, workers=1, updating='immediate')
        lbfgs = minimize(obj, de.x, method="L-BFGS-B", bounds=bounds_scipy,
                         options={"maxiter": MAX_ITER, "ftol": 1e-12})
        best_x = lbfgs.x if lbfgs.fun < de.fun else de.x
        res = decode_result(best_x, option, linked, obs, total,
                            cd63_pos, cd9_pos,
                            POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)
        # Compute rmse and r2 against the training data (same as Cells 1.06/1.07)
        pp = res["predicted_probs"]
        pred_counts = pp * total
        ss_res = float(np.sum((obs - pred_counts)**2))
        ss_tot = float(np.sum((obs - obs.mean())**2))
        res["rmse"] = float(np.sqrt(np.mean((obs - pred_counts)**2)))
        res["r2"]   = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
        return res
    except Exception as e:
        import traceback as _tb
        print(f"\n⚠ _fit_single FAILED (option={option}, linked={linked}, "
              f"total={total}): {type(e).__name__}: {e}")
        _tb.print_exc()
        return None

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1: LEAVE-ONE-OUT CROSS-VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
#print("SECTION 1: Leave-One-Out Cross-Validation")
print("SECTION 1: Replicate Consistency Check")
print("  (Technical replicates: measures internal consistency, NOT generalizability)")
print("─" * 60)

loo_results = []
n_rep = replicate_data.shape[1]   # 5

for option in ROB_OPTIONS:
    for linked in ROB_LINKED:
        suffix = "B" if linked else "A"
        label  = f"Option {option}{suffix}"
        print(f"\n  {label}...")
        fold_rmse = []; fold_r2 = []

        for held_out in range(n_rep):
            train_idx = [j for j in range(n_rep) if j != held_out]
            train_obs = replicate_data[:, train_idx].sum(axis=1).astype(float)
            train_tot = int(train_obs.sum())
            train_cd63 = int(train_obs[CD63_POS].sum())
            train_cd9  = int(train_obs[CD9_POS].sum())

            test_obs  = replicate_data[:, held_out].astype(float)
            test_tot  = int(test_obs.sum())

            res = _fit_single(train_obs, train_tot, train_cd63, train_cd9,
                              option, linked, fast_mode=True)
            if res is None:
                continue

            # Predict held-out replicate
            pp_pred  = res["predicted_probs"]
            pred_counts = pp_pred * test_tot
            rmse = float(np.sqrt(np.mean((test_obs - pred_counts)**2)))
            ss_res = float(np.sum((test_obs - pred_counts)**2))
            ss_tot = float(np.sum((test_obs - test_obs.mean())**2))
            r2 = 1 - ss_res/ss_tot if ss_tot > 0 else float('nan')
            fold_rmse.append(rmse); fold_r2.append(r2)
            print(f"    Rep {held_out+1} (held out, consistency check): RMSE={rmse:.1f}  R²={r2:.4f}")

        mean_rmse = np.mean(fold_rmse) if fold_rmse else float('nan')
        mean_r2   = np.mean(fold_r2)   if fold_r2   else float('nan')
        sd_rmse   = np.std(fold_rmse, ddof=1) if len(fold_rmse)>1 else float('nan')
        print(f"  → Consistency RMSE: {mean_rmse:.1f} ± {sd_rmse:.1f}   Consistency R²: {mean_r2:.4f}")
        loo_results.append({
            "Model":        label,
            "Option":       option,
            "Linked":       linked,
            "LOO_RMSE_mean": round(mean_rmse, 3),
            "LOO_RMSE_sd":   round(sd_rmse, 3),
            "LOO_R2_mean":   round(mean_r2, 6),
            "LOO_RMSE_CV":   round(sd_rmse / mean_rmse, 4) if mean_rmse > 0 else None,
            "N_folds":       len(fold_rmse),
        })

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2: PER-REPLICATE FITTING
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
#print("SECTION 2: Per-Replicate Parameter Stability")
print("SECTION 2: Per-Replicate Parameter Precision")
print("  (Technical replicates: CV reflects measurement precision, not biological variability)")
print("─" * 60)

per_rep_results = {}
for option in ROB_OPTIONS:
    for linked in ROB_LINKED:
        suffix = "B" if linked else "A"
        label  = f"Option {option}{suffix}"
        print(f"\n  {label}...")
        rep_params = []
        for j in range(n_rep):
            rep_obs  = replicate_data[:, j].astype(float)
            rep_tot  = int(rep_obs.sum())
            rep_cd63 = int(rep_obs[CD63_POS].sum())
            rep_cd9  = int(rep_obs[CD9_POS].sum())
            res = _fit_single(rep_obs, rep_tot, rep_cd63, rep_cd9, option, linked,
                              fast_mode=True)
            if res:
                rep_params.append(res)
                print(f"    Rep {j+1}: RMSE={res.get('rmse', float('nan')):.1f}  "
                      f"p9_A={res.get('p9_A', float('nan')):.4f}  "
                      f"phi_9_81_A={res.get('phi_9_81_A', float('nan')):.4f}")
        per_rep_results[(option, linked)] = rep_params

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 3: REGULARIZATION PATH
# ═══════════════════════════════════════════════════════════════════════════════
# Two-part test:
#   Part A — Fixed empirical checkpoints at 0, 10, 50, 100, 500, 1000, 2000.
#             These large values demonstrate conclusively that the optimum is
#             at reg=0 and not a local minimum concealed in a narrow low range.
#   Part B — Fine log-scale grid (existing REG_PATH_VALUES) for smooth curve.
#   Both parts are written to the Reg_Path tab, separated by a blank row.
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
#print("SECTION 3: Regularization Path (LOO-RMSE vs regularization weight)")
print("SECTION 3: Regularization Path (Consistency-RMSE vs regularization weight)")
print("  (Technical replicates: assesses whether regularization improves consistency across runs)")
print("─" * 60)

_reg_weight_before_sweep = REG_FRAC

def _loo_rmse_at(reg_w, option=2, linked=True):
    """Evaluate LOO-RMSE at a single regularization weight for any option.
    REG_FRAC only affects fB/fC population size fraction parameters.
    Options 1A/1B have no sub-populations so are invariant to REG_FRAC.
    Options 2A/2B have fB; Options 3A/3B have both fB and fC.
    """
    globals()['REG_FRAC'] = reg_w
    fold_rmse = []
    for held_out in range(n_rep):
        train_idx  = [j for j in range(n_rep) if j != held_out]
        train_obs  = replicate_data[:, train_idx].sum(axis=1).astype(float)
        train_tot  = int(train_obs.sum())
        train_cd63 = int(train_obs[CD63_POS].sum())
        train_cd9  = int(train_obs[CD9_POS].sum())
        test_obs   = replicate_data[:, held_out].astype(float)
        test_tot   = int(test_obs.sum())
        res = _fit_single(train_obs, train_tot, train_cd63, train_cd9, option, linked,
                          fast_mode=True)
        if res:
            pred = res["predicted_probs"] * test_tot
            fold_rmse.append(float(np.sqrt(np.mean((test_obs - pred)**2))))
    globals()['REG_FRAC'] = _reg_weight_before_sweep
    return float(np.mean(fold_rmse)) if fold_rmse else float('nan')

# ── Part A: Empirical fixed checkpoints — all REG_FRAC-sensitive models ───────
# Only Options 2 and 3 (both independent and linked) have fB/fC parameters
# that REG_FRAC acts on. Option 1 models are invariant and are skipped.
_REG_SENSITIVE_MODELS = [
    (2, False, "2A"), (2, True, "2B"),
    (3, False, "3A"), (3, True, "3B"),
]
EMPIRICAL_REG_VALUES = REG_PATH_EMPIRICAL   # defined in Cell 1.01
print(f"  Part A — Empirical checkpoints: {EMPIRICAL_REG_VALUES}")
print(f"  (Options 1A/1B omitted — REG_FRAC has no effect when there are no sub-populations)")
empirical_records = []
for reg_w in EMPIRICAL_REG_VALUES:
    for opt, lnk, mdl_label in _REG_SENSITIVE_MODELS:
        rmse = _loo_rmse_at(reg_w, option=opt, linked=lnk)
        print(f"    reg={reg_w:6g}  {mdl_label}  LOO-RMSE={rmse:.4f}")
        empirical_records.append({"Model": mdl_label, "Reg_Weight": reg_w,
                                   "LOO_RMSE": round(rmse, 4), "Section": "A_Empirical"})

# ── Part B: Targeted fine grid ────────────────────────────────────────────────
# Reduced from 25 to 6 points (76% fewer fits) while preserving:
#   - reg=0 anchor (the published optimal value)
#   - sufficient resolution to show the curve is flat near 0 and rises at large reg
# Full 25-point grid is no longer needed: Part A already showed reg=0 is optimal
# at large values; Part B confirms the near-zero region and provides a publishable curve.
_FINE_REG_VALUES = np.array(REG_PATH_FINE)   # defined in Cell 1.01
print(f"\n  Part B — Targeted fine grid ({len(_FINE_REG_VALUES)} points: {list(_FINE_REG_VALUES)})")
fine_records = []
for reg_w in _FINE_REG_VALUES:
    for opt, lnk, mdl_label in _REG_SENSITIVE_MODELS:
        rmse = _loo_rmse_at(reg_w, option=opt, linked=lnk)
        print(f"    reg={reg_w:8.4f}  {mdl_label}  LOO-RMSE={rmse:.4f}")
        fine_records.append({"Model": mdl_label, "Reg_Weight": round(reg_w, 6),
                              "LOO_RMSE": round(rmse, 4), "Section": "B_Fine_Grid"})

globals()['REG_FRAC'] = _reg_weight_before_sweep

# Find optimal per model (minimum LOO-RMSE over fine grid)
reg_path_records = empirical_records + fine_records
_optimal_by_model = {}
for mdl_label in [m[2] for m in _REG_SENSITIVE_MODELS]:
    model_fine = [r for r in fine_records if r["Model"] == mdl_label]
    if model_fine:
        best = min(model_fine, key=lambda x: x["LOO_RMSE"])
        _optimal_by_model[mdl_label] = best
        emp_zero = next((r for r in empirical_records
                         if r["Model"] == mdl_label and r["Reg_Weight"] == 0), None)
        print(f"\n  {mdl_label}: fine grid min reg={best['Reg_Weight']}  "
              f"LOO-RMSE={best['LOO_RMSE']:.4f}")
        if emp_zero and emp_zero['LOO_RMSE'] <= best['LOO_RMSE'] + 0.1:
            print(f"    ✓ reg=0 performs as well — regularization not beneficial for {mdl_label}")
            _optimal_by_model[mdl_label] = emp_zero

# Use Option 2B result for the legacy optimal_reg variable (used in Summary tab)
optimal_reg = _optimal_by_model.get("2B", empirical_records[0])

# ═══════════════════════════════════════════════════════════════════════════════
# COMPOSITE MODEL RANKING SCORE
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
print("COMPOSITE MODEL RANKING (Evidence-Weighted Composite — see EWC weights above)")
print("─" * 60)

# Pull BIC, bootstrap success rate, and n_params from nested result dicts
# BIC lives in result["main_res"]["BIC"]
# bootstrap success rate = non-None rows in result["boot_decoded"] / total boot samples
# n_params lives in result["main_res"]["n_params"] (via param_count stored in decode_result)
_bic_map  = {}
_boot_map = {}
_k_map    = {}
_rmse_map = {}   # in-sample RMSE from main fit
_r2_map   = {}   # in-sample R² from main fit
for opt in [1, 2, 3]:
    for linked, suffix, src_dict in [(False, "A", indep_results), (True, "B", linked_results)]:
        label = f"Option {opt}{suffix}"
        if opt in src_dict:
            res = src_dict[opt]
            _bic_map[label]  = res["main_res"].get("BIC",  float("nan"))
            _k_map[label]    = res["main_res"].get("n_params", float("nan"))
            _rmse_map[label] = res["main_res"].get("rmse",  float("nan"))
            _r2_map[label]   = res["main_res"].get("r2",    float("nan"))
            bd = res.get("boot_decoded", [])
            if bd:
                n_ok = sum(1 for d in bd if d is not None)
                _boot_map[label] = round(100.0 * n_ok / len(bd), 1)
            else:
                _boot_map[label] = 100.0
        else:
            _bic_map[label]  = float("nan")
            _boot_map[label] = 100.0
            _k_map[label]    = param_count(opt, linked)  # always computable
            _rmse_map[label] = float("nan")
            _r2_map[label]   = float("nan")

_loo_rmse_vals = [r["LOO_RMSE_mean"] for r in loo_results]
_loo_r2_vals   = [r["LOO_R2_mean"]   for r in loo_results]
_loo_sd_vals   = [r["LOO_RMSE_sd"]   for r in loo_results]
_bic_vals      = [_bic_map.get(r["Model"], float("nan")) for r in loo_results]
_boot_vals     = [_boot_map.get(r["Model"], 100.0) for r in loo_results]
_k_vals        = [_k_map.get(r["Model"], float("nan")) for r in loo_results]
_bic_finite    = [v for v in _bic_vals if not np.isnan(v)]
_bic_lo, _bic_hi = min(_bic_finite), max(_bic_finite)
_bic_min_val   = _bic_lo

# DELTA_BIC within linked group — for group-normalized BIC component.
# Independent models are separated from linked by n_BF (cross-group);
# within the linked group, DELTA_BIC correctly distinguishes 1B/2B/3B.
_linked_bic_vals = [_bic_map.get(f"Option {opt}B", float("nan"))
                    for opt in [1, 2, 3]]
_linked_bic_finite = [v for v in _linked_bic_vals if not np.isnan(v)]
_linked_bic_min = min(_linked_bic_finite) if _linked_bic_finite else float("nan")
_linked_bic_max = max(_linked_bic_finite) if _linked_bic_finite else float("nan")
# Range for normalizing DELTA_BIC among linked models only
_linked_dbic_hi = _linked_bic_max - _linked_bic_min  # e.g. 20.5

def _norm01(x, lo, hi, invert=False):
    if hi == lo: return 0.5
    s = (x - lo) / (hi - lo)
    return float(np.clip(1 - s if invert else s, 0, 1))

# ── Evidence-Weighted Composite (EWC) scoring ─────────────────────────────────
# All EWC_W_* weights and EWC_FALLBACK_* values are defined in Cell 1.01.
# Seven components, each normalized to [0,1]:
# 1. Consistency-RMSE accuracy  (EWC_W_ACC)  — lower error is better
# 2. Consistency-R²             (EWC_W_R2)   — higher variance explained is better
# 3. Consistency stability      (EWC_W_STAB) — lower SD across replicates is better
# 4. ΔBIC within linked group   (EWC_W_BIC)  — normalized among linked models only;
#                                               0 for independent models (cross-group
#                                               penalty carried entirely by BF)
# 5. Bayes factor approx        (EWC_W_BF)   — exp(-0.5·ΔBIC); near-zero when ΔBIC>10
# 6. Bootstrap convergence rate (EWC_W_BOOT) — fraction of fits that converged
# 7. Parameter count k          (EWC_W_K)    — direct parsimony; fewer params = better

print("\n" + "─" * 72)
print("EVIDENCE-WEIGHTED COMPOSITE (EWC) MODEL RANKING")
#print("LOO-Acc 20% | LOO-R² 15% | LOO-Stab 10% | BIC 25% | BF 15% | Boot 10% | k 5%")
print(f"Consistency-Acc {EWC_W_ACC*100:.0f}% | Consistency-R² {EWC_W_R2*100:.0f}% | "
      f"Consistency-Stab {EWC_W_STAB*100:.0f}% | ΔBIC-linked {EWC_W_BIC*100:.0f}% | "
      f"BF {EWC_W_BF*100:.0f}% | Boot {EWC_W_BOOT*100:.0f}% | k {EWC_W_K*100:.0f}%")
print(f"  (BIC+BF={( EWC_W_BIC+EWC_W_BF)*100:.0f}%, "
      f"Consistency metrics={(EWC_W_ACC+EWC_W_R2+EWC_W_STAB)*100:.0f}%)")
print("─" * 72)

# Bayes factor approximation: exp(-0.5 * ΔBIC) relative to best BIC model
_bf_vals = []
for bic in _bic_vals:
    if np.isnan(bic):
        _bf_vals.append(float("nan"))
    else:
        _bf_vals.append(float(np.exp(-0.5 * (bic - _bic_min_val))))
_bf_finite = [v for v in _bf_vals if not np.isnan(v)]
_bf_lo, _bf_hi = min(_bf_finite), max(_bf_finite)

# LOO SD (stability): lower SD = more consistent across folds = better
_loo_sd_finite = [v for v in _loo_sd_vals if not np.isnan(v)]
_loo_sd_lo, _loo_sd_hi = min(_loo_sd_finite), max(_loo_sd_finite)

# k (parameter count): fewer = better
_k_finite = [v for v in _k_vals if not np.isnan(v)]
_k_lo, _k_hi = min(_k_finite), max(_k_finite)

composite_scores = []
ewc_table_rows   = []

print(f"  {'Model':<10} {'LOO-RMSE':>9} {'LOO-R²':>7} {'LOO-SD':>7} "
      f"{'ΔBIC':>7} {'BF':>6} {'Boot%':>6} {'k':>4} {'EWC':>7}")
print(f"  {'-'*72}")

for r, bic, boot, bf, k, loo_sd in zip(
        loo_results, _bic_vals, _boot_vals, _bf_vals, _k_vals, _loo_sd_vals):
    n_acc   = _norm01(r["LOO_RMSE_mean"], min(_loo_rmse_vals), max(_loo_rmse_vals), invert=True)
    n_r2    = _norm01(r["LOO_R2_mean"],   min(_loo_r2_vals),   max(_loo_r2_vals))
    n_stab  = _norm01(loo_sd, _loo_sd_lo, _loo_sd_hi, invert=True)
    # n_bic: ΔBIC normalized within linked group only.
    # Independent models receive 0 — cross-group separation is carried by n_bf.
    _is_linked = r["Model"].endswith("B")
    if _is_linked and not np.isnan(bic) and _linked_dbic_hi > 0:
        _model_dbic = bic - _linked_bic_min
        n_bic = float(np.clip(1.0 - _model_dbic / _linked_dbic_hi, 0.0, 1.0))
    elif _is_linked and not np.isnan(bic):
        n_bic = 1.0   # only one linked model with a valid BIC
    else:
        n_bic = 0.0   # independent models: no credit on this component
    # EWC_FALLBACK_BF / EWC_FALLBACK_K defined in Cell 1.01.
    # 0.0 = a missing model is fully penalized, not awarded a neutral midpoint score.
    n_bf    = _norm01(bf,  _bf_lo,  _bf_hi)               if not np.isnan(bf)  else EWC_FALLBACK_BF
    n_boot  = _norm01(boot, min(_boot_vals), max(_boot_vals))
    n_k     = _norm01(k,   _k_lo,   _k_hi,  invert=True)  if not np.isnan(k)   else EWC_FALLBACK_K

    ewc = (EWC_W_ACC  * n_acc  +
           EWC_W_R2   * n_r2   +
           EWC_W_STAB * n_stab +
           EWC_W_BIC  * n_bic  +
           EWC_W_BF   * n_bf   +
           EWC_W_BOOT * n_boot +
           EWC_W_K    * n_k)
    # Biological replicate weights (swap EWC_W_* definitions in Cell 1.08 config block above):
    # ewc = (0.20 * n_acc  +
    #        0.15 * n_r2   +
    #        0.10 * n_stab +
    #        0.25 * n_bic  +
    #        0.15 * n_bf   +
    #        0.10 * n_boot +
    #        0.05 * n_k)
    delta_bic = bic - _bic_min_val if not np.isnan(bic) else float("nan")
    bf_disp   = bf  if not np.isnan(bf)  else float("nan")
    k_disp    = int(k) if not np.isnan(k) else "?"

    composite_scores.append({
        "Model":     r["Model"],
        "EWC_Score": round(ewc, 4),
        "LOO_RMSE":  r["LOO_RMSE_mean"],
        "LOO_R2":    r["LOO_R2_mean"],
        "LOO_SD":    round(loo_sd, 2),
        "BIC":       round(bic, 1) if not np.isnan(bic) else None,
        "DELTA_BIC": round(delta_bic, 1) if not np.isnan(delta_bic) else None,
        "BF_Approx": round(bf_disp, 5) if not np.isnan(bf_disp) else None,
        "Boot_Pct":  boot,
        "k":         k_disp,
        "n_LOO_RMSE": round(n_acc,  3),
        "n_LOO_R2":   round(n_r2,   3),
        "n_LOO_Stab": round(n_stab, 3),
        "n_DELTA_BIC": round(n_bic,  3),
        "n_BF":       round(n_bf,   3),
        "n_Boot":     round(n_boot, 3),
        "n_k":        round(n_k,    3),
    })
    print(f"  {r['Model']:<10} {r['LOO_RMSE_mean']:>9.1f} {r['LOO_R2_mean']:>7.4f} "
          f"{loo_sd:>7.1f} {delta_bic:>7.1f} {bf_disp:>6.4f} {boot:>6.1f} "
          f"{k_disp:>4} {ewc:>7.4f}")

best_ewc = max(composite_scores, key=lambda x: x["EWC_Score"])
print(f"\n  → Best model by EWC score: {best_ewc['Model']}  (EWC={best_ewc['EWC_Score']:.4f})")
print(f"  → BF interpretation: BF=1.0 = best BIC model; BF<0.05 = ΔBIC>6 (strong evidence against)")
print(f"  → BF<0.01 = ΔBIC>9.2 (very strong evidence against adding complexity)")

# Identify the minimum sufficient model (best EWC that is also within ΔBIC<10 of the winner)
_linked_candidates = [s for s in composite_scores if "B" in s["Model"]
                      and s["DELTA_BIC"] is not None and s["DELTA_BIC"] < 10]
if _linked_candidates:
    _min_sufficient = min(_linked_candidates, key=lambda x: x["k"] if isinstance(x["k"], int) else 99)
    print(f"  → Minimum sufficient linked model (ΔBIC<10): {_min_sufficient['Model']}")

# (duplicate variable initialization removed — all variables already computed above)

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD EXCEL OUTPUT
# ═══════════════════════════════════════════════════════════════════════════════
print("\nBuilding Excel output...")
wb = Workbook(); wb.remove(wb.active)

# LOO-CV sheet
ws_loo = wb.create_sheet("LOO_CV_Results")
ws_loo.append(["Model","Option","Linked","LOO_RMSE_mean","LOO_RMSE_sd","LOO_R2_mean","LOO_RMSE_CV","N_folds"])
style_header_row(ws_loo, 1)
for r in loo_results:
    ws_loo.append([r["Model"], r["Option"], r["Linked"],
                   r["LOO_RMSE_mean"], r["LOO_RMSE_sd"], r["LOO_R2_mean"],
                   r.get("LOO_RMSE_CV"), r["N_folds"]])
auto_width(ws_loo)

# Per-replicate sheet
ws_rep = wb.create_sheet("Per_Replicate_Fits")
param_keys = ["p9_A","p81_A","p63_A","phi_9_81_A","phi_9_63_A",
              "p9_B","p81_B","phi_9_81_B",
              "p81_C","p63_C","phi_81_63_C",
              "fB","fC","rmse","r2"]
ws_rep.append(["Model","Replicate"] + param_keys)
style_header_row(ws_rep, 1)
for (option, linked), reps in per_rep_results.items():
    suffix = "B" if linked else "A"
    label  = f"Option {option}{suffix}"
    for j, res in enumerate(reps):
        row = [label, f"Rep {j+1}"]
        for k in param_keys:
            v = res.get(k, float('nan'))
            if v is None or (isinstance(v, float) and np.isnan(v)):
                row.append(None)
            else:
                row.append(round(float(v), 6))
        ws_rep.append(row)
    # Summary stats
    if reps:
        ws_rep.append([label, "MEAN"] +
                      [round(np.nanmean([r.get(k, float('nan')) for r in reps]), 6) for k in param_keys])
        ws_rep.append([label, "SD"] +
                      [round(np.nanstd([r.get(k, float('nan')) for r in reps], ddof=1), 6) for k in param_keys])
        ws_rep.append([label, "CV"] +
                      [round(abs(np.nanstd([r.get(k, float('nan')) for r in reps], ddof=1) /
                                np.nanmean([r.get(k, float('nan')) for r in reps])), 4)
                       if np.nanmean([r.get(k, float('nan')) for r in reps]) != 0 else None
                       for k in param_keys])
        # Identifiability score: mean(CI_width / |point_estimate|) across bootstrap
        # A score near 0 = well-determined; >0.5 = poorly identified
        _means = [round(np.nanmean([r.get(k, float('nan')) for r in reps]), 6) for k in param_keys]
        _sds   = [round(np.nanstd( [r.get(k, float('nan')) for r in reps], ddof=1), 6) for k in param_keys]
        ws_rep.append([label, "ID_SCORE"] +
                      [(round(sd / abs(mean), 4) if (mean is not None and mean != 0 and sd is not None)
                        else None)
                       for mean, sd in zip(_means, _sds)])
        # Multimodality flag: CV > 0.5 on any structural parameter signals multiple local minima
        cvs = []
        for k in param_keys:
            vals = [r.get(k, float('nan')) for r in reps]
            mean_v = np.nanmean(vals)
            std_v  = np.nanstd(vals, ddof=1)
            cvs.append(abs(std_v / mean_v) if mean_v != 0 else None)
        # ── Parameter stability assessment ────────────────────────────────────
        # Strategy: compare inter-replicate range of each phi parameter to the
        # bootstrap CI width from the full pooled fit. If the range is > 2× the
        # CI width, the parameter is unstable across replicates beyond what
        # sampling variation explains. This avoids the arbitrary CV > 0.5 problem
        # and correctly handles parameters with small means (like fB, fC).
        _instability_flags = []
        _instability_reasons = []
        _phi_keys_to_check = ["phi_9_81_A", "phi_9_63_A", "phi_9_81_B", "phi_81_63_C"]
        _prob_keys_to_check = ["p9_A", "p81_A", "p63_A", "p9_B", "p81_B", "p81_C", "p63_C"]

        # Get bootstrap CIs from main fit for this option (if available)
        _main_res_ref = linked_results.get(option, {}) if linked else indep_results.get(option, {})
        _boot_params = _main_res_ref.get("boot_decoded", [])

        for _pk in _phi_keys_to_check + _prob_keys_to_check:
            _rep_vals = [r.get(_pk) for r in reps if r.get(_pk) is not None
                         and not (isinstance(r.get(_pk), float) and np.isnan(r.get(_pk)))]
            if len(_rep_vals) < 2:
                continue
            _rep_range = max(_rep_vals) - min(_rep_vals)
            # Get bootstrap CI width for comparison
            _boot_vals_pk = [d.get(_pk) for d in _boot_params
                             if d is not None and d.get(_pk) is not None]
            if len(_boot_vals_pk) > 10:
                _ci_width = float(np.percentile(_boot_vals_pk, 97.5) -
                                  np.percentile(_boot_vals_pk, 2.5))
                _ratio = _rep_range / _ci_width if _ci_width > 1e-10 else 0.0
                if _ratio > 2.0:
                    _instability_flags.append(_pk)
                    _instability_reasons.append(f"{_pk}: range={_rep_range:.4f} "
                                                f"({_ratio:.1f}× bootstrap CI width)")
            else:
                # Fallback: flag if CV > 0.3 for phi params specifically
                _mean_v = float(np.nanmean(_rep_vals))
                _sd_v   = float(np.nanstd(_rep_vals, ddof=1))
                if abs(_mean_v) > 1e-6 and (_sd_v / abs(_mean_v)) > 0.3 and _pk in _phi_keys_to_check:
                    _instability_flags.append(_pk)
                    _instability_reasons.append(f"{_pk}: CV={_sd_v/abs(_mean_v):.2f} (no bootstrap ref)")

        _stability_result = "UNSTABLE" if _instability_flags else "STABLE"
        _reason_str = "; ".join(_instability_reasons) if _instability_reasons else "All key parameters stable across replicates"
        ws_rep.append([label, "STABILITY_FLAG", _stability_result, _reason_str])
        ws_rep.append([label, "UNSTABLE_PARAMS"] +
                      ([p for p in param_keys if p in _instability_flags] or ["none"]))
        ws_rep.append([])
auto_width(ws_rep)

# Regularization path sheet
ws_reg = wb.create_sheet("Reg_Path")
ws_reg.append(["Model", "Section", "Reg_Weight", "LOO_RMSE", "Notes"])
style_header_row(ws_reg, 1)
for r in reg_path_records:
    mdl = r.get("Model", "2B")
    opt_for_mdl = _optimal_by_model.get(mdl, {})
    is_opt = (r["Reg_Weight"] == opt_for_mdl.get("Reg_Weight") and
              r["Section"] == opt_for_mdl.get("Section", ""))
    note = "← OPTIMAL for this model" if is_opt else ""
    ws_reg.append([mdl, r.get("Section",""), r["Reg_Weight"], r["LOO_RMSE"], note])
ws_reg.append([])
ws_reg.append(["Note:", "REG_FRAC penalizes fB and fC toward CNV-derived prior means.",
               "Options 1A/1B omitted — they have no population fractions to regularize.",
               "Optimal = reg_w minimizing LOO-RMSE over the fine grid for each model."])
auto_width(ws_reg)

# Summary
ws_sum = wb.create_sheet("Summary")
ws_sum.append(["Metric", "Value", "Description"])
style_header_row(ws_sum, 1)
# Best LOO-CV model
best_loo = min(loo_results, key=lambda x: x["LOO_RMSE_mean"])
ws_sum.append(["Best model (LOO-RMSE)", best_loo["Model"],
               f"Mean LOO-RMSE = {best_loo['LOO_RMSE_mean']}"])
ws_sum.append(["Optimal reg weight", optimal_reg["Reg_Weight"],
               f"Consistency-RMSE = {optimal_reg['LOO_RMSE']} for {optimal_reg.get('Model', '2B')}"])
ws_sum.append(["Current reg weight", _reg_weight_before_sweep,
               "Set in Cell 1.01 (N-scaled: effective penalty = weight × N × ||Δparams||²)"])
ws_sum.append(["N (pooled EVs)", TOTAL_EVS,
               "Regularization penalty is proportional to this"])
ws_sum.append(["Effective penalty at opt reg", 
               round(optimal_reg["Reg_Weight"] * TOTAL_EVS * 0.002, 4),
               "= opt_reg × N × typical_||Δparams||² (||Δ||²≈0.002 for 5-param model)"])
auto_width(ws_sum)

# Composite ranking sheet
ws_ewc = wb.create_sheet("EWC_Model_Ranking")
ewc_cols = ["Model","EWC_Score","LOO_RMSE","LOO_R2","LOO_SD","BIC","DELTA_BIC",
            "BF_Approx","Boot_Pct","k","n_LOO_RMSE","n_LOO_R2","n_LOO_Stab",
            "n_DELTA_BIC","n_BF","n_Boot","n_k"]
ws_ewc.append(ewc_cols)
style_header_row(ws_ewc, 1)
for s in sorted(composite_scores, key=lambda x: x["EWC_Score"], reverse=True):
    ws_ewc.append([s.get(c) for c in ewc_cols])
ws_ewc.append([])
ws_ewc.append(["Weights (active):",
               f"Consistency-Acc={EWC_W_ACC*100:.0f}%",
               f"Consistency-R²={EWC_W_R2*100:.0f}%",
               f"Consistency-Stab={EWC_W_STAB*100:.0f}%",
               f"ΔBIC-linked={EWC_W_BIC*100:.0f}%",
               f"BF={EWC_W_BF*100:.0f}%",
               f"Boot={EWC_W_BOOT*100:.0f}%",
               f"k={EWC_W_K*100:.0f}%"])
ws_ewc.append(["NOTE:", "Biological replicate weights: LOO-Acc=20%, LOO-R²=15%, LOO-Stab=10%, BIC=25%, BF=15%, Boot=10%, k=5%"])
ws_ewc.append(["BF note:", "exp(-0.5*ΔBIC); BF<0.05 when ΔBIC>6 (strong evidence against)"])
auto_width(ws_ewc)

# ── Chi-square and Cramér's V from bootstrap predicted probabilities ──────────
# Computed for (a) the total mixed population and (b) each sub-population
# separately using pop_probs from decode_result.
# p-value from chi2 distribution (df=1 for 2x2 table).
print("\n" + "─" * 60)
print("SECTION 4: Bootstrap Chi-square and Cramér's V Ranges")
print("─" * 60)

_CHI2_PAIRS = [
    ("CD9",  "CD81",  CD9_POS,  CD81_POS),
    ("CD9",  "CD63",  CD9_POS,  CD63_POS),
    ("CD81", "CD63",  CD81_POS, CD63_POS),
]

from scipy.stats import chi2 as _chi2_dist_s4

def _pred_chi2_full(pred_probs, n, idx_A, idx_B):
    # Compute chi2, Cramers V, p-value, and observed/expected counts from predicted 8-prob vector.
    obs_pp = n * sum(pred_probs[i] for i in idx_A if i in idx_B)
    obs_pn = n * sum(pred_probs[i] for i in idx_A if i not in idx_B)
    obs_np = n * sum(pred_probs[i] for i in idx_B if i not in idx_A)
    obs_nn = n * sum(pred_probs[i] for i in range(8) if i not in idx_A and i not in idx_B)
    p_a = (obs_pp + obs_pn) / n
    p_b = (obs_pp + obs_np) / n
    exp_pp = n * p_a * p_b
    exp_pn = n * p_a * (1 - p_b)
    exp_np = n * (1 - p_a) * p_b
    exp_nn = n * (1 - p_a) * (1 - p_b)
    def _t(o, e): return (o - e)**2 / e if e > 1e-6 else 0.0
    chi2_val = _t(obs_pp, exp_pp) + _t(obs_pn, exp_pn) + _t(obs_np, exp_np) + _t(obs_nn, exp_nn)
    v = float(np.sqrt(chi2_val / n)) if n > 0 else float("nan")
    p_val = float(_chi2_dist_s4.sf(chi2_val, 1)) if chi2_val > 0 else 1.0
    return (float(chi2_val), v, p_val,
            round(obs_pp,1), round(obs_pn,1), round(obs_np,1), round(obs_nn,1),
            round(exp_pp,1), round(exp_pn,1), round(exp_np,1), round(exp_nn,1))

chi2_range_records = []

for opt in [1, 2, 3]:
    for linked, suffix, src_dict in [(False, "A", indep_results), (True, "B", linked_results)]:
        label = f"Option {opt}{suffix}"
        if opt not in src_dict:
            continue
        res = src_dict[opt]
        pp_main = res["main_res"].get("predicted_probs")
        if pp_main is None:
            continue
        bd = res.get("boot_decoded", [])
        boot_probs = [d["predicted_probs"] for d in bd if d is not None
                      and "predicted_probs" in d]
        # Also get per-sub-population probability vectors
        pop_probs_dict = res["main_res"].get("pop_probs", {"Total": pp_main})
        # Build list of (pop_label, n_evs, prob_vector) to analyze
        pop_analyses = [("Total (mixed)", TOTAL_EVS, pp_main)]
        fracs = res["main_res"].get("pop_fracs", {})
        for pop_key, pop_pp in pop_probs_dict.items():
            if pop_key == "A":
                frac = fracs.get("A", 1.0)
                pop_analyses.append((f"Pop A (background)", max(1, int(TOTAL_EVS * frac)), pop_pp))
            elif pop_key == "B":
                frac = fracs.get("B", 0.0)
                pop_analyses.append((f"Pop B (CD63-high)", max(1, int(TOTAL_EVS * frac)), pop_pp))
            elif pop_key == "C":
                frac = fracs.get("C", 0.0)
                pop_analyses.append((f"Pop C (CD9-high)", max(1, int(TOTAL_EVS * frac)), pop_pp))

        print(f"\n  {label}:")
        for pop_lbl, n_pop, pp_vec in pop_analyses:
            print(f"    [{pop_lbl}, n≈{n_pop}]")
            for mA, mB, idxA, idxB in _CHI2_PAIRS:
                chi2_m, v_m, p_m, o_pp,o_pn,o_np,o_nn, e_pp,e_pn,e_np,e_nn = \
                    _pred_chi2_full(pp_vec, n_pop, idxA, idxB)
                sig = ("***" if p_m < 0.001 else ("**" if p_m < 0.01 else
                       ("*" if p_m < 0.05 else "ns")))
                # Bootstrap CI for all populations.
                # boot_decoded entries contain pop_probs and pop_fracs from
                # decode_result, so sub-population vectors are already available.
                # For sub-populations, n_evs per bootstrap sample scales with
                # the bootstrap frac estimate, keeping counts consistent.
                b_chi2, b_v = [], []
                if boot_probs:
                    for bd_entry in res.get("boot_decoded", []):
                        if bd_entry is None:
                            continue
                        if pop_lbl.startswith("Total"):
                            bp = bd_entry.get("predicted_probs")
                            n_b = n_pop
                        else:
                            # Extract the matching sub-population vector
                            _pk = ("A" if "background" in pop_lbl else
                                   "B" if "CD63" in pop_lbl else
                                   "C" if "CD9" in pop_lbl else None)
                            if _pk is None:
                                continue
                            bp = bd_entry.get("pop_probs", {}).get(_pk)
                            if bp is None:
                                continue
                            # Scale n by the bootstrap frac estimate for this pop
                            _boot_frac = bd_entry.get("pop_fracs", {}).get(_pk, fracs.get(_pk, 0.0))
                            n_b = max(1, int(TOTAL_EVS * _boot_frac))
                        c2, vv, *_ = _pred_chi2_full(bp, n_b, idxA, idxB)
                        b_chi2.append(c2)
                        b_v.append(vv)

                if b_chi2:
                    chi2_lo = round(float(np.percentile(b_chi2, 2.5)), 2)
                    chi2_hi = round(float(np.percentile(b_chi2, 97.5)), 2)
                    v_lo    = round(float(np.percentile(b_v, 2.5)), 5)
                    v_hi    = round(float(np.percentile(b_v, 97.5)), 5)
                    n_boot  = len(b_chi2)
                else:
                    chi2_lo = chi2_hi = v_lo = v_hi = None
                    n_boot  = 0
                print(f"      {mA} vs {mB}: chi2={chi2_m:.1f}  p={p_m:.3e} {sig}  V={v_m:.4f}")
                chi2_range_records.append({
                    "Model": label, "Population": pop_lbl, "N_EVs": n_pop,
                    "Pair": f"{mA} vs {mB}",
                    "Chi2": round(chi2_m, 2),
                    "Chi2_CI_lo_95": chi2_lo, "Chi2_CI_hi_95": chi2_hi,
                    "p_value": round(p_m, 8),
                    "Significance": sig,
                    "CramersV": round(v_m, 5),
                    "CramersV_CI_lo_95": v_lo, "CramersV_CI_hi_95": v_hi,
                    "Obs_AB_pp": o_pp, "Obs_AB_pn": o_pn,
                    "Obs_AB_np": o_np, "Obs_AB_nn": o_nn,
                    "Exp_AB_pp": e_pp, "Exp_AB_pn": e_pn,
                    "Exp_AB_np": e_np, "Exp_AB_nn": e_nn,
                    "N_boot": n_boot,
                })

ws_chi2 = wb.create_sheet("Bootstrap_Chi2_CramerV")
chi2_cols = ["Model","Population","N_EVs","Pair",
             "Chi2","Chi2_CI_lo_95","Chi2_CI_hi_95",
             "p_value","Significance",
             "CramersV","CramersV_CI_lo_95","CramersV_CI_hi_95",
             "Obs_AB_pp","Obs_AB_pn","Obs_AB_np","Obs_AB_nn",
             "Exp_AB_pp","Exp_AB_pn","Exp_AB_np","Exp_AB_nn",
             "N_boot"]
ws_chi2.append(chi2_cols)
style_header_row(ws_chi2, 1)
for rec in chi2_range_records:
    ws_chi2.append([rec.get(c) for c in chi2_cols])
ws_chi2.append([])
ws_chi2.append(["COLUMN GUIDE"])
ws_chi2.append(["Chi2",          "Pearson chi-square statistic (df=1). Tests whether two markers co-package independently within this population."])
ws_chi2.append(["Chi2_CI_lo/hi_95", "95% bootstrap CI on chi2. For sub-populations, CIs use the per-bootstrap pop_probs and scaled n. Interpret with caution when N_EVs < 100 (expected cell counts may fall below 5)."])
ws_chi2.append(["p_value",       "P-value from chi2(df=1) distribution. H0: markers assort independently in this population."])
ws_chi2.append(["Significance",  "*** p<0.001  ** p<0.01  * p<0.05  ns not significant"])
ws_chi2.append(["CramersV",      "Effect size: V = sqrt(chi2/N). 0=independent, 1=perfectly linked. <0.1 weak, 0.1-0.3 moderate, >0.3 strong."])
ws_chi2.append(["Obs_AB_pp/pn/np/nn", "Predicted counts (from fitted model): A+B+, A+B-, A-B+, A-B- (scaled to N_EVs)."])
ws_chi2.append(["Exp_AB_pp/pn/np/nn", "Expected counts under independence given the marginals: P(A+)×P(B+)×N, etc."])
ws_chi2.append(["N_boot",        "Bootstrap samples used for CI (0 = CI not computed for this row)."])
ws_chi2.append([])
ws_chi2.append(["INTERPRETATION GUIDE"])
ws_chi2.append(["Independent models (1A/2A/3A) always show chi2≈0 by construction: they force independence within each population."])
ws_chi2.append(["Linked models (1B/2B/3B) show the CD9-CD81 co-packaging detected by the phi parameter."])
ws_chi2.append(["Per-sub-population rows show whether that specific sub-population contributes its own linkage signal."])
ws_chi2.append(["If V > 0.3 for a sub-population, the linkage within that sub-population is biologically substantial."])
auto_width(ws_chi2)

# ── Likelihood Ratio Tests ────────────────────────────────────────────────────
# Test nested models: 1A vs 1B, 2A vs 2B, 3A vs 3B (assortment type),
# and 1B vs 2B vs 3B (population structure)
print("\n" + "─" * 60)
print("SECTION 5: Likelihood Ratio Tests (LRT)")
print("─" * 60)

from scipy.stats import chi2 as chi2_dist

def _get_nll(opt, linked):
    d = linked_results if linked else indep_results
    if opt not in d:
        return float("nan")
    return float(d[opt]["main_res"].get("nll", float("nan")))

def _get_k(opt, linked):
    return param_count(opt, linked)

lrt_tests = [
    # (label, simple_opt, simple_linked, complex_opt, complex_linked, description)
    ("1A→1B", 1, False, 1, True,  "Add phi (linked assortment) to single population"),
    ("2A→2B", 2, False, 2, True,  "Add phi to two-population model"),
    ("3A→3B", 3, False, 3, True,  "Add phi to three-population model"),
    ("1B→2B", 1, True,  2, True,  "Add CD63-high sub-population (Pop B) to linked model"),
    ("2B→3B", 2, True,  3, True,  "Add CD9-high sub-population (Pop C) to linked model"),
    ("1B→3B", 1, True,  3, True,  "Add both sub-populations to simplest linked model"),
]

lrt_records = []
print(f"\n  {'Test':<10} {'LRT stat':>10} {'df':>4} {'p-value':>12} {'Sig':>6}  Description")
print(f"  {'-'*80}")
for label, s_opt, s_lnk, c_opt, c_lnk, desc in lrt_tests:
    nll_s = _get_nll(s_opt, s_lnk)
    nll_c = _get_nll(c_opt, c_lnk)
    k_s   = _get_k(s_opt, s_lnk)
    k_c   = _get_k(c_opt, c_lnk)
    if np.isnan(nll_s) or np.isnan(nll_c):
        continue
    lrt_stat = float(2 * (nll_s - nll_c))   # nll_s > nll_c since complex fits better
    df       = k_c - k_s
    p_val    = float(chi2_dist.sf(lrt_stat, df)) if df > 0 and lrt_stat > 0 else 1.0
    sig = ("***" if p_val < 0.001 else ("**" if p_val < 0.01 else
           ("*" if p_val < 0.05 else "ns")))
    d_s = linked_results if s_lnk else indep_results
    d_c = linked_results if c_lnk else indep_results
    bic_s = float(d_s[s_opt]["main_res"].get("BIC", float("nan"))) if s_opt in d_s else float("nan")
    bic_c = float(d_c[c_opt]["main_res"].get("BIC", float("nan"))) if c_opt in d_c else float("nan")
    delta_bic = bic_c - bic_s if not (np.isnan(bic_c) or np.isnan(bic_s)) else float("nan")
    print(f"  {label:<10} {lrt_stat:>10.2f} {df:>4} {p_val:>12.3e} {sig:>6}  {desc}")
    lrt_records.append({
        "Test": label,
        "Simple_Model": f"Option {s_opt}{'B' if s_lnk else 'A'}",
        "Complex_Model": f"Option {c_opt}{'B' if c_lnk else 'A'}",
        "NLL_Simple": round(nll_s, 3),
        "NLL_Complex": round(nll_c, 3),
        "k_Simple": k_s, "k_Complex": k_c, "df": df,
        "LRT_Statistic": round(lrt_stat, 3),
        "p_value": round(p_val, 8),
        "Significant_0.05": "Yes" if p_val < 0.05 else "No",
        "DELTA_BIC": round(delta_bic, 2) if not np.isnan(delta_bic) else None,
        "BIC_Favors": ("Simple" if delta_bic > 0 else "Complex") if not np.isnan(delta_bic) else None,
        "Interpretation": desc,
    })

ws_lrt = wb.create_sheet("LRT_Tests")
lrt_cols = ["Test","Simple_Model","Complex_Model","NLL_Simple","NLL_Complex",
            "k_Simple","k_Complex","df","LRT_Statistic","p_value",
            "Significant_0.05","DELTA_BIC","BIC_Favors","Interpretation"]
ws_lrt.append(lrt_cols)
style_header_row(ws_lrt, 1)
for rec in lrt_records:
    ws_lrt.append([rec.get(c) for c in lrt_cols])
ws_lrt.append([])
ws_lrt.append(["HOW TO READ THIS TABLE"])
ws_lrt.append(["LRT_Statistic",  "2 × (NLL_simple − NLL_complex). Larger = greater improvement from adding complexity."])
ws_lrt.append(["df",             "Degrees of freedom = k_complex − k_simple (number of additional parameters)."])
ws_lrt.append(["p_value",        "Probability of this LRT stat under H0 (simple model is true). Chi2(df) distribution."])
ws_lrt.append(["Significant_0.05",
               f"Yes if p < 0.05. At N={TOTAL_EVS:,} virtually any improvement will be "
               f"significant — see ΔBIC."])
ws_lrt.append(["DELTA_BIC",      "BIC_complex − BIC_simple. Positive = BIC penalizes the complex model more than it gains."])
ws_lrt.append(["BIC_Favors",
               f"Which model BIC supports. BIC penalizes each parameter by "
               f"ln({TOTAL_EVS})≈{np.log(TOTAL_EVS):.1f} vs AIC's 2."])
ws_lrt.append([])
ws_lrt.append(["SIGNIFICANCE THRESHOLDS"])
ws_lrt.append(["LRT p<0.05",     "Statistically significant improvement — but at large N even trivial gains are detectable."])
ws_lrt.append(["ΔBIC < 0",       "Complex model wins on parsimony-adjusted basis (strong evidence)."])
ws_lrt.append(["ΔBIC 0–2",       "Models indistinguishable; prefer simpler."])
ws_lrt.append(["ΔBIC 2–6",       "Positive evidence for simpler model."])
ws_lrt.append(["ΔBIC 6–10",      "Strong evidence for simpler model."])
ws_lrt.append(["ΔBIC > 10",      "Very strong evidence for simpler model — complex model not supported."])
ws_lrt.append([])
ws_lrt.append(["KEY INSIGHT: THE LARGE-N PARADOX"])
ws_lrt.append([f"With N={TOTAL_EVS:,}, the LRT will flag p<0.001 for improvements of "
               f"even 1-2 NLL units."])
ws_lrt.append(["This does not mean the complex model is biologically meaningful."])
ws_lrt.append([f"ΔBIC accounts for sample size: it costs ln({TOTAL_EVS})="
               f"{np.log(TOTAL_EVS):.2f} BIC units per extra parameter."])
ws_lrt.append(["Use LRT to confirm a real improvement exists; use ΔBIC to judge if it is worth the complexity."])
ws_lrt.append(["A significant LRT with ΔBIC > 0 means: detectable but not justified by parsimony."])
auto_width(ws_lrt)

# ── Akaike Weights ────────────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("SECTION 6: Akaike Weights")
print("─" * 60)

_all_models = [(opt, lnk) for opt in [1,2,3] for lnk in [False,True]]
_aic_vals = {}
for opt, lnk in _all_models:
    d = linked_results if lnk else indep_results
    label = f"Option {opt}{'B' if lnk else 'A'}"
    if opt in d:
        _aic_vals[label] = float(d[opt]["main_res"].get("AIC", float("nan")))

_aic_finite = {k: v for k, v in _aic_vals.items() if not np.isnan(v)}
_aic_min    = min(_aic_finite.values()) if _aic_finite else float("nan")
_delta_aics = {k: v - _aic_min for k, v in _aic_finite.items()}
_exp_terms  = {k: float(np.exp(-0.5 * _da)) for k, _da in _delta_aics.items()}
_sum_exp    = sum(_exp_terms.values())
_aic_weights = {k: v / _sum_exp for k, v in _exp_terms.items()}

_bic_vals_aw = {}
for opt, lnk in _all_models:
    _d = linked_results if lnk else indep_results
    label = f"Option {opt}{'B' if lnk else 'A'}"
    if opt in _d:
        _bic_vals_aw[label] = float(_d[opt]["main_res"].get("BIC", float("nan")))
_bic_finite_aw  = {k: v for k, v in _bic_vals_aw.items() if not np.isnan(v)}
_bic_min_aw     = min(_bic_finite_aw.values()) if _bic_finite_aw else float("nan")
_delta_bics_aw  = {k: v - _bic_min_aw for k, v in _bic_finite_aw.items()}
_bf_terms       = {k: float(np.exp(-0.5 * _db)) for k, _db in _delta_bics_aw.items()}
_sum_bf         = sum(_bf_terms.values())
_bic_weights_aw = {k: v / _sum_bf for k, v in _bf_terms.items()}

aw_records = []
print(f"\n  {'Model':<12} {'AIC':>10} {'ΔAIC':>8} {'AIC_weight':>12} "
      f"{'BIC':>10} {'ΔBIC':>8} {'BIC_weight':>12}")
print(f"  {'-'*80}")
for label in sorted(_aic_finite.keys()):
    aic   = _aic_finite.get(label, float("nan"))
    daic  = _delta_aics.get(label, float("nan"))
    w_aic = _aic_weights.get(label, float("nan"))
    bic   = _bic_finite_aw.get(label, float("nan"))
    dbic  = _delta_bics_aw.get(label, float("nan"))
    w_bic = _bic_weights_aw.get(label, float("nan"))
    print(f"  {label:<12} {aic:>10.1f} {daic:>8.1f} {w_aic:>12.5f} "
          f"{bic:>10.1f} {dbic:>8.1f} {w_bic:>12.5f}")
    aw_records.append({
        "Model": label, "AIC": round(aic,2),
        "Delta_AIC": round(daic,2),
        "AIC_Weight": round(w_aic,6),
        "AIC_Weight_Pct": round(w_aic*100,3),
        "BIC": round(bic,2),
        "Delta_BIC": round(dbic,2),
        "BIC_Weight": round(w_bic,6),
        "BIC_Weight_Pct": round(w_bic*100,3),
        "AIC_Cumulative_Wt": None,  # filled below
        "BIC_Cumulative_Wt": None,
    })

# Add cumulative weights (sorted by AIC weight)
aw_records_sorted_aic = sorted(aw_records, key=lambda x: x["AIC_Weight"], reverse=True)
cumsum = 0.0
for r in aw_records_sorted_aic:
    cumsum += r["AIC_Weight"]
    r["AIC_Cumulative_Wt"] = round(cumsum, 6)
aw_records_sorted_bic = sorted(aw_records, key=lambda x: x["BIC_Weight"], reverse=True)
cumsum = 0.0
for r in aw_records_sorted_bic:
    cumsum += r["BIC_Weight"]
    r["BIC_Cumulative_Wt"] = round(cumsum, 6)

# ── Print explicit conclusions from AIC and BIC weights ──────────────────────
_bic_best_label  = max(_bic_weights_aw, key=_bic_weights_aw.get)
_aic_best_label  = max(_aic_weights,    key=_aic_weights.get)
_bic_best_wt     = _bic_weights_aw[_bic_best_label]
_aic_best_wt     = _aic_weights[_aic_best_label]
_ln_N            = float(np.log(TOTAL_EVS))
_bic_aic_ratio   = _ln_N / 2.0
# BIC model-averaged phi for CD9-CD81 (1B and 2B are the only models with >1% BIC weight)
_phi_avg_components = []
for _lbl, _wt in _bic_weights_aw.items():
    if _wt > 0.001:
        _opt  = int(_lbl.split()[1][0])
        _lnk  = _lbl.endswith("B")
        _d    = linked_results if _lnk else indep_results
        if _opt in _d:
            _phi_val = _d[_opt]["main_res"].get("phi_9_81_A")
            if _phi_val is not None:
                _phi_avg_components.append((_lbl, _wt, float(_phi_val)))
_phi_model_avg = (sum(w * p for _, w, p in _phi_avg_components) /
                  sum(w for _, w, _ in _phi_avg_components)) if _phi_avg_components else float("nan")

print(f"\n  → Best model by BIC weight : {_bic_best_label}  (BIC_weight={_bic_best_wt:.4f}, "
      f"{_bic_best_wt*100:.1f}% posterior probability)")
print(f"  → Best model by AIC weight : {_aic_best_label}  (AIC_weight={_aic_best_wt:.4f})")
print(f"  → BIC penalizes each param by ln({TOTAL_EVS})={_ln_N:.2f} units "
      f"(vs AIC's 2.0, ratio={_bic_aic_ratio:.2f}×)")
if not np.isnan(_phi_model_avg):
    print(f"  → BIC-weighted model-averaged phi(CD9–CD81) = {_phi_model_avg:.4f}")
    _phi_components_str = " + ".join(
        f"{w*100:.1f}%×{p:.4f}({l})" for l, w, p in _phi_avg_components)
    print(f"     = {_phi_components_str}")
print(f"  → Recommendation: use BIC weights for mechanistic inference "
      f"(N={TOTAL_EVS:,}, large-N regime where BIC is appropriate)")

ws_aw = wb.create_sheet("Akaike_BIC_Weights")
aw_cols = ["Model","AIC","Delta_AIC","AIC_Weight","AIC_Weight_Pct","AIC_Cumulative_Wt",
           "BIC","Delta_BIC","BIC_Weight","BIC_Weight_Pct","BIC_Cumulative_Wt"]
ws_aw.append(aw_cols)
style_header_row(ws_aw, 1)
for rec in sorted(aw_records, key=lambda x: x["BIC_Weight"], reverse=True):
    ws_aw.append([rec.get(c) for c in aw_cols])
ws_aw.append([])
ws_aw.append(["HOW TO READ THIS TABLE"])
ws_aw.append(["Delta_AIC",        "AIC of this model minus lowest AIC. Models within ΔAIC<2 are essentially equivalent predictors."])
ws_aw.append(["AIC_Weight",       "Probability this is the best-predicting model (Akaike weight). All weights sum to 1."])
ws_aw.append(["AIC_Weight_Pct",   "Same as AIC_Weight expressed as a percentage."])
ws_aw.append(["AIC_Cumulative_Wt","Cumulative AIC weight (sorted best-first). Top-N models capturing 95% of weight form the 'confidence set'."])
ws_aw.append(["Delta_BIC",
              f"BIC of this model minus lowest BIC. BIC penalizes each parameter by "
              f"ln({TOTAL_EVS})={np.log(TOTAL_EVS):.2f} (vs AIC's 2.0)."])
ws_aw.append(["BIC_Weight",       "Posterior model probability (Bayes factor approximation). All weights sum to 1."])
ws_aw.append(["BIC_Cumulative_Wt","Cumulative BIC weight (sorted best-first). Analogous to posterior model probability."])
ws_aw.append([])
ws_aw.append(["SIGNIFICANCE THRESHOLDS"])
ws_aw.append(["ΔAIC < 2",  "Models are indistinguishable in predictive accuracy; prefer simpler."])
ws_aw.append(["ΔAIC 2–7",  "Moderate evidence for the lower-AIC model."])
ws_aw.append(["ΔAIC > 10", "Strong evidence for the lower-AIC model."])
ws_aw.append(["ΔBIC < 2",  "Models indistinguishable on parsimony-adjusted basis."])
ws_aw.append(["ΔBIC 2–6",  "Positive evidence for simpler (lower-BIC) model (Kass & Raftery 1995)."])
ws_aw.append(["ΔBIC 6–10", "Strong evidence for simpler model."])
ws_aw.append(["ΔBIC > 10", "Very strong evidence for simpler model — complex model not supported."])
ws_aw.append([])
ws_aw.append(["AIC vs BIC — WHEN TO USE WHICH"])
ws_aw.append(["AIC weights",  "Best for prediction goals: which model will generalize best to new EVs?"])
ws_aw.append(["BIC weights",  "Best for identification goals: which model most likely represents the true mechanism?"])
ws_aw.append(["At large N",
              f"BIC penalizes extra parameters {np.log(TOTAL_EVS)/2:.1f}× more than AIC "
              f"(ln({TOTAL_EVS})/2 = {np.log(TOTAL_EVS)/2:.2f}). "
              f"AIC will favor larger models; BIC will favor parsimonious ones."])
ws_aw.append(["Recommendation","For mechanistic interpretation of EV co-packaging, BIC weights are the appropriate criterion."])
ws_aw.append([])
ws_aw.append(["MODEL AVERAGING NOTE"])
ws_aw.append(["BIC weights can be used to average parameter estimates across models.",
               "e.g. phi_averaged = BIC_weight(1B)×phi(1B) + BIC_weight(2B)×phi(2B) + ...",
               "This gives a model-averaged phi that honestly reflects uncertainty about which model is true."])
auto_width(ws_aw)

# ── Profile Likelihood CIs for all phi parameters ────────────────────────────
# For each linked model and each free phi parameter, sweep phi across a fine
# grid centered on the MLE, optimize all other parameters via L-BFGS-B, record NLL.
# 95% CI = range where ΔNLL ≤ 1.92 (= 0.5 × chi²₀.₀₅(df=1)).
# Grid is centered on MLE ± 5× the bootstrap CI half-width for adequate resolution.
print("\n" + "─" * 60)
print("SECTION 7: Profile Likelihood CIs for all phi parameters")
print("─" * 60)
print("  Each phi fixed in turn; all other params optimized via L-BFGS-B.")
_PROFILE_N_GRID = 80
_PROFILE_THRESHOLD = 1.92

print(f"  95% CI threshold: ΔNLL = {_PROFILE_THRESHOLD} (= 0.5 × chi²₀.₀₅(df=1))")
print(f"  Grid: {_PROFILE_N_GRID} points centered on MLE ± max(0.05, 5×bootstrap_halfwidth)")

# All free phi parameters per option (linked only), with their param vector index,
# the pair of markers they describe, the population label, and the indices of the
# two marginal probability parameters used to compute the feasibility bounds.
_PHI_PARAMS = {
    1: [
        {"name": "phi_9_81_A",  "idx": 3,  "pair": ("CD9","CD81"),  "pop": "A", "p_idx": (0, 1)},
        {"name": "phi_9_63_A",  "idx": 4,  "pair": ("CD9","CD63"),  "pop": "A", "p_idx": (0, 2)},
    ],
    2: [
        {"name": "phi_9_81_B",  "idx": 4,  "pair": ("CD9","CD81"),  "pop": "B", "p_idx": (1, 2)},
        {"name": "phi_9_81_A",  "idx": 8,  "pair": ("CD9","CD81"),  "pop": "A", "p_idx": (5, 6)},
        {"name": "phi_9_63_A",  "idx": 9,  "pair": ("CD9","CD63"),  "pop": "A", "p_idx": (5, 7)},
    ],
    3: [
        {"name": "phi_9_81_B",  "idx": 5,  "pair": ("CD9","CD81"),  "pop": "B", "p_idx": (2, 3)},
        {"name": "phi_81_63_C", "idx": 9,  "pair": ("CD81","CD63"), "pop": "C", "p_idx": (7, 8)},
        {"name": "phi_9_81_A",  "idx": 13, "pair": ("CD9","CD81"),  "pop": "A", "p_idx": (10,11)},
        {"name": "phi_9_63_A",  "idx": 14, "pair": ("CD9","CD63"),  "pop": "A", "p_idx": (10,12)},
    ],
}

profile_records    = []
profile_ci_summary = []

for opt in [1, 2, 3]:
    if opt not in linked_results:
        continue
    res_main = linked_results[opt]
    model_label = f"Option {opt}B"
    best_x   = res_main["best_x"]
    best_nll = float(res_main["main_res"]["nll"])

    bounds_lo, bounds_hi = make_bounds(opt, True,
                                       POP_B_FRAC_OF_CD63, POP_B_FRAC_SD,
                                       POP_C_FRAC_OF_CD9,  POP_C_FRAC_SD)
    bounds_scipy = list(zip(bounds_lo, bounds_hi))

    # Get bootstrap CI half-widths for phi params (from boot_decoded)
    bd = res_main.get("boot_decoded", [])
    boot_phi_vals = {}
    for phi_info in _PHI_PARAMS[opt]:
        pname = phi_info["name"]
        vals = [d.get(pname) for d in bd if d is not None and d.get(pname) is not None]
        if vals:
            boot_phi_vals[pname] = (float(np.percentile(vals, 2.5)),
                                    float(np.percentile(vals, 97.5)))

    for phi_info in _PHI_PARAMS[opt]:
        pname   = phi_info["name"]
        phi_idx = phi_info["idx"]
        pair    = phi_info["pair"]
        pop_lbl = phi_info["pop"]
        p_i, p_j = phi_info["p_idx"]

        phi_best = float(best_x[phi_idx])

        # Build grid centered on MLE with width ≥ 0.10, refined around bootstrap CI
        bci = boot_phi_vals.get(pname, (phi_best - 0.05, phi_best + 0.05))
        half_w = max(0.05, 2.5 * (bci[1] - bci[0]))
        phi_bnd = PHI_BOUNDS.get(pair, (PHI_BOUND_LO_FALLBACK, PHI_BOUND_HI_FALLBACK))
        grid_lo = max(phi_bnd[0] * 0.98, phi_best - half_w)
        grid_hi = min(phi_bnd[1] * 0.98, phi_best + half_w)
        # Guard: ensure phi_best is always inside the grid.
        # Without this, parameters at or near their theoretical bound (e.g.
        # phi_9_81_B = 0.1635 with bound_hi * 0.98 = 0.1602) fall outside
        # the grid, causing the profile to find a pseudo-minimum and report
        # a CI shifted below the true MLE.
        _eps = 1e-5
        grid_lo = min(grid_lo, phi_best - _eps)
        grid_hi = max(grid_hi, phi_best + _eps)
        # Re-clip to just inside the hard parameter bounds
        grid_lo = max(grid_lo, phi_bnd[0] + _eps)
        grid_hi = min(grid_hi, phi_bnd[1] - _eps)
        phi_grid = np.linspace(grid_lo, grid_hi, _PROFILE_N_GRID)

        obj = make_objective(opt, True, observed_counts, TOTAL_EVS,
                             CD63_POSITIVE_EVS, CD9_POSITIVE_EVS,
                             POP_B_FRAC_OF_CD63, POP_C_FRAC_OF_CD9)

        print(f"\n  {model_label} / {pname}: MLE={phi_best:.5f}  "
              f"grid=[{grid_lo:.4f}, {grid_hi:.4f}]")

        profile_nlls = []
        for phi_val in phi_grid:
            x0 = best_x.copy()
            x0[phi_idx] = phi_val
            eps_phi = 1e-7
            bounds_fixed = list(bounds_scipy)
            bounds_fixed[phi_idx] = (phi_val - eps_phi, phi_val + eps_phi)
            try:
                r_lbfgs = minimize(obj, x0, method="L-BFGS-B",
                                   bounds=bounds_fixed,
                                   options={"maxiter": 2000, "ftol": 1e-12, "gtol": 1e-8})
                profile_nlls.append(float(r_lbfgs.fun))
            except Exception:
                profile_nlls.append(float("nan"))

        profile_nlls = np.array(profile_nlls)
        valid_mask   = ~np.isnan(profile_nlls)
        if not valid_mask.any():
            print(f"    ⚠ All grid points failed")
            min_profile = float("nan")
            delta_nlls  = np.full_like(profile_nlls, float("nan"))
            ci_lo = ci_hi = float("nan")
        else:
            min_profile = float(np.nanmin(profile_nlls))
            delta_nlls  = profile_nlls - min_profile
            ci_mask = valid_mask & (delta_nlls <= _PROFILE_THRESHOLD)
            ci_lo   = float(phi_grid[ci_mask][0])  if ci_mask.any() else float("nan")
            ci_hi   = float(phi_grid[ci_mask][-1]) if ci_mask.any() else float("nan")

        bci_lo = round(bci[0], 6) if pname in boot_phi_vals else None
        bci_hi = round(bci[1], 6) if pname in boot_phi_vals else None
        print(f"    Profile 95% CI: [{ci_lo:.5f}, {ci_hi:.5f}]  "
              f"Bootstrap 95% CI: [{bci_lo}, {bci_hi}]")

        profile_ci_summary.append({
            "Model":            model_label,
            "Phi_Parameter":    pname,
            "Population":       f"Pop {pop_lbl}",
            "Marker_Pair":      f"{pair[0]}-{pair[1]}",
            "MLE":              round(phi_best, 6),
            "Profile_CI_Lo_95": round(ci_lo, 6) if not np.isnan(ci_lo) else None,
            "Profile_CI_Hi_95": round(ci_hi, 6) if not np.isnan(ci_hi) else None,
            "Bootstrap_CI_Lo_95": bci_lo,
            "Bootstrap_CI_Hi_95": bci_hi,
            "NLL_at_MLE":       round(best_nll, 4),
            "Min_Profile_NLL":  round(min_profile, 4) if not np.isnan(min_profile) else None,
            "N_grid_points":    _PROFILE_N_GRID,
            "Grid_Lo":          round(grid_lo, 5),
            "Grid_Hi":          round(grid_hi, 5),
            "ΔNLL_threshold":   _PROFILE_THRESHOLD,
        })

        for phi_v, nll_v, dnll_v in zip(phi_grid, profile_nlls, delta_nlls):
            profile_records.append({
                "Model":         model_label,
                "Phi_Parameter": pname,
                "Population":    f"Pop {pop_lbl}",
                "Marker_Pair":   f"{pair[0]}-{pair[1]}",
                "phi_value":     round(float(phi_v), 6),
                "Profile_NLL":   round(float(nll_v), 4) if not np.isnan(nll_v) else None,
                "Delta_NLL":     round(float(dnll_v), 4) if not np.isnan(dnll_v) else None,
                f"Within_{CONFIDENCE*100:.0f}CI": "Yes" if not np.isnan(nll_v) and dnll_v <= _PROFILE_THRESHOLD else "No",
            })

ws_profile = wb.create_sheet("Profile_Likelihood_phi")

ws_profile.append(["HOW TO READ THIS TAB"])
ws_profile.append(["Profile likelihood is the gold-standard method for confidence intervals when parameters may be correlated."])
ws_profile.append(["For each phi parameter, the value is fixed at a grid point; all other parameters are re-optimized."])
ws_profile.append([f"The 95% CI is the contiguous range of phi values where the profile NLL "
                   f"is within {_PROFILE_THRESHOLD} of its minimum."])
ws_profile.append([f"{_PROFILE_THRESHOLD} = 0.5 × chi²({ALPHA}, df=1) — "
                   f"this is the exact likelihood-ratio-based CI threshold."])
ws_profile.append(["Bootstrap CIs are shown for comparison; profile CIs are generally more reliable near parameter boundaries."])
ws_profile.append([])
ws_profile.append(["COLUMN GUIDE"])
ws_profile.append(["MLE",               "Maximum likelihood estimate of this phi parameter from the main fit."])
ws_profile.append(["Profile_CI_Lo/Hi_95",
                   f"Profile likelihood {CONFIDENCE*100:.0f}% CI: phi values where "
                   f"ΔNLL ≤ {_PROFILE_THRESHOLD}."])
ws_profile.append(["Bootstrap_CI_Lo/Hi_95",
                   f"Bootstrap {CONFIDENCE*100:.0f}% CI from {N_BOOTSTRAP:,} resamples "
                   f"for comparison."])
ws_profile.append(["Min_Profile_NLL",   "Minimum NLL found in profile sweep (should ≈ NLL_at_MLE if grid is dense enough)."])
ws_profile.append(["Delta_NLL",
                   f"NLL at this grid point minus Min_Profile_NLL. CI includes points "
                   f"where this ≤ {_PROFILE_THRESHOLD}."])
ws_profile.append([])
ws_profile.append(["INTERPRETATION THRESHOLDS"])
ws_profile.append(["phi = 0",           "Complete independence. CI excluding 0 = statistically significant co-packaging bias."])
ws_profile.append(["phi > 0",           "Co-occurrence excess (positive linkage): markers tend to appear together."])
ws_profile.append(["phi < 0",           "Mutual exclusion (negative linkage): markers tend to appear on different EVs."])
ws_profile.append(["phi/phi_max > 0.5", "Strong linkage: more than half the theoretical maximum co-occurrence is realized."])
ws_profile.append([])

# Summary table
ws_profile.append([f"SUMMARY — Profile Likelihood {CONFIDENCE*100:.0f}% CIs for All Phi Parameters"])
sum_cols = ["Model","Phi_Parameter","Population","Marker_Pair","MLE",
            "Profile_CI_Lo_95","Profile_CI_Hi_95",
            "Bootstrap_CI_Lo_95","Bootstrap_CI_Hi_95",
            "Min_Profile_NLL","N_grid_points","Grid_Lo","Grid_Hi","ΔNLL_threshold"]
ws_profile.append(sum_cols)
style_header_row(ws_profile, ws_profile.max_row)
for rec in profile_ci_summary:
    ws_profile.append([rec.get(c) for c in sum_cols])
ws_profile.append([])

# Full profile data
ws_profile.append(["FULL PROFILE DATA (one row per grid point × phi parameter × model)"])
data_cols = ["Model","Phi_Parameter","Population","Marker_Pair",
             "phi_value","Profile_NLL","Delta_NLL",
             f"Within_{CONFIDENCE*100:.0f}CI"]
ws_profile.append(data_cols)
style_header_row(ws_profile, ws_profile.max_row)
for rec in profile_records:
    ws_profile.append([rec.get(c) for c in data_cols])
auto_width(ws_profile)

save_wb(wb, "Cell_1.08_Robustness", ts)
print(f"\n✓ Robustness analysis complete.")
print(f"  Total cell wall time: {time.time()-_cell_start:.1f}s")
stop_logging(lf)